In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize
import os
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [2]:
from Model import *

In [3]:
pd.set_option("display.max_rows", 11)

In [4]:
date = datetime.today().strftime('%m%d')
num = 1

In [5]:
cancer = "LGG"

In [6]:
data = pd.read_csv(f"/home/koe3/Bioinformatics/Data/{cancer}/0220_LGG_Normed_Entire_Data.csv")
x = torch.from_numpy(data.values).to(dtype = torch.float)

In [7]:
data

,HMGB1P1,A2M,AACS,AADAT,AANAT,AARS2,AASDHPPT,AASDH,AASS,ABAT,...,ZIC2,ZMAT2,ZMAT3,ZNF274,ZNRF3,ZW10,ZWILCH,ZWINT,ZYX,Sex
0,1.777716,0.389232,-1.177514,0.517646,-0.527263,0.475746,-1.120785,0.469608,-0.263035,0.845147,...,0.837401,0.514185,-0.578702,1.071558,0.765242,-0.439994,-1.164296,-1.448524,-0.809574,0
1,0.233934,0.707874,-0.794603,-0.908896,0.586772,-0.776142,-0.485973,-0.160776,0.959209,0.646833,...,1.963239,-0.155367,0.486913,-0.527983,0.277609,-0.035972,0.323295,-0.505197,0.818143,1
2,0.711468,0.824952,-0.271857,1.699544,-0.146400,-1.522602,0.322414,-0.092264,-1.469562,-0.260285,...,-1.301798,0.803925,0.508958,-0.842355,-0.304808,0.249729,-0.458007,0.128614,-0.213299,0
3,1.791092,0.007414,-0.303279,-0.655909,-1.619284,0.736664,0.497619,-1.197012,-1.604390,-0.459375,...,-1.647006,0.889620,-0.325959,1.012508,-1.208278,1.686594,2.562832,2.373111,-0.283270,1
4,1.157604,-0.112626,-0.342409,-1.595685,-0.101821,-0.458447,-0.456942,-0.001335,-2.456418,0.024421,...,0.952211,1.766846,-0.962883,-1.490118,0.266920,-0.494128,-1.748391,-1.310929,-0.705312,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
241,-0.399459,-1.481720,2.889208,-0.322603,-0.221191,1.158866,0.301227,-0.936558,0.256070,-0.789024,...,-0.383512,-0.226762,-1.715259,-1.081246,0.072150,-0.608469,0.286122,0.132115,-0.470074,0
242,-0.413783,-0.370962,-0.024195,1.461690,0.159579,-0.219504,-0.095633,0.518314,0.278427,0.283403,...,0.264129,0.068849,-0.081157,1.076537,0.171947,-1.680221,-0.745575,-0.659906,-0.929458,1
243,-0.218842,0.715807,-0.877743,1.792280,-0.769873,0.728809,0.759816,0.728172,0.676274,-0.173310,...,-0.776744,0.608524,-1.197759,1.242038,-1.331913,1.028516,0.769277,0.889907,-0.837304,1
244,-0.506558,-0.106510,-0.508298,1.856691,0.339280,0.724135,1.219016,0.623828,0.863336,0.192061,...,1.101532,0.308432,-1.414768,2.049526,0.537743,2.332232,-0.836792,-0.923038,-0.467301,1


In [8]:
x, x.shape

(tensor([[ 1.7777e+00,  3.8923e-01, -1.1775e+00,  ..., -1.4485e+00,
          -8.0957e-01,  0.0000e+00],
         [ 2.3393e-01,  7.0787e-01, -7.9460e-01,  ..., -5.0520e-01,
           8.1814e-01,  1.0000e+00],
         [ 7.1147e-01,  8.2495e-01, -2.7186e-01,  ...,  1.2861e-01,
          -2.1330e-01,  0.0000e+00],
         ...,
         [-2.1884e-01,  7.1581e-01, -8.7774e-01,  ...,  8.8991e-01,
          -8.3730e-01,  1.0000e+00],
         [-5.0656e-01, -1.0651e-01, -5.0830e-01,  ..., -9.2304e-01,
          -4.6730e-01,  1.0000e+00],
         [-8.4891e-01, -6.1385e-04,  2.3801e-01,  ..., -4.8876e-01,
          -6.9080e-01,  1.0000e+00]]),
 torch.Size([246, 4790]))

In [9]:
male_indices, female_indices = (x[:, -1] == 1), (x[:, -1] == 0)

In [10]:
label = pd.read_csv(f"/home/koe3/Bioinformatics/Data/{cancer}/0220_LGG_Entire_Label.csv")
label

,Label
0,1
1,0
2,1
3,1
4,1
...,...
241,1
242,1
243,1
244,1


In [11]:
def fixed_s_mask(w, idx):
    '''
    Input: 
        w: weight matrix.
        idx: the indices of having values (or connections).
    Output:
        returns the weight matrix that has been forced the connections.
    '''
    sp_w = torch.sparse_coo_tensor(idx, w[idx[0], idx[1]], w.size())
    
    return sp_w.to_dense()


In [12]:
experiment = 1

In [13]:
pathway_list = pd.read_csv("/home/koe3/Bioinformatics/Data/pathway_list.txt", header=None).values.ravel()

In [14]:
from scipy import sparse
def load_sparse_indices(path):
    coo = sparse.load_npz(path)
    indices = np.vstack((coo.row, coo.col))
    return indices

In [15]:
sparse_indices = load_sparse_indices(f"/home/koe3/Bioinformatics/Data/TCGA_{cancer}_Pathway_Mask.npz")

In [16]:
def get_num_nodes(data_name):
    """Returns the number of input features (genes) for a given dataset."""
    node_map = {
        "LIHC": 4781, "STAD": 4790, "LUAD": 4786,
        "LUSC": 4787, "LGG": 4789, "GSE240567": 4500
    }
    if data_name not in node_map:
        raise ValueError(f"Unknown dataset name for get_num_nodes: {data_name}")
        
    return node_map[data_name]

---

## CL Part

In [17]:
model_date="0813"
model_num=1

In [18]:
hparam_map = {
    1: ("Run_v19", "he_normal", 0.08356555216558127),
    2: ("Run_v19", "he_normal", 0.08356555216558127),
    3: ("Run_v19", "he_normal", 0.08356555216558127),
    4: ("Run_v18", "he_uniform", 0.12032037668773907),
    5: ("Run_v19", "he_normal", 0.07581217434771849),
    6: ("Run_v18", "he_normal", 0.12032037668773907),
    7: ("Run_v18", "he_uniform", 0.12032037668773907),
    8: ("Run_v19", "he_normal", 0.08356555216558127),
    9: ("Run_v35", "he_normal", 0.07581217434771849), ###clf-6
    10: ("Run_v18", "he_uniform", 0.12032037668773907)
}

In [19]:
x.shape[0], male_indices.sum(), female_indices.sum()

(246, tensor(141), tensor(105))

In [20]:
data.iloc[:, -1].sum(), (data.iloc[:, -1] == 0).sum()

(141, 105)

In [21]:
sum_cmn_pathwy_node = torch.zeros((x.shape[0], 395))
sum_cmn_repr_node = torch.zeros((x.shape[0], 128))
sum_cmn_hid_node = torch.zeros((x.shape[0], 128))
sum_cmn_proj_node = torch.zeros((x.shape[0], 16))

In [22]:
sum_male_pathwy_node = torch.zeros((data.iloc[:, -1].sum(), 395))
sum_male_repr_node = torch.zeros((data.iloc[:, -1].sum(), 128))
sum_male_hid_node = torch.zeros((data.iloc[:, -1].sum(), 128))
sum_male_proj_node = torch.zeros((data.iloc[:, -1].sum(), 16))

In [23]:
sum_female_pathwy_node = torch.zeros(((data.iloc[:, -1] == 0).sum(), 395))
sum_female_repr_node = torch.zeros(((data.iloc[:, -1] == 0).sum(), 128))
sum_female_hid_node = torch.zeros(((data.iloc[:, -1] == 0).sum(), 128))
sum_female_proj_node = torch.zeros(((data.iloc[:, -1] == 0).sum(), 16))

In [24]:
sum_cmn_w1 = torch.zeros((395, x.shape[1] - 1))
sum_cmn_w2 = torch.zeros((128, 395))
sum_cmn_w3 = torch.zeros((128, 128))
sum_cmn_w4 = torch.zeros((16, 128))

In [25]:
sum_male_w1 = torch.zeros((395, x.shape[1] - 1))
sum_male_w2 = torch.zeros((128, 395))
sum_male_w3 = torch.zeros((128, 128))
sum_male_w4 = torch.zeros((16, 128))

In [26]:
sum_female_w1 = torch.zeros((395, x.shape[1] - 1))
sum_female_w2 = torch.zeros((128, 395))
sum_female_w3 = torch.zeros((128, 128))
sum_female_w4 = torch.zeros((16, 128))

In [27]:
avg_cmn_pathwy_node, avg_cmn_repr_node, avg_cmn_hid_node, avg_cmn_proj_node = [], [], [], []
avg_male_pathwy_node, avg_male_repr_node, avg_male_hid_node, avg_male_proj_node = [], [], [], []
avg_female_pathwy_node, avg_female_repr_node, avg_female_hid_node, avg_female_proj_node = [], [], [], []
avg_cmn_w1, avg_cmn_w2, avg_cmn_w3, avg_cmn_w4 = [], [], [], []
avg_male_w1, avg_male_w2, avg_male_w3, avg_male_w4 = [], [], [], []
avg_female_w1, avg_female_w2, avg_female_w3, avg_female_w4 = [], [], [], []
for experiment in range(1, 11):
    print("######################## %d experiment ########################" % (experiment))
    cl_net_hparams = [get_num_nodes(cancer), [395, 128, 16], 1, hparam_map[experiment][1], "sigmoid", 0.4] ### LGG
    cl_model = CLModule(cl_net_hparams, sparse_indices)
    cl_model.load_state_dict(torch.load(f"/home/koe3/Bioinformatics/Proposed/SHCL_v3/{cancer}_Result/{hparam_map[experiment][0]}/Saved_Model/[{model_date}_{model_num}]_[{experiment}]Opt_Model_StateDict.pth", map_location='cpu'))
    
    cl_model.eval()
    with torch.no_grad():
        ### common
        common_pathway_nodes = cl_model.encoder_common.encoder.act1(cl_model.encoder_common.encoder.bn1(cl_model.encoder_common.encoder.layer1(x[:, :-1])))
        common_repr_nodes = cl_model.encoder_common.encoder.act2(cl_model.encoder_common.encoder.bn2(cl_model.encoder_common.encoder.layer2(common_pathway_nodes)))
        common_head_hidden_nodes = cl_model.encoder_common.head[2](cl_model.encoder_common.head[1](cl_model.encoder_common.head[0](common_repr_nodes)))
        common_proj_embeddings = cl_model.encoder_common.head[4](common_head_hidden_nodes)
        
        sum_cmn_pathwy_node += common_pathway_nodes
        sum_cmn_repr_node += common_repr_nodes
        sum_cmn_hid_node += common_head_hidden_nodes
        sum_cmn_proj_node += common_proj_embeddings
        ### male
        male_pathway_nodes = cl_model.encoder_male.encoder.act1(cl_model.encoder_male.encoder.bn1(cl_model.encoder_male.encoder.layer1(x[male_indices, :-1])))
        male_repr_nodes = cl_model.encoder_male.encoder.act2(cl_model.encoder_male.encoder.bn2(cl_model.encoder_male.encoder.layer2(male_pathway_nodes)))
        male_head_hidden_nodes = cl_model.encoder_male.head[2](cl_model.encoder_male.head[1](cl_model.encoder_male.head[0](male_repr_nodes)))
        male_proj_embeddings = cl_model.encoder_male.head[4](male_head_hidden_nodes)
        
        sum_male_pathwy_node += male_pathway_nodes
        sum_male_repr_node += male_repr_nodes
        sum_male_hid_node += male_head_hidden_nodes
        sum_male_proj_node += male_proj_embeddings
        ### female
        female_pathway_nodes = cl_model.encoder_female.encoder.act1(cl_model.encoder_female.encoder.bn1(cl_model.encoder_female.encoder.layer1(x[female_indices, :-1])))
        female_repr_nodes = cl_model.encoder_female.encoder.act2(cl_model.encoder_female.encoder.bn2(cl_model.encoder_female.encoder.layer2(female_pathway_nodes)))
        female_head_hidden_nodes = cl_model.encoder_female.head[2](cl_model.encoder_female.head[1](cl_model.encoder_female.head[0](female_repr_nodes)))
        female_proj_embeddings = cl_model.encoder_female.head[4](female_head_hidden_nodes)
        
        sum_female_pathwy_node += female_pathway_nodes
        sum_female_repr_node += female_repr_nodes
        sum_female_hid_node += female_head_hidden_nodes
        sum_female_proj_node += female_proj_embeddings
        
        sum_cmn_w1 += cl_model.encoder_common.encoder.layer1.weight.data
        sum_cmn_w2 += cl_model.encoder_common.encoder.layer2.weight.data
        sum_cmn_w3 += cl_model.encoder_common.head[0].weight.data
        sum_cmn_w4 += cl_model.encoder_common.head[4].weight.data
        
        sum_male_w1 += cl_model.encoder_male.encoder.layer1.weight.data
        sum_male_w2 += cl_model.encoder_male.encoder.layer2.weight.data
        sum_male_w3 += cl_model.encoder_male.head[0].weight.data
        sum_male_w4 += cl_model.encoder_male.head[4].weight.data
        
        sum_female_w1 += cl_model.encoder_female.encoder.layer1.weight.data
        sum_female_w2 += cl_model.encoder_female.encoder.layer2.weight.data
        sum_female_w3 += cl_model.encoder_female.head[0].weight.data
        sum_female_w4 += cl_model.encoder_female.head[4].weight.data
    
avg_cmn_pathwy_node = sum_cmn_pathwy_node/10
avg_cmn_repr_node = sum_cmn_repr_node/10
avg_cmn_hid_node = sum_cmn_hid_node/10
avg_cmn_proj_node = sum_cmn_proj_node/10
avg_male_pathwy_node = sum_male_pathwy_node/10
avg_male_repr_node = sum_male_repr_node/10
avg_male_hid_node = sum_male_hid_node/10
avg_male_proj_node = sum_male_proj_node/10
avg_female_pathwy_node = sum_female_pathwy_node/10
avg_female_repr_node = sum_female_repr_node/10
avg_female_hid_node = sum_female_hid_node/10
avg_female_proj_node = sum_female_proj_node/10

avg_cmn_w1 = sum_cmn_w1/10
avg_cmn_w2 = sum_cmn_w2/10
avg_cmn_w3 = sum_cmn_w3/10
avg_cmn_w4 = sum_cmn_w4/10
avg_male_w1 = sum_male_w1/10
avg_male_w2 = sum_male_w2/10
avg_male_w3 = sum_male_w3/10
avg_male_w4 = sum_male_w4/10
avg_female_w1 = sum_female_w1/10
avg_female_w2 = sum_female_w2/10
avg_female_w3 = sum_female_w3/10
avg_female_w4 = sum_female_w4/10

######################## 1 experiment ########################
######################## 2 experiment ########################
######################## 3 experiment ########################
######################## 4 experiment ########################
######################## 5 experiment ########################
######################## 6 experiment ########################
######################## 7 experiment ########################
######################## 8 experiment ########################
######################## 9 experiment ########################
######################## 10 experiment ########################


In [28]:
date

'0825'

In [29]:
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Common_Pathway_Nodes.csv", avg_cmn_pathwy_node.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Common_Representation_Nodes.csv", avg_cmn_repr_node.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Common_Head_Hidden_Nodes.csv", avg_cmn_hid_node.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Common_Projection_Nodes.csv", avg_cmn_proj_node.cpu().detach().numpy(), delimiter = ",")

In [30]:
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Male_Pathway_Nodes.csv", avg_male_pathwy_node.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Male_Representation_Nodes.csv", avg_male_repr_node.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Male_Head_Hidden_Nodes.csv", avg_male_hid_node.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Male_Projection_Nodes.csv", avg_male_proj_node.cpu().detach().numpy(), delimiter = ",")

In [31]:
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Female_Pathway_Nodes.csv", avg_female_pathwy_node.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Female_Representation_Nodes.csv", avg_female_repr_node.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Female_Head_Hidden_Nodes.csv", avg_female_hid_node.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Female_Projection_Nodes.csv", avg_female_proj_node.cpu().detach().numpy(), delimiter = ",")

In [32]:
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Common_Encoder_Layer1_Weights.csv", avg_cmn_w1.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Common_Encoder_Layer2_Weights.csv", avg_cmn_w2.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Common_Head_Layer1_Weights.csv", avg_cmn_w3.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Common_Head_Layer2_Weights.csv", avg_cmn_w4.cpu().detach().numpy(), delimiter = ",")

In [33]:
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Male_Encoder_Layer1_Weights.csv", avg_male_w1.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Male_Encoder_Layer2_Weights.csv", avg_male_w2.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Male_Head_Layer1_Weights.csv", avg_male_w3.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Male_Head_Layer2_Weights.csv", avg_male_w4.cpu().detach().numpy(), delimiter = ",")

In [34]:
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Female_Encoder_Layer1_Weights.csv", avg_female_w1.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Female_Encoder_Layer2_Weights.csv", avg_female_w2.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Female_Head_Layer1_Weights.csv", avg_female_w3.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Female_Head_Layer2_Weights.csv", avg_female_w4.cpu().detach().numpy(), delimiter = ",")

In [35]:
cl_model.eval()

CLModule(
  (encoder_common): GeneExpressionBackbone(
    (act): Sigmoid()
    (encoder): GeneExpressionEncoder(
      (act1): Sigmoid()
      (act2): Sigmoid()
      (dropout): Dropout(p=0.4, inplace=False)
      (layer1): Linear(in_features=4789, out_features=395, bias=True)
      (bn1): BatchNorm1d(395, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)
      (layer2): Linear(in_features=395, out_features=128, bias=True)
      (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)
    )
    (head): Sequential(
      (0): Linear(in_features=128, out_features=128, bias=True)
      (1): BatchNorm1d(128, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)
      (2): Sigmoid()
      (3): Dropout(p=0.4, inplace=False)
      (4): Linear(in_features=128, out_features=16, bias=True)
    )
  )
  (encoder_male): GeneExpressionBackbone(
    (act): Sigmoid()
    (encoder): GeneExpressionEncoder(
      (act1): Sigmoid()
      (act2): Si

---

## CLF Part

In [36]:
from sklearn.preprocessing import StandardScaler

In [37]:
sum_hidden_1 = torch.zeros((x.shape[0], 128))
sum_hidden_2 = torch.zeros((x.shape[0], 64))
sum_hidden_3 = torch.zeros((x.shape[0], 32))
sum_pred = torch.zeros((x.shape[0], 1))

In [40]:
sum_clf_w1 = torch.zeros((128, 256))
sum_clf_w2 = torch.zeros((64, 128))
sum_clf_w3 = torch.zeros((32, 64))
sum_clf_w4 = torch.zeros((1, 32))

In [48]:
avg_hidden_1, avg_hidden_2, avg_hidden_3, avg_pred = [], [], [], []
avg_clf_w1, avg_clf_w2, avg_clf_w3, avg_clf_w4 = [], [], [], []
avg_embeddings = torch.zeros((x.shape[0], 256))
for experiment in range(1, 11):
    print("######################## %d experiment ########################" % (experiment))
    cl_net_hparams = [get_num_nodes(cancer), [395, 128, 16], 1, hparam_map[experiment][1], "sigmoid", 0.4] ### LGG
    cl_model = CLModule(cl_net_hparams, sparse_indices)
    cl_model.load_state_dict(torch.load(f"/home/koe3/Bioinformatics/Proposed/SHCL_v3/{cancer}_Result/{hparam_map[experiment][0]}/Saved_Model/[{model_date}_{model_num}]_[{experiment}]Opt_Model_StateDict.pth", map_location='cpu'))
    
    cl_model.eval()
    with torch.no_grad():
        cmn_embeddings, spf_embeddings = cl_model(x)
        embeddings = torch.cat((cmn_embeddings[0], spf_embeddings[0]), dim=1).cpu().detach().numpy()

        scaler = StandardScaler()
        scaled_embeddings = scaler.fit_transform(embeddings)
        x_embeddings = torch.from_numpy(scaled_embeddings).float()
        
    avg_embeddings += x_embeddings

    clf_net_hparams = [256, [128, 64, 32], 1, hparam_map[experiment][1], "tanh", hparam_map[experiment][2]]
    clf = Classifier(clf_net_hparams)
    clf.load_state_dict(torch.load(f"/home/koe3/Bioinformatics/Proposed/SHCL_v3/Opt_Params/SHCL/{cancer}/[{experiment}]Opt_Model_StateDict.pth", map_location='cpu'))
    clf.eval()
    with torch.no_grad():
        hidden_1 = clf.act1(clf.clf_bn1(clf.clf_layer1(x_embeddings)))
        hidden_2 = clf.act2(clf.clf_bn2(clf.clf_layer2(hidden_1)))
        hidden_3 = clf.act3(clf.clf_bn3(clf.clf_layer3(hidden_2)))
        pred = clf.output_activation(clf.clf_layer4(hidden_3))
        
        sum_hidden_1 += hidden_1
        sum_hidden_2 += hidden_2
        sum_hidden_3 += hidden_3
        sum_pred += pred
        
        sum_clf_w1 += clf.clf_layer1.weight.data
        sum_clf_w2 += clf.clf_layer2.weight.data
        sum_clf_w3 += clf.clf_layer3.weight.data
        sum_clf_w4 += clf.clf_layer4.weight.data

avg_embeddings /= 10

avg_hidden_1 = sum_hidden_1 / 10
avg_hidden_2 = sum_hidden_2 / 10
avg_hidden_3 = sum_hidden_3 / 10
avg_pred = sum_pred / 10

avg_clf_w1 = sum_clf_w1 / 10
avg_clf_w2 = sum_clf_w2 / 10
avg_clf_w3 = sum_clf_w3 / 10
avg_clf_w4 = sum_clf_w4 / 10

######################## 1 experiment ########################
######################## 2 experiment ########################
######################## 3 experiment ########################
######################## 4 experiment ########################
######################## 5 experiment ########################
######################## 6 experiment ########################
######################## 7 experiment ########################
######################## 8 experiment ########################
######################## 9 experiment ########################
######################## 10 experiment ########################


In [145]:
clf

Classifier(
  (clf_layer1): Linear(in_features=256, out_features=128, bias=True)
  (clf_bn1): BatchNorm1d(128, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)
  (clf_layer2): Linear(in_features=128, out_features=64, bias=True)
  (clf_bn2): BatchNorm1d(64, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)
  (clf_layer3): Linear(in_features=64, out_features=32, bias=True)
  (clf_bn3): BatchNorm1d(32, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)
  (clf_layer4): Linear(in_features=32, out_features=1, bias=True)
  (act1): Tanh()
  (act2): Tanh()
  (act3): Tanh()
  (output_activation): Sigmoid()
  (dropout): Dropout(p=0.12032037668773907, inplace=False)
)

In [43]:
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Hidden1_Nodes.csv", avg_hidden_1.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Hidden2_Nodes.csv", avg_hidden_2.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Hidden3_Nodes.csv", avg_hidden_3.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Pred.csv", avg_pred.cpu().detach().numpy(), delimiter = ",")

In [44]:
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Classifier_Layer1_Weights.csv", avg_clf_w1.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Classifier_Layer2_Weights.csv", avg_clf_w2.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Classifier_Layer3_Weights.csv", avg_clf_w3.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Classifier_Layer4_Weights.csv", avg_clf_w4.cpu().detach().numpy(), delimiter = ",")

---

# Partial Derivatives

---

## Derivative of activation

In [45]:
def dTanh(x):
    return 1.0 - np.tanh(x)**2

In [46]:
def dSigmoid(x):
    sigmoid = 1 / (1 + np.exp(-x))
    return sigmoid * (1 - sigmoid)

---

In [43]:
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Hidden1_Nodes.csv", avg_hidden_1.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Hidden2_Nodes.csv", avg_hidden_2.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Hidden3_Nodes.csv", avg_hidden_3.cpu().detach().numpy(), delimiter = ",")
np.savetxt(f"Opt_Params/[{date}_{num}]AVG_Pred.csv", avg_pred.cpu().detach().numpy(), delimiter = ",")

In [83]:
avg_hidden_1 = pd.read_csv(f"Opt_Params/[0825_1]AVG_Hidden1_Nodes.csv", header=None).values
avg_hidden_2 = pd.read_csv(f"Opt_Params/[0825_1]AVG_Hidden2_Nodes.csv", header=None).values
avg_hidden_3 = pd.read_csv(f"Opt_Params/[0825_1]AVG_Hidden3_Nodes.csv", header=None).values

In [84]:
avg_hidden_1.shape, avg_hidden_2.shape, avg_hidden_3.shape

((246, 128), (246, 64), (246, 32))

In [85]:
embeddings = avg_embeddings.cpu().detach().numpy()

In [86]:
embeddings, embeddings.shape

(array([[-0.21734229,  0.16367278, -0.08737297, ..., -0.36137527,
          0.01913761, -0.3474447 ],
        [ 0.31568947, -0.2170327 ,  0.34615088, ...,  0.5376745 ,
         -0.06319244,  0.79010695],
        [-0.03181416,  0.18808314,  0.25334364, ..., -0.24553967,
         -0.01216366, -0.20858307],
        ...,
        [-0.04744827,  0.2972142 ,  0.05489291, ..., -0.22453193,
          0.02896296, -0.3045965 ],
        [-0.12834409,  0.14207944, -0.03775351, ..., -0.25571662,
         -0.01693831, -0.16306461],
        [-0.07411804,  0.0412467 , -0.09000495, ..., -0.3012169 ,
          0.16325124, -0.1312675 ]], dtype=float32),
 (246, 256))

In [73]:
avg_clf_w1 = pd.read_csv(f"Opt_Params/[0825_1]AVG_Classifier_Layer1_Weights.csv", header=None).values
avg_clf_w2 = pd.read_csv(f"Opt_Params/[0825_1]AVG_Classifier_Layer2_Weights.csv", header=None).values
avg_clf_w3 = pd.read_csv(f"Opt_Params/[0825_1]AVG_Classifier_Layer3_Weights.csv", header=None).values
avg_clf_w4 = pd.read_csv(f"Opt_Params/[0825_1]AVG_Classifier_Layer4_Weights.csv", header=None).values

In [74]:
avg_clf_w1.shape, avg_clf_w2.shape, avg_clf_w3.shape, avg_clf_w4.shape

((128, 256), (64, 128), (32, 64), (1, 32))

---

---

In [88]:
clf_hidden2_into_w3 = np.dot(avg_hidden_2, avg_clf_w3.T)
clf_hidden2_into_w3, clf_hidden2_into_w3.shape

(array([[-0.1342879 ,  0.14620394, -0.12367464, ...,  0.0134123 ,
         -0.16140059,  0.03990884],
        [ 0.06023911, -0.09359307, -0.03426519, ..., -0.035614  ,
          0.0654332 ,  0.0091695 ],
        [-0.1784884 ,  0.10320726, -0.08956369, ...,  0.07229864,
         -0.05668308, -0.07431935],
        ...,
        [-0.1432731 ,  0.05292214, -0.02123726, ...,  0.05157791,
         -0.03413963,  0.06088368],
        [-0.00091247,  0.08786163, -0.0368561 , ..., -0.08187666,
          0.16833354, -0.00691918],
        [-0.09874203, -0.06446181,  0.02736224, ...,  0.02001386,
          0.06845085, -0.02275203]]),
 (246, 32))

In [89]:
df_hidden2_into_w3 = dTanh(clf_hidden2_into_w3)
df_hidden2_into_w3, df_hidden2_into_w3.shape

(array([[0.98218136, 0.97892537, 0.98485921, ..., 0.99982013, 0.97439567,
         0.99840897],
        [0.99638001, 0.99129124, 0.99882681, ..., 0.99873271, 0.99573069,
         0.99991592],
        [0.9688065 , 0.98942345, 0.99202105, ..., 0.99479107, 0.9967939 ,
         0.99449691],
        ...,
        [0.97975049, 0.99720447, 0.99954911, ..., 0.99734443, 0.99883539,
         0.99630232],
        [0.99999917, 0.99231989, 0.99864286, ..., 0.99332606, 0.97219064,
         0.99995213],
        [0.99031304, 0.99585616, 0.99925168, ..., 0.99959955, 0.99532908,
         0.99948252]]),
 (246, 32))

In [90]:
clf_w4_into_df_hidden2_into_w3 = np.multiply(avg_clf_w4, df_hidden2_into_w3)
clf_w4_into_df_hidden2_into_w3, clf_w4_into_df_hidden2_into_w3.shape

(array([[0.00439217, 0.11989644, 0.03903407, ..., 0.1190898 , 0.06958928,
         0.06957685],
        [0.00445566, 0.12141098, 0.03958766, ..., 0.11896028, 0.07111298,
         0.06968187],
        [0.00433235, 0.12118221, 0.03931792, ..., 0.11849078, 0.07118891,
         0.06930423],
        ...,
        [0.00438129, 0.12213522, 0.03961629, ..., 0.11879492, 0.07133471,
         0.06943004],
        [0.00447184, 0.12153696, 0.03958037, ..., 0.11831628, 0.0694318 ,
         0.06968439],
        [0.00442853, 0.12197008, 0.0396045 , ..., 0.11906353, 0.0710843 ,
         0.06965166]]),
 (246, 32))

In [92]:
h2_updated = np.dot(clf_w4_into_df_hidden2_into_w3, avg_clf_w3)
h2_updated, h2_updated.shape

(array([[-8.24506815e-03,  1.27847119e-05,  2.12664606e-03, ...,
         -1.44068992e-02, -1.55706801e-02, -1.47811185e-02],
        [-7.74685306e-03, -1.41788272e-04,  2.56036619e-03, ...,
         -1.44378787e-02, -1.55088618e-02, -1.46688257e-02],
        [-8.33522771e-03,  3.07540670e-04,  2.31836180e-03, ...,
         -1.40226619e-02, -1.57810502e-02, -1.43318376e-02],
        ...,
        [-7.79347771e-03, -1.83832840e-03,  2.52454554e-03, ...,
         -1.44779458e-02, -1.55490081e-02, -1.54616700e-02],
        [-7.78508943e-03, -1.11823507e-03,  2.61841902e-03, ...,
         -1.48138891e-02, -1.54507143e-02, -1.54522725e-02],
        [-8.04263210e-03, -7.33877572e-04,  2.52858013e-03, ...,
         -1.46793209e-02, -1.56060934e-02, -1.50754104e-02]]),
 (246, 64))

---

In [93]:
clf_hidden1_into_w2 = np.dot(avg_hidden_1, avg_clf_w2.T)
clf_hidden1_into_w2, clf_hidden1_into_w2.shape

(array([[ 0.0727811 , -0.05921338, -0.00375523, ..., -0.12191762,
         -0.00256494,  0.0824086 ],
        [-0.17832543,  0.10053613,  0.09603798, ...,  0.03688287,
          0.11896042, -0.01871182],
        [ 0.0044881 , -0.06203564, -0.03185056, ..., -0.05898675,
         -0.08488613, -0.03524341],
        ...,
        [-0.03286112, -0.05648847, -0.0815179 , ..., -0.09370972,
         -0.13470198, -0.036513  ],
        [ 0.01017568,  0.05641809, -0.11397128, ..., -0.04654982,
          0.00144702, -0.07835928],
        [ 0.01488524, -0.10610935, -0.09526478, ..., -0.0336847 ,
          0.03106532,  0.09211453]]),
 (246, 64))

In [94]:
df_hidden1_into_w2 = dTanh(clf_hidden1_into_w2)
df_hidden1_into_w2, df_hidden1_into_w2.shape

(array([[0.99472156, 0.99650195, 0.9999859 , ..., 0.98528215, 0.99999342,
         0.99323945],
        [0.96886225, 0.98996021, 0.99083312, ..., 0.99864089, 0.98598087,
         0.99964995],
        [0.99997986, 0.99616143, 0.99898623, ..., 0.99652862, 0.99282882,
         0.99875893],
        ...,
        [0.99892092, 0.99681583, 0.99338416, ..., 0.99126964, 0.98207263,
         0.99866799],
        [0.99989646, 0.99682374, 0.98712221, ..., 0.99783624, 0.99999791,
         0.99388487],
        [0.99977846, 0.98882478, 0.99097925, ..., 0.9988662 , 0.99903557,
         0.99156268]]),
 (246, 64))

In [95]:
h2_updated_into_df_hidden1_into_w2 = np.multiply(h2_updated, df_hidden1_into_w2)
h2_updated_into_df_hidden1_into_w2, h2_updated_into_df_hidden1_into_w2.shape

(array([[-8.20154707e-03,  1.27399904e-05,  2.12661607e-03, ...,
         -1.41948606e-02, -1.55705777e-02, -1.46811900e-02],
        [-7.50563348e-03, -1.40364747e-04,  2.53689563e-03, ...,
         -1.44182560e-02, -1.52914410e-02, -1.46636909e-02],
        [-8.33505981e-03,  3.06360154e-04,  2.31601151e-03, ...,
         -1.39739839e-02, -1.56678814e-02, -1.43140508e-02],
        ...,
        [-7.78506795e-03, -1.83247484e-03,  2.50784355e-03, ...,
         -1.43515481e-02, -1.52702552e-02, -1.54410748e-02],
        [-7.78428338e-03, -1.11468326e-03,  2.58469957e-03, ...,
         -1.47818354e-02, -1.54506819e-02, -1.53577798e-02],
        [-8.04085036e-03, -7.25676330e-04,  2.50577043e-03, ...,
         -1.46626775e-02, -1.55910424e-02, -1.49482143e-02]]),
 (246, 64))

In [96]:
h1_updated = np.dot(h2_updated_into_df_hidden1_into_w2, avg_clf_w2)
h1_updated, h1_updated.shape

(array([[ 0.01105868,  0.01001941, -0.00826629, ..., -0.004888  ,
         -0.00220961, -0.00236927],
        [ 0.01118327,  0.00990735, -0.00840762, ..., -0.00475072,
         -0.00173998, -0.00251364],
        [ 0.01135154,  0.01012833, -0.00847035, ..., -0.0049305 ,
         -0.00218094, -0.00248729],
        ...,
        [ 0.01181641,  0.01015679, -0.00820446, ..., -0.00469091,
         -0.00224185, -0.00288494],
        [ 0.01144288,  0.0099779 , -0.00834269, ..., -0.00464198,
         -0.00210165, -0.00276858],
        [ 0.01136292,  0.01011318, -0.00829064, ..., -0.00476916,
         -0.00209688, -0.00266138]]),
 (246, 128))

---

In [97]:
clf_e_into_w1 = np.dot(embeddings, avg_clf_w1.T)
clf_e_into_w1, clf_e_into_w1.shape

(array([[-0.04057354,  0.14306367, -0.03639676, ..., -0.02503197,
          0.24858135,  0.07877441],
        [-0.07234334,  0.06409086,  0.30160366, ...,  0.20522645,
         -0.37860159, -0.08568002],
        [ 0.08344102,  0.10816413, -0.00959616, ...,  0.04132646,
          0.18154405,  0.06777557],
        ...,
        [ 0.09656687,  0.01077943, -0.28113107, ..., -0.10749289,
          0.26946726,  0.03861079],
        [ 0.08073885, -0.09013037, -0.12753668, ..., -0.02277415,
          0.18501363, -0.03969435],
        [ 0.0703409 ,  0.0331807 , -0.11076929, ..., -0.14903506,
          0.08514708, -0.03857626]]),
 (246, 128))

In [98]:
df_e_into_w1 = dTanh(clf_e_into_w1)
df_e_into_w1, df_e_into_w1.shape

(array([[0.99835559, 0.97980885, 0.99867644, ..., 0.99937366, 0.94066652,
         0.99382017],
        [0.99478465, 0.99590358, 0.91428017, ..., 0.9590371 , 0.86932324,
         0.99269471],
        [0.99306979, 0.98839117, 0.99990792, ..., 0.99829407, 0.96775263,
         0.9954205 ],
        ...,
        [0.99073251, 0.99988381, 0.92495053, ..., 0.98853371, 0.9307631 ,
         0.99851069],
        [0.99350946, 0.99192031, 0.98390916, ..., 0.99948152, 0.9665362 ,
         0.99842601],
        [0.99506843, 0.99889985, 0.98782984, ..., 0.97811336, 0.99278487,
         0.99851335]]),
 (246, 128))

In [99]:
h1_updated_into_df_e_into_w1 = np.multiply(h1_updated, df_e_into_w1)
h1_updated_into_df_e_into_w1, h1_updated_into_df_e_into_w1.shape

(array([[ 0.0110405 ,  0.00981711, -0.00825535, ..., -0.00488494,
         -0.0020785 , -0.00235463],
        [ 0.01112494,  0.00986676, -0.00768692, ..., -0.00455612,
         -0.00151261, -0.00249528],
        [ 0.01127287,  0.01001076, -0.00846958, ..., -0.00492209,
         -0.00211061, -0.0024759 ],
        ...,
        [ 0.01170691,  0.01015561, -0.00758872, ..., -0.00463712,
         -0.00208663, -0.00288064],
        [ 0.01136861,  0.00989728, -0.00820845, ..., -0.00463958,
         -0.00203132, -0.00276422],
        [ 0.01130688,  0.01010205, -0.00818974, ..., -0.00466477,
         -0.00208175, -0.00265742]]),
 (246, 128))

In [100]:
e_updated = np.dot(h1_updated_into_df_e_into_w1, avg_clf_w1)
e_updated, e_updated.shape

(array([[ 0.00119752, -0.00133664,  0.00034114, ...,  0.00060182,
         -0.00037011,  0.0030234 ],
        [ 0.00123668, -0.00128755,  0.0003467 , ...,  0.00066208,
         -0.00034416,  0.00308413],
        [ 0.0011676 , -0.00125859,  0.00030382, ...,  0.00056412,
         -0.00039555,  0.00312045],
        ...,
        [ 0.00120978, -0.00128625,  0.00016279, ...,  0.00054207,
         -0.00040756,  0.00312991],
        [ 0.00117397, -0.00128424,  0.00019833, ...,  0.00060921,
         -0.00038286,  0.00314404],
        [ 0.00116711, -0.00128289,  0.00018915, ...,  0.00059852,
         -0.00041336,  0.00312178]]),
 (246, 256))

---

---

In [102]:
common_embeddings_updated = e_updated[:, :128]
male_spf_embeddings_updated = e_updated[male_indices, 128:]
female_spf_embeddings_updated = e_updated[female_indices, 128:]

In [110]:
common_embeddings_updated.shape, male_spf_embeddings_updated.shape, female_spf_embeddings_updated.shape

((246, 128), (141, 128), (105, 128))

In [103]:
avg_cmn_pathwy_node = pd.read_csv(f"Opt_Params/[0825_1]AVG_Common_Pathway_Nodes.csv", header=None).values
avg_cmn_repr_node = pd.read_csv(f"Opt_Params/[0825_1]AVG_Common_Representation_Nodes.csv", header=None).values
avg_cmn_hid_node = pd.read_csv(f"Opt_Params/[0825_1]AVG_Common_Head_Hidden_Nodes.csv", header=None).values
avg_cmn_proj_node = pd.read_csv(f"Opt_Params/[0825_1]AVG_Common_Projection_Nodes.csv", header=None).values

In [104]:
avg_male_pathwy_node = pd.read_csv(f"Opt_Params/[0825_1]AVG_Male_Pathway_Nodes.csv", header=None).values
avg_male_repr_node = pd.read_csv(f"Opt_Params/[0825_1]AVG_Male_Representation_Nodes.csv", header=None).values
avg_male_hid_node = pd.read_csv(f"Opt_Params/[0825_1]AVG_Male_Head_Hidden_Nodes.csv", header=None).values
avg_male_proj_node = pd.read_csv(f"Opt_Params/[0825_1]AVG_Male_Projection_Nodes.csv", header=None).values

In [105]:
avg_female_pathwy_node = pd.read_csv(f"Opt_Params/[0825_1]AVG_Female_Pathway_Nodes.csv", header=None).values
avg_female_repr_node = pd.read_csv(f"Opt_Params/[0825_1]AVG_Female_Representation_Nodes.csv", header=None).values
avg_female_hid_node = pd.read_csv(f"Opt_Params/[0825_1]AVG_Female_Head_Hidden_Nodes.csv", header=None).values
avg_female_proj_node = pd.read_csv(f"Opt_Params/[0825_1]AVG_Female_Projection_Nodes.csv", header=None).values

In [106]:
avg_cmn_w1 = pd.read_csv(f"Opt_Params/[0825_1]AVG_Common_Encoder_Layer1_Weights.csv", header=None).values
avg_cmn_w2 = pd.read_csv(f"Opt_Params/[0825_1]AVG_Common_Encoder_Layer2_Weights.csv", header=None).values
avg_cmn_w3 = pd.read_csv(f"Opt_Params/[0825_1]AVG_Common_Head_Layer1_Weights.csv", header=None).values
avg_cmn_w4 = pd.read_csv(f"Opt_Params/[0825_1]AVG_Common_Head_Layer2_Weights.csv", header=None).values

In [108]:
avg_male_w1 = pd.read_csv(f"Opt_Params/[0825_1]AVG_Male_Encoder_Layer1_Weights.csv", header=None).values
avg_male_w2 = pd.read_csv(f"Opt_Params/[0825_1]AVG_Male_Encoder_Layer2_Weights.csv", header=None).values
avg_male_w3 = pd.read_csv(f"Opt_Params/[0825_1]AVG_Male_Head_Layer1_Weights.csv", header=None).values
avg_male_w4 = pd.read_csv(f"Opt_Params/[0825_1]AVG_Male_Head_Layer2_Weights.csv", header=None).values

In [109]:
avg_female_w1 = pd.read_csv(f"Opt_Params/[0825_1]AVG_Female_Encoder_Layer1_Weights.csv", header=None).values
avg_female_w2 = pd.read_csv(f"Opt_Params/[0825_1]AVG_Female_Encoder_Layer2_Weights.csv", header=None).values
avg_female_w3 = pd.read_csv(f"Opt_Params/[0825_1]AVG_Female_Head_Layer1_Weights.csv", header=None).values
avg_female_w4 = pd.read_csv(f"Opt_Params/[0825_1]AVG_Female_Head_Layer2_Weights.csv", header=None).values

---

## Sex-Common

In [117]:
cmn_pathway_into_w2 = np.dot(avg_cmn_pathwy_node, avg_cmn_w2.T)
cmn_pathway_into_w2, cmn_pathway_into_w2.shape

(array([[-0.26369316,  0.13430116,  0.34880476, ..., -0.46063015,
          0.23599833, -0.32535136],
        [-0.22017677,  0.09049661,  0.41296573, ..., -0.32342878,
          0.23989396, -0.16238887],
        [-0.25056253,  0.13935294,  0.39302642, ..., -0.41885081,
          0.22076233, -0.24288556],
        ...,
        [-0.26178151,  0.19415996,  0.32952593, ..., -0.4111003 ,
          0.25786488, -0.25744428],
        [-0.25182422,  0.18054477,  0.35419873, ..., -0.39904736,
          0.24362266, -0.26501758],
        [-0.21743029,  0.17084628,  0.3095506 , ..., -0.44151363,
          0.2492242 , -0.30336213]]),
 (246, 128))

In [119]:
df_cmn_pathway_into_w2 = dSigmoid(cmn_pathway_into_w2)
df_cmn_pathway_into_w2, df_cmn_pathway_into_w2.shape

(array([[0.24570399, 0.24887608, 0.24254753, ..., 0.23719399, 0.24655111,
         0.24349915],
        [0.24699445, 0.24948885, 0.23963701, ..., 0.24357443, 0.2464374 ,
         0.24835908],
        [0.24611685, 0.24879021, 0.24058886, ..., 0.23934806, 0.24697857,
         0.24634886],
        ...,
        [0.24576535, 0.24765859, 0.24333425, ..., 0.23972784, 0.24588973,
         0.24590298],
        [0.24607805, 0.24797374, 0.24232004, ..., 0.24030587, 0.24632689,
         0.24566123],
        [0.24706838, 0.24818456, 0.24410551, ..., 0.23820177, 0.24615779,
         0.2443353 ]]),
 (246, 128))

In [120]:
e_updated_into_df_cmn_pathway_into_w2 = np.multiply(common_embeddings_updated, df_cmn_pathway_into_w2)
e_updated_into_df_cmn_pathway_into_w2, e_updated_into_df_cmn_pathway_into_w2.shape

(array([[ 2.94235214e-04, -3.32658779e-04,  8.27427961e-05, ...,
          1.19559002e-04, -7.35014869e-04,  9.16817565e-05],
        [ 3.05453414e-04, -3.21230071e-04,  8.30828834e-05, ...,
          1.46328499e-04, -7.52220885e-04,  7.42054443e-05],
        [ 2.87365057e-04, -3.13126030e-04,  7.30956466e-05, ...,
          1.23481729e-04, -7.45008641e-04,  8.45945550e-05],
        ...,
        [ 2.97322403e-04, -3.18551010e-04,  3.96132216e-05, ...,
          1.50268984e-04, -7.50690983e-04,  5.91347456e-05],
        [ 2.88889381e-04, -3.18458138e-04,  4.80581913e-05, ...,
          1.34625121e-04, -7.56010879e-04,  7.97953125e-05],
        [ 2.88355529e-04, -3.18392298e-04,  4.61713736e-05, ...,
          1.32113701e-04, -7.51939681e-04,  7.11431054e-05]]),
 (246, 128))

In [121]:
cmn_pathway_updated = np.dot(e_updated_into_df_cmn_pathway_into_w2, avg_cmn_w2)
cmn_pathway_updated, cmn_pathway_updated.shape

(array([[ 1.12391617e-04, -7.09567896e-05,  1.86930094e-04, ...,
         -2.58332154e-05,  3.32511025e-06, -1.07622630e-05],
        [ 1.16076904e-04, -5.31122422e-05,  1.86701266e-04, ...,
         -2.28636819e-05,  6.17639584e-06, -1.09116806e-05],
        [ 1.11363192e-04, -7.03825559e-05,  1.93957693e-04, ...,
         -3.06456779e-05,  9.36826532e-06, -1.43015123e-05],
        ...,
        [ 1.14271990e-04, -6.95619753e-05,  2.00737161e-04, ...,
         -3.63784643e-05,  5.81112431e-06, -6.99178105e-06],
        [ 1.15577721e-04, -7.03639798e-05,  1.96181347e-04, ...,
         -2.85847930e-05,  6.36697888e-06, -1.12871959e-05],
        [ 1.16221460e-04, -6.85956919e-05,  1.98635635e-04, ...,
         -3.02572293e-05,  8.16454582e-06, -1.28068651e-05]]),
 (246, 395))

---

In [122]:
cmn_pathway_df = pd.DataFrame(cmn_pathway_updated, columns=pathway_list)
cmn_pathway_df

,KEGG_ABC_TRANSPORTERS,KEGG_ACUTE_MYELOID_LEUKEMIA,KEGG_ADHERENS_JUNCTION,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,KEGG_ALDOSTERONE_REGULATED_SODIUM_REABSORPTION,KEGG_ALLOGRAFT_REJECTION,KEGG_ALPHA_LINOLENIC_ACID_METABOLISM,KEGG_ALZHEIMERS_DISEASE,KEGG_AMINOACYL_TRNA_BIOSYNTHESIS,...,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PRKAR1A_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PTCH1_TO_HEDGEHOG_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_RASD1_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_UBQLN2_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_VCP_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_NOTCH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_MGLUR5_CA2_APOPTOTIC_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_PRNP_PI3K_NOX2_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_TRANSPORT_OF_CALCIUM
0,0.000112,-0.000071,0.000187,0.000103,-0.000196,0.000312,0.000106,0.000160,-0.000131,0.000028,...,0.000081,-1.079780e-07,-0.000059,0.000109,0.000026,-0.000158,0.000054,-0.000026,0.000003,-0.000011
1,0.000116,-0.000053,0.000187,0.000091,-0.000187,0.000311,0.000111,0.000155,-0.000124,0.000027,...,0.000084,1.768239e-06,-0.000060,0.000107,0.000011,-0.000147,0.000061,-0.000023,0.000006,-0.000011
2,0.000111,-0.000070,0.000194,0.000103,-0.000208,0.000312,0.000100,0.000160,-0.000136,0.000028,...,0.000082,2.863122e-06,-0.000061,0.000111,0.000027,-0.000155,0.000054,-0.000031,0.000009,-0.000014
3,0.000117,-0.000068,0.000198,0.000098,-0.000203,0.000316,0.000105,0.000164,-0.000134,0.000026,...,0.000086,9.628364e-08,-0.000064,0.000113,0.000029,-0.000163,0.000058,-0.000022,0.000004,-0.000009
4,0.000115,-0.000071,0.000195,0.000103,-0.000206,0.000318,0.000105,0.000165,-0.000135,0.000032,...,0.000079,-1.716539e-06,-0.000060,0.000113,0.000027,-0.000160,0.000052,-0.000033,0.000011,-0.000012
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
241,0.000122,-0.000069,0.000190,0.000095,-0.000199,0.000321,0.000103,0.000164,-0.000135,0.000028,...,0.000079,-1.683155e-06,-0.000063,0.000112,0.000023,-0.000155,0.000054,-0.000039,0.000010,-0.000005
242,0.000110,-0.000065,0.000195,0.000089,-0.000200,0.000308,0.000110,0.000162,-0.000127,0.000024,...,0.000079,1.144773e-06,-0.000061,0.000108,0.000024,-0.000145,0.000057,-0.000031,0.000008,-0.000007
243,0.000114,-0.000070,0.000201,0.000099,-0.000208,0.000315,0.000108,0.000164,-0.000137,0.000028,...,0.000082,4.426949e-06,-0.000064,0.000110,0.000034,-0.000154,0.000057,-0.000036,0.000006,-0.000007
244,0.000116,-0.000070,0.000196,0.000102,-0.000203,0.000317,0.000105,0.000165,-0.000134,0.000027,...,0.000084,1.105756e-06,-0.000061,0.000114,0.000030,-0.000158,0.000055,-0.000029,0.000006,-0.000011


In [124]:
cmn_pathway_df.to_csv(f"[{date}_{num}]Common_Pathway_Importance_Dataframe.csv", header = True, index = False)

---

In [125]:
gene_nodes = x[:, :-1].cpu().detach().numpy()
gene_nodes, gene_nodes.shape

(array([[ 1.77771616e+00,  3.89231533e-01, -1.17751443e+00, ...,
         -1.16429639e+00, -1.44852364e+00, -8.09574306e-01],
        [ 2.33933672e-01,  7.07874417e-01, -7.94603288e-01, ...,
          3.23295236e-01, -5.05196929e-01,  8.18143368e-01],
        [ 7.11468458e-01,  8.24952304e-01, -2.71857411e-01, ...,
         -4.58007276e-01,  1.28614321e-01, -2.13299081e-01],
        ...,
        [-2.18842149e-01,  7.15807080e-01, -8.77743006e-01, ...,
          7.69277275e-01,  8.89907420e-01, -8.37303758e-01],
        [-5.06558299e-01, -1.06509574e-01, -5.08298218e-01, ...,
         -8.36791754e-01, -9.23037827e-01, -4.67300683e-01],
        [-8.48914862e-01, -6.13847747e-04,  2.38007635e-01, ...,
         -1.99058795e+00, -4.88763362e-01, -6.90799892e-01]], dtype=float32),
 (246, 4789))

In [126]:
cmn_gene_into_w1 = np.dot(gene_nodes, avg_cmn_w1.T)
cmn_gene_into_w1, cmn_gene_into_w1.shape

(array([[ 0.12235253,  0.02688699,  0.07895023, ...,  0.04969554,
          0.05694637,  0.0504192 ],
        [-0.02717527,  0.02720193, -0.16744082, ..., -0.04158164,
          0.0087801 , -0.06485891],
        [ 0.02506956,  0.03918368,  0.1084141 , ...,  0.0170871 ,
          0.05707216,  0.00974205],
        ...,
        [ 0.02219523,  0.07329785,  0.13065187, ...,  0.02137843,
          0.02526381,  0.04528632],
        [ 0.01400523,  0.09834776,  0.06567948, ...,  0.01441073,
          0.00162879,  0.04772688],
        [ 0.07881949,  0.04919782,  0.0392882 , ...,  0.01944552,
          0.02760281,  0.06813405]]),
 (246, 395))

In [127]:
df_cmn_gene_into_w1 = dSigmoid(cmn_gene_into_w1)
df_cmn_gene_into_w1, df_cmn_gene_into_w1.shape

(array([[0.2490667 , 0.24995482, 0.24961083, ..., 0.24984571, 0.24979743,
         0.24984119],
        [0.24995385, 0.24995376, 0.24825588, ..., 0.24989197, 0.24999518,
         0.24973727],
        [0.24996072, 0.24990406, 0.24926684, ..., 0.24998175, 0.24979653,
         0.24999407],
        ...,
        [0.24996921, 0.24966451, 0.24893616, ..., 0.24997144, 0.24996011,
         0.24987187],
        [0.24998774, 0.24939646, 0.24973058, ..., 0.24998702, 0.24999983,
         0.24985769],
        [0.24961212, 0.24984878, 0.24990355, ..., 0.24997637, 0.24995239,
         0.24971008]]),
 (246, 395))

In [128]:
p_updated_into_df_cmn_gene_into_w1 = np.multiply(cmn_pathway_updated, df_cmn_gene_into_w1)
p_updated_into_df_cmn_gene_into_w1, p_updated_into_df_cmn_gene_into_w1.shape

(array([[ 2.79930088e-05, -1.77359918e-05,  4.66597764e-05, ...,
         -6.45431806e-06,  8.30603991e-07, -2.68885655e-06],
        [ 2.90138690e-05, -1.32756046e-05,  4.63496869e-05, ...,
         -5.71345043e-06,  1.54406920e-06, -2.72505330e-06],
        [ 2.78364242e-05, -1.75888868e-05,  4.83472205e-05, ...,
         -7.66086029e-06,  2.34016020e-06, -3.57529323e-06],
        ...,
        [ 2.85644794e-05, -1.73671568e-05,  4.99707378e-05, ...,
         -9.09357701e-06,  1.45254929e-06, -1.74704938e-06],
        [ 2.88930135e-05, -1.75485272e-05,  4.89924820e-05, ...,
         -7.14582724e-06,  1.59174366e-06, -2.82019268e-06],
        [ 2.90102851e-05, -1.71385502e-05,  4.96397507e-05, ...,
         -7.56359229e-06,  2.04074771e-06, -3.19800337e-06]]),
 (246, 395))

In [129]:
cmn_gene_updated = np.dot(p_updated_into_df_cmn_gene_into_w1, avg_cmn_w1)
cmn_gene_updated, cmn_gene_updated.shape

(array([[-1.57240889e-07,  1.30886380e-10, -1.42822508e-07, ...,
          7.77683504e-08,  2.49397495e-07, -3.54606189e-08],
        [-1.63192704e-07,  1.35706739e-10, -1.40534592e-07, ...,
          7.40483017e-08,  2.42563981e-07, -2.48143035e-08],
        [-1.60956039e-07,  1.32858421e-10, -1.44367690e-07, ...,
          7.60769857e-08,  2.50099134e-07, -3.22510770e-08],
        ...,
        [-1.74844020e-07,  1.39965590e-10, -1.54294995e-07, ...,
          8.13486290e-08,  2.57772861e-07, -2.63946946e-08],
        [-1.65776503e-07,  1.31577369e-10, -1.45312525e-07, ...,
          8.16568347e-08,  2.58068043e-07, -3.18545623e-08],
        [-1.67049575e-07,  1.42173169e-10, -1.46588029e-07, ...,
          8.25001855e-08,  2.51527402e-07, -3.05862470e-08]]),
 (246, 4789))

In [130]:
data.columns[:-1]

Index(['HMGB1P1', 'A2M', 'AACS', 'AADAT', 'AANAT', 'AARS2', 'AASDHPPT',
       'AASDH', 'AASS', 'ABAT',
       ...
       'ZFYVE9', 'ZIC2', 'ZMAT2', 'ZMAT3', 'ZNF274', 'ZNRF3', 'ZW10', 'ZWILCH',
       'ZWINT', 'ZYX'],
      dtype='object', length=4789)

In [131]:
cmn_gene_df = pd.DataFrame(cmn_gene_updated, columns=data.columns[:-1])
cmn_gene_df

,HMGB1P1,A2M,AACS,AADAT,AANAT,AARS2,AASDHPPT,AASDH,AASS,ABAT,...,ZFYVE9,ZIC2,ZMAT2,ZMAT3,ZNF274,ZNRF3,ZW10,ZWILCH,ZWINT,ZYX
0,-1.572409e-07,1.308864e-10,-1.428225e-07,-4.024398e-07,-9.694059e-09,-1.592726e-09,-3.446902e-08,-7.421311e-07,6.649394e-07,-8.168771e-07,...,-3.053214e-08,4.145366e-08,-2.932781e-08,1.901506e-08,-1.461214e-07,6.285771e-09,1.754992e-07,7.776835e-08,2.493975e-07,-3.546062e-08
1,-1.631927e-07,1.357067e-10,-1.405346e-07,-3.861722e-07,-1.000760e-08,-1.529972e-09,-3.389818e-08,-7.298408e-07,6.539274e-07,-8.149336e-07,...,-3.126189e-08,6.509326e-08,-3.232577e-08,1.732916e-08,-1.378858e-07,8.452137e-09,1.671042e-07,7.404830e-08,2.425640e-07,-2.481430e-08
2,-1.609560e-07,1.328584e-10,-1.443677e-07,-4.119727e-07,-9.221083e-09,-1.544799e-09,-3.446627e-08,-7.420720e-07,6.648865e-07,-8.255333e-07,...,-3.055070e-08,4.744472e-08,-2.995696e-08,1.856484e-08,-1.430642e-07,6.131696e-09,1.716823e-07,7.607699e-08,2.500991e-07,-3.225108e-08
3,-1.697293e-07,1.370235e-10,-1.466823e-07,-4.160705e-07,-9.425442e-09,-1.439584e-09,-3.494044e-08,-7.522810e-07,6.740335e-07,-8.422985e-07,...,-3.195782e-08,5.289466e-08,-3.460804e-08,1.650757e-08,-1.450545e-07,7.270973e-09,1.852241e-07,8.207774e-08,2.546801e-07,-2.182655e-08
4,-1.619695e-07,1.385700e-10,-1.362884e-07,-3.933535e-07,-9.307880e-09,-1.781716e-09,-3.349569e-08,-7.211751e-07,6.461631e-07,-8.117805e-07,...,-3.093460e-08,3.251402e-08,-3.220633e-08,1.702684e-08,-1.468351e-07,6.253101e-09,1.789269e-07,7.928729e-08,2.391218e-07,-3.651810e-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
241,-1.725463e-07,9.325295e-11,-1.258283e-07,-4.181908e-07,-8.340723e-09,-1.587638e-09,-3.379771e-08,-7.276776e-07,6.519893e-07,-7.678357e-07,...,-3.140589e-08,7.208366e-08,-3.225540e-08,1.805860e-08,-1.501580e-07,6.468860e-09,1.690839e-07,7.492556e-08,2.457643e-07,-3.321745e-08
242,-1.631375e-07,1.469547e-10,-1.396224e-07,-3.875112e-07,-9.501811e-09,-1.323327e-09,-3.338551e-08,-7.188029e-07,6.440376e-07,-8.159623e-07,...,-3.091795e-08,5.263204e-08,-3.060873e-08,1.697772e-08,-1.435197e-07,6.944685e-09,1.715935e-07,7.603767e-08,2.453299e-07,-2.770432e-08
243,-1.748440e-07,1.399656e-10,-1.542950e-07,-4.208745e-07,-8.931765e-09,-1.550533e-09,-3.464135e-08,-7.458415e-07,6.682638e-07,-8.687987e-07,...,-3.090543e-08,5.697609e-08,-3.204529e-08,1.707305e-08,-1.445101e-07,6.396599e-09,1.835787e-07,8.134863e-08,2.577729e-07,-2.639469e-08
244,-1.657765e-07,1.315774e-10,-1.453125e-07,-4.119418e-07,-9.138313e-09,-1.508004e-09,-3.436799e-08,-7.399559e-07,6.629904e-07,-8.372546e-07,...,-3.107222e-08,4.562949e-08,-3.211573e-08,1.807357e-08,-1.455206e-07,6.564548e-09,1.842743e-07,8.165683e-08,2.580680e-07,-3.185456e-08


In [132]:
cmn_gene_df.to_csv(f"[{date}_{num}]Common_Gene_Importance_Dataframe.csv", header = True, index = False)

---

---

## Male-Specific

In [133]:
male_pathway_into_w2 = np.dot(avg_male_pathwy_node, avg_male_w2.T)
male_pathway_into_w2, male_pathway_into_w2.shape

(array([[ 0.33134563, -0.1840296 ,  0.22569108, ..., -0.09111415,
          0.00289943, -0.40600286],
        [ 0.38671782, -0.18283196,  0.26158462, ..., -0.21447257,
          0.05320038, -0.60613396],
        [ 0.37979272, -0.17000802,  0.21904593, ..., -0.2406615 ,
          0.02539876, -0.56994177],
        ...,
        [ 0.39128281, -0.19268414,  0.25750545, ..., -0.23445567,
          0.00177683, -0.55191595],
        [ 0.40546386, -0.17806582,  0.2377021 , ..., -0.22934152,
          0.02444947, -0.5586467 ],
        [ 0.36304291, -0.1641166 ,  0.23148319, ..., -0.22044623,
          0.02279248, -0.5751016 ]]),
 (141, 128))

In [134]:
df_male_pathway_into_w2 = dSigmoid(male_pathway_into_w2)
df_male_pathway_into_w2, df_male_pathway_into_w2.shape

(array([[0.24326176, 0.24789521, 0.2468433 , ..., 0.24948186, 0.24999947,
         0.23997417],
        [0.24088121, 0.24792236, 0.24577165, ..., 0.24714699, 0.24982319,
         0.22837383],
        [0.24119722, 0.24820225, 0.247025  , ..., 0.24641479, 0.24995969,
         0.23074851],
        ...,
        [0.24067009, 0.24769383, 0.24590106, ..., 0.24659564, 0.2499998 ,
         0.23188823],
        [0.24000006, 0.24802871, 0.2465016 , ..., 0.24674126, 0.24996264,
         0.23146608],
        [0.24194012, 0.24832414, 0.24668065, ..., 0.24698715, 0.24996753,
         0.23041693]]),
 (141, 128))

In [136]:
e_updated_into_df_male_pathway_into_w2 = np.multiply(male_spf_embeddings_updated, df_male_pathway_into_w2)
e_updated_into_df_male_pathway_into_w2, e_updated_into_df_male_pathway_into_w2.shape

(array([[-1.05598301e-04,  2.53769720e-04,  6.11777189e-04, ...,
          1.65176091e-04, -8.60407617e-05,  7.40110633e-04],
        [-1.23653581e-04,  2.21793224e-04,  6.16776776e-04, ...,
          1.42387483e-04, -8.98815177e-05,  7.17660458e-04],
        [-1.52172221e-04,  2.36511490e-04,  6.60611436e-04, ...,
          1.43685002e-04, -6.95719472e-05,  7.28117054e-04],
        ...,
        [-1.39332600e-04,  2.20133915e-04,  5.93272669e-04, ...,
          1.33672530e-04, -1.01889120e-04,  7.25788453e-04],
        [-1.27013607e-04,  2.24711020e-04,  6.14086211e-04, ...,
          1.50318084e-04, -9.57008047e-05,  7.27738893e-04],
        [-1.29358830e-04,  2.21817835e-04,  6.23008280e-04, ...,
          1.47826193e-04, -1.03327159e-04,  7.19311568e-04]]),
 (141, 128))

In [137]:
male_pathway_updated = np.dot(e_updated_into_df_male_pathway_into_w2, avg_male_w2)
male_pathway_updated, male_pathway_updated.shape

(array([[-1.32495234e-04,  3.34052320e-05,  2.31061548e-05, ...,
          5.48062097e-05,  1.64268291e-04,  8.81482613e-05],
        [-1.41200483e-04,  2.85063511e-05,  2.45928195e-05, ...,
          5.71423434e-05,  1.67326795e-04,  9.60741991e-05],
        [-1.44893363e-04,  2.55884947e-05,  2.50225235e-05, ...,
          6.77138080e-05,  1.68066380e-04,  9.86232278e-05],
        ...,
        [-1.39575391e-04,  3.03906909e-05,  2.54739118e-05, ...,
          6.18536050e-05,  1.58844724e-04,  9.42598768e-05],
        [-1.41711063e-04,  2.53112757e-05,  2.33973701e-05, ...,
          6.38841332e-05,  1.68063900e-04,  9.68162839e-05],
        [-1.41290418e-04,  2.63303062e-05,  2.54396731e-05, ...,
          6.52686714e-05,  1.65591130e-04,  9.67604796e-05]]),
 (141, 395))

---

In [138]:
male_pathway_df = pd.DataFrame(male_pathway_updated, columns=pathway_list)
male_pathway_df

,KEGG_ABC_TRANSPORTERS,KEGG_ACUTE_MYELOID_LEUKEMIA,KEGG_ADHERENS_JUNCTION,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,KEGG_ALDOSTERONE_REGULATED_SODIUM_REABSORPTION,KEGG_ALLOGRAFT_REJECTION,KEGG_ALPHA_LINOLENIC_ACID_METABOLISM,KEGG_ALZHEIMERS_DISEASE,KEGG_AMINOACYL_TRNA_BIOSYNTHESIS,...,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PRKAR1A_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PTCH1_TO_HEDGEHOG_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_RASD1_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_UBQLN2_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_VCP_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_NOTCH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_MGLUR5_CA2_APOPTOTIC_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_PRNP_PI3K_NOX2_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_TRANSPORT_OF_CALCIUM
0,-0.000132,0.000033,0.000023,-0.000082,-0.000085,0.000189,0.000035,0.000070,0.000140,0.000086,...,3.836328e-07,-0.000247,-0.000027,0.000060,-0.000087,0.000048,-0.000038,0.000055,0.000164,0.000088
1,-0.000141,0.000029,0.000025,-0.000078,-0.000081,0.000197,0.000039,0.000069,0.000141,0.000091,...,-5.729461e-07,-0.000244,-0.000026,0.000071,-0.000091,0.000053,-0.000034,0.000057,0.000167,0.000096
2,-0.000145,0.000026,0.000025,-0.000080,-0.000084,0.000199,0.000043,0.000066,0.000149,0.000092,...,1.943093e-06,-0.000254,-0.000025,0.000067,-0.000089,0.000048,-0.000031,0.000068,0.000168,0.000099
3,-0.000140,0.000035,0.000023,-0.000088,-0.000078,0.000193,0.000036,0.000068,0.000140,0.000092,...,1.140293e-05,-0.000249,-0.000026,0.000064,-0.000088,0.000049,-0.000039,0.000064,0.000166,0.000090
4,-0.000132,0.000029,0.000027,-0.000080,-0.000083,0.000194,0.000037,0.000074,0.000134,0.000091,...,7.041079e-06,-0.000246,-0.000027,0.000070,-0.000087,0.000046,-0.000034,0.000060,0.000157,0.000097
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136,-0.000138,0.000029,0.000023,-0.000082,-0.000084,0.000197,0.000042,0.000074,0.000143,0.000091,...,8.639964e-06,-0.000245,-0.000029,0.000064,-0.000090,0.000045,-0.000035,0.000061,0.000166,0.000090
137,-0.000138,0.000030,0.000028,-0.000081,-0.000081,0.000196,0.000043,0.000063,0.000149,0.000084,...,3.756694e-06,-0.000244,-0.000028,0.000061,-0.000088,0.000045,-0.000041,0.000063,0.000156,0.000095
138,-0.000140,0.000030,0.000025,-0.000083,-0.000080,0.000201,0.000047,0.000070,0.000145,0.000092,...,8.417630e-06,-0.000249,-0.000023,0.000072,-0.000091,0.000048,-0.000040,0.000062,0.000159,0.000094
139,-0.000142,0.000025,0.000023,-0.000082,-0.000080,0.000199,0.000043,0.000071,0.000147,0.000095,...,5.150428e-06,-0.000250,-0.000027,0.000069,-0.000088,0.000049,-0.000038,0.000064,0.000168,0.000097


In [139]:
male_pathway_df.to_csv(f"[{date}_{num}]Male_Pathway_Importance_Dataframe.csv", header = True, index = False)

---

In [141]:
gene_nodes = x[:, :-1].cpu().detach().numpy()
gene_nodes, gene_nodes.shape

(array([[ 1.77771616e+00,  3.89231533e-01, -1.17751443e+00, ...,
         -1.16429639e+00, -1.44852364e+00, -8.09574306e-01],
        [ 2.33933672e-01,  7.07874417e-01, -7.94603288e-01, ...,
          3.23295236e-01, -5.05196929e-01,  8.18143368e-01],
        [ 7.11468458e-01,  8.24952304e-01, -2.71857411e-01, ...,
         -4.58007276e-01,  1.28614321e-01, -2.13299081e-01],
        ...,
        [-2.18842149e-01,  7.15807080e-01, -8.77743006e-01, ...,
          7.69277275e-01,  8.89907420e-01, -8.37303758e-01],
        [-5.06558299e-01, -1.06509574e-01, -5.08298218e-01, ...,
         -8.36791754e-01, -9.23037827e-01, -4.67300683e-01],
        [-8.48914862e-01, -6.13847747e-04,  2.38007635e-01, ...,
         -1.99058795e+00, -4.88763362e-01, -6.90799892e-01]], dtype=float32),
 (246, 4789))

In [142]:
gene_nodes[male_indices], gene_nodes[male_indices].shape

(array([[ 2.33933672e-01,  7.07874417e-01, -7.94603288e-01, ...,
          3.23295236e-01, -5.05196929e-01,  8.18143368e-01],
        [ 1.79109168e+00,  7.41434842e-03, -3.03278774e-01, ...,
          2.56283212e+00,  2.37311125e+00, -2.83269674e-01],
        [ 1.15760398e+00, -1.12626299e-01, -3.42409074e-01, ...,
         -1.74839079e+00, -1.31092882e+00, -7.05311656e-01],
        ...,
        [-2.18842149e-01,  7.15807080e-01, -8.77743006e-01, ...,
          7.69277275e-01,  8.89907420e-01, -8.37303758e-01],
        [-5.06558299e-01, -1.06509574e-01, -5.08298218e-01, ...,
         -8.36791754e-01, -9.23037827e-01, -4.67300683e-01],
        [-8.48914862e-01, -6.13847747e-04,  2.38007635e-01, ...,
         -1.99058795e+00, -4.88763362e-01, -6.90799892e-01]], dtype=float32),
 (141, 4789))

In [143]:
male_gene_into_w1 = np.dot(gene_nodes[male_indices], avg_male_w1.T)
male_gene_into_w1, male_gene_into_w1.shape

(array([[-0.00493643, -0.10424093, -0.04093749, ..., -0.02652473,
         -0.00191192, -0.04188421],
        [ 0.0192274 ,  0.16508973,  0.04661284, ...,  0.01761714,
          0.00264175, -0.01265506],
        [-0.07732226,  0.03213375,  0.0500045 , ...,  0.00504272,
          0.01201911, -0.00464098],
        ...,
        [ 0.00777504,  0.11115957,  0.03874365, ...,  0.02451749,
          0.00897933,  0.00055898],
        [ 0.04858785,  0.06652381,  0.0664733 , ...,  0.02993922,
          0.0131316 ,  0.02243104],
        [-0.0127893 , -0.03895041,  0.05226962, ...,  0.02368206,
          0.00970148,  0.01922887]]),
 (141, 395))

In [146]:
df_male_gene_into_w1 = dSigmoid(male_gene_into_w1)
df_male_gene_into_w1, df_male_gene_into_w1.shape

(array([[0.24999848, 0.24932209, 0.24989529, ..., 0.24995603, 0.24999977,
         0.24989039],
        [0.2499769 , 0.24830429, 0.24986425, ..., 0.2499806 , 0.24999956,
         0.24998999],
        [0.2496267 , 0.24993547, 0.24984379, ..., 0.24999841, 0.24999097,
         0.24999865],
        ...,
        [0.24999622, 0.24922931, 0.24990621, ..., 0.24996243, 0.24999496,
         0.24999998],
        [0.24985251, 0.24972362, 0.24972403, ..., 0.24994399, 0.24998922,
         0.24996856],
        [0.24998978, 0.2499052 , 0.24982932, ..., 0.24996495, 0.24999412,
         0.24997689]]),
 (141, 395))

In [147]:
p_updated_into_df_male_gene_into_w1 = np.multiply(male_pathway_updated, df_male_gene_into_w1)
p_updated_into_df_male_gene_into_w1, p_updated_into_df_male_gene_into_w1.shape

(array([[-3.31236068e-05,  8.32866235e-06,  5.77411918e-06, ...,
          1.36991427e-05,  4.10670351e-05,  2.20274033e-05],
        [-3.52968585e-05,  7.07824939e-06,  6.14486644e-06, ...,
          1.42844775e-05,  4.18316258e-05,  2.40175881e-05],
        [-3.61692522e-05,  6.39547258e-06,  6.25172203e-06, ...,
          1.69283444e-05,  4.20150775e-05,  2.46556742e-05],
        ...,
        [-3.48933204e-05,  7.57425092e-06,  6.36608868e-06, ...,
          1.54610777e-05,  3.97103805e-05,  2.35649674e-05],
        [-3.54068648e-05,  6.32082328e-06,  5.84288565e-06, ...,
          1.59674549e-05,  4.20141637e-05,  2.42010266e-05],
        [-3.53211602e-05,  6.58008053e-06,  6.35557625e-06, ...,
          1.63148802e-05,  4.13968084e-05,  2.41878840e-05]]),
 (141, 395))

In [148]:
male_gene_updated = np.dot(p_updated_into_df_male_gene_into_w1, avg_male_w1)
male_gene_updated, male_gene_updated.shape

(array([[-6.25079122e-07, -5.67281273e-09,  6.92317887e-09, ...,
         -1.63405974e-07,  3.21545111e-08,  7.38266336e-08],
        [-6.24714897e-07, -3.10621471e-09,  6.62071880e-09, ...,
         -1.70767121e-07,  3.22873145e-08,  7.65613243e-08],
        [-6.28534504e-07, -5.08306172e-09,  7.00902897e-09, ...,
         -1.63921589e-07,  3.29178205e-08,  7.81369345e-08],
        ...,
        [-6.29292079e-07, -8.27666637e-09,  6.26148175e-09, ...,
         -1.92774238e-07,  3.23955691e-08,  8.02525196e-08],
        [-6.24420544e-07, -5.98502955e-09,  6.91390231e-09, ...,
         -1.79510410e-07,  3.25487600e-08,  7.84422387e-08],
        [-6.28145823e-07, -4.86864631e-09,  6.40714517e-09, ...,
         -1.89784970e-07,  3.22536649e-08,  7.70853344e-08]]),
 (141, 4789))

In [149]:
male_gene_df = pd.DataFrame(male_gene_updated, columns=data.columns[:-1])
male_gene_df

,HMGB1P1,A2M,AACS,AADAT,AANAT,AARS2,AASDHPPT,AASDH,AASS,ABAT,...,ZFYVE9,ZIC2,ZMAT2,ZMAT3,ZNF274,ZNRF3,ZW10,ZWILCH,ZWINT,ZYX
0,-6.250791e-07,-5.672813e-09,6.923179e-09,-6.211402e-08,-2.396544e-07,-2.276046e-07,-5.478589e-07,9.701522e-08,4.033995e-07,-1.782961e-07,...,8.728684e-08,1.478508e-07,6.002200e-08,2.058850e-07,1.173557e-07,-2.766050e-07,1.333146e-07,-1.634060e-07,3.215451e-08,7.382663e-08
1,-6.247149e-07,-3.106215e-09,6.620719e-09,-6.759893e-08,-2.357103e-07,-2.384544e-07,-5.914177e-07,1.047287e-07,4.354728e-07,-1.900802e-07,...,9.047989e-08,1.298457e-07,5.167089e-08,2.111320e-07,1.122433e-07,-2.843229e-07,1.393202e-07,-1.707671e-07,3.228731e-08,7.656132e-08
2,-6.285345e-07,-5.083062e-09,7.009029e-09,-6.751605e-08,-2.338888e-07,-2.418922e-07,-5.903985e-07,1.045482e-07,4.347223e-07,-1.847194e-07,...,9.195165e-08,1.404341e-07,5.851587e-08,2.078595e-07,1.100720e-07,-2.670476e-07,1.337352e-07,-1.639216e-07,3.291782e-08,7.813693e-08
3,-6.251969e-07,-3.630910e-09,6.830019e-09,-6.369154e-08,-2.440345e-07,-2.415967e-07,-5.614454e-07,9.942113e-08,4.134035e-07,-2.010746e-07,...,9.157572e-08,1.376055e-07,5.637221e-08,2.097452e-07,1.121494e-07,-2.827464e-07,1.398664e-07,-1.714366e-07,3.233278e-08,7.733524e-08
4,-6.222525e-07,-5.741904e-09,5.662533e-09,-6.322551e-08,-2.350166e-07,-2.408771e-07,-5.559490e-07,9.844783e-08,4.093565e-07,-1.700842e-07,...,8.973872e-08,1.387313e-07,5.522175e-08,2.122231e-07,1.098607e-07,-2.701180e-07,1.383347e-07,-1.695592e-07,3.149339e-08,7.895552e-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136,-6.205690e-07,-4.743541e-09,6.528122e-09,-6.418947e-08,-2.396465e-07,-2.390791e-07,-5.646262e-07,9.998439e-08,4.157456e-07,-1.908154e-07,...,8.935086e-08,1.357988e-07,5.735156e-08,2.103476e-07,1.168029e-07,-2.833804e-07,1.463733e-07,-1.794122e-07,3.192181e-08,7.718810e-08
137,-6.127714e-07,-3.484433e-09,6.343067e-09,-6.419562e-08,-2.326163e-07,-2.209736e-07,-5.633264e-07,9.975422e-08,4.147886e-07,-1.793506e-07,...,8.787157e-08,1.373754e-07,6.507361e-08,2.019867e-07,1.177443e-07,-2.672790e-07,1.390942e-07,-1.704901e-07,3.234010e-08,7.610453e-08
138,-6.292921e-07,-8.276666e-09,6.261482e-09,-6.677714e-08,-2.208956e-07,-2.415386e-07,-5.819343e-07,1.030493e-07,4.284900e-07,-1.890310e-07,...,9.083551e-08,1.190301e-07,5.799908e-08,2.114922e-07,1.152296e-07,-2.830260e-07,1.572746e-07,-1.927742e-07,3.239557e-08,8.025252e-08
139,-6.244205e-07,-5.985030e-09,6.913902e-09,-6.643939e-08,-2.410528e-07,-2.503159e-07,-5.830746e-07,1.032512e-07,4.293296e-07,-1.852491e-07,...,9.109623e-08,1.257767e-07,6.190167e-08,2.074964e-07,1.124434e-07,-2.814423e-07,1.464534e-07,-1.795104e-07,3.254876e-08,7.844224e-08


In [150]:
male_gene_df.to_csv(f"[{date}_{num}]Male_Gene_Importance_Dataframe.csv", header = True, index = False)

---

---

## Female-Specific

In [151]:
female_pathway_into_w2 = np.dot(avg_female_pathwy_node, avg_female_w2.T)
female_pathway_into_w2, female_pathway_into_w2.shape

(array([[-0.04328333, -0.13549766, -0.28260269, ...,  0.20243666,
          0.04069014, -0.12068168],
        [-0.07895945, -0.10958549, -0.28030484, ...,  0.21915799,
          0.00449788, -0.08020124],
        [-0.02982777, -0.10064855, -0.31174542, ...,  0.19271022,
          0.05682433, -0.10868924],
        ...,
        [-0.11655625, -0.13452587, -0.32115928, ...,  0.19762109,
          0.04571105, -0.06800227],
        [-0.03626637, -0.10760507, -0.27064938, ...,  0.26942403,
         -0.02117162, -0.09665203],
        [-0.03874812, -0.11527942, -0.32665552, ...,  0.16975299,
         -0.0269165 , -0.09207529]]),
 (105, 128))

In [152]:
df_female_pathway_into_w2 = dSigmoid(female_pathway_into_w2)
df_female_pathway_into_w2, df_female_pathway_into_w2.shape

(array([[0.24988295, 0.24885603, 0.24507418, ..., 0.24745611, 0.24989655,
         0.24909195],
        [0.24961074, 0.24925094, 0.24515292, ..., 0.24702198, 0.24999874,
         0.24959842],
        [0.2499444 , 0.24936793, 0.24402297, ..., 0.24769321, 0.2497983 ,
         0.24926312],
        ...,
        [0.24915283, 0.24887233, 0.24366277, ..., 0.24757492, 0.24986945,
         0.2497112 ],
        [0.24991781, 0.24927772, 0.24547713, ..., 0.2455175 , 0.24997199,
         0.24941706],
        [0.24990618, 0.24917125, 0.24344784, ..., 0.24820761, 0.24995472,
         0.24947088]]),
 (105, 128))

In [153]:
e_updated_into_df_female_pathway_into_w2 = np.multiply(female_spf_embeddings_updated, df_female_pathway_into_w2)
e_updated_into_df_female_pathway_into_w2, e_updated_into_df_female_pathway_into_w2.shape

(array([[-1.35109402e-04,  2.48396600e-04,  6.17216676e-04, ...,
          1.48923160e-04, -9.24899632e-05,  7.53103931e-04],
        [-1.51992746e-04,  2.28931773e-04,  6.23083632e-04, ...,
          1.39349847e-04, -9.88870681e-05,  7.78859676e-04],
        [-1.63496738e-04,  2.10156800e-04,  6.07544775e-04, ...,
          1.39712388e-04, -9.10149202e-05,  7.92771742e-04],
        ...,
        [-1.76329267e-04,  2.08384903e-04,  6.07894765e-04, ...,
          1.04035316e-04, -7.94333434e-05,  8.16716896e-04],
        [-1.56917952e-04,  1.90365068e-04,  5.91965252e-04, ...,
          1.23306361e-04, -9.09242954e-05,  8.00008309e-04],
        [-1.98249419e-04,  2.52835039e-04,  6.26935931e-04, ...,
          1.39660565e-04, -7.40374696e-05,  8.03349968e-04]]),
 (105, 128))

In [154]:
female_pathway_updated = np.dot(e_updated_into_df_female_pathway_into_w2, avg_female_w2)
female_pathway_updated, female_pathway_updated.shape

(array([[-4.45049870e-05,  1.34489157e-04, -3.39071423e-05, ...,
          4.35965498e-05, -6.20598097e-05, -3.86674716e-05],
        [-4.15618417e-05,  1.34146486e-04, -3.81798152e-05, ...,
          4.84018984e-05, -6.36443555e-05, -4.46108344e-05],
        [-3.74464862e-05,  1.41789185e-04, -4.28703312e-05, ...,
          4.82609344e-05, -6.68675760e-05, -4.40199388e-05],
        ...,
        [-4.13740451e-05,  1.31445393e-04, -4.66232924e-05, ...,
          4.97501858e-05, -6.95963071e-05, -4.82068815e-05],
        [-3.84521866e-05,  1.33221957e-04, -4.91676679e-05, ...,
          4.61856556e-05, -6.60063592e-05, -4.28249353e-05],
        [-5.00525772e-05,  1.37148438e-04, -3.96181618e-05, ...,
          3.82786117e-05, -6.77296055e-05, -4.08976292e-05]]),
 (105, 395))

---

In [155]:
female_pathway_df = pd.DataFrame(female_pathway_updated, columns=pathway_list)
female_pathway_df

,KEGG_ABC_TRANSPORTERS,KEGG_ACUTE_MYELOID_LEUKEMIA,KEGG_ADHERENS_JUNCTION,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,KEGG_ALDOSTERONE_REGULATED_SODIUM_REABSORPTION,KEGG_ALLOGRAFT_REJECTION,KEGG_ALPHA_LINOLENIC_ACID_METABOLISM,KEGG_ALZHEIMERS_DISEASE,KEGG_AMINOACYL_TRNA_BIOSYNTHESIS,...,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PRKAR1A_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PTCH1_TO_HEDGEHOG_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_RASD1_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_UBQLN2_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_VCP_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_NOTCH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_MGLUR5_CA2_APOPTOTIC_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_PRNP_PI3K_NOX2_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_TRANSPORT_OF_CALCIUM
0,-0.000045,0.000134,-0.000034,-0.000372,0.000134,-0.000017,-0.000020,-6.479428e-06,0.000006,-0.000122,...,0.000083,-0.000065,-0.000134,0.000010,-0.000051,0.000031,-0.000144,0.000044,-0.000062,-0.000039
1,-0.000042,0.000134,-0.000038,-0.000371,0.000140,-0.000018,-0.000020,-5.266082e-07,0.000005,-0.000125,...,0.000088,-0.000063,-0.000134,0.000012,-0.000053,0.000030,-0.000149,0.000048,-0.000064,-0.000045
2,-0.000037,0.000142,-0.000043,-0.000384,0.000144,-0.000022,-0.000022,-5.507924e-06,0.000008,-0.000134,...,0.000093,-0.000071,-0.000137,0.000009,-0.000051,0.000032,-0.000164,0.000048,-0.000067,-0.000044
3,-0.000039,0.000136,-0.000052,-0.000380,0.000140,-0.000026,-0.000021,-4.372620e-06,0.000006,-0.000136,...,0.000093,-0.000073,-0.000137,0.000011,-0.000049,0.000034,-0.000159,0.000049,-0.000064,-0.000045
4,-0.000046,0.000135,-0.000029,-0.000363,0.000124,-0.000005,-0.000013,-1.846329e-05,0.000012,-0.000124,...,0.000067,-0.000058,-0.000136,0.000007,-0.000042,0.000035,-0.000139,0.000046,-0.000054,-0.000032
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,-0.000046,0.000132,-0.000043,-0.000382,0.000136,-0.000027,-0.000020,-3.352919e-06,0.000010,-0.000133,...,0.000090,-0.000066,-0.000143,0.000010,-0.000053,0.000031,-0.000155,0.000049,-0.000063,-0.000048
101,-0.000048,0.000134,-0.000033,-0.000368,0.000132,-0.000011,-0.000017,-1.157617e-05,0.000009,-0.000126,...,0.000083,-0.000069,-0.000135,0.000008,-0.000051,0.000029,-0.000143,0.000048,-0.000064,-0.000037
102,-0.000041,0.000131,-0.000047,-0.000378,0.000142,-0.000019,-0.000017,-4.639647e-06,0.000009,-0.000132,...,0.000095,-0.000070,-0.000139,0.000003,-0.000050,0.000030,-0.000157,0.000050,-0.000070,-0.000048
103,-0.000038,0.000133,-0.000049,-0.000380,0.000143,-0.000022,-0.000025,-7.279073e-06,0.000011,-0.000134,...,0.000091,-0.000070,-0.000137,0.000008,-0.000050,0.000033,-0.000159,0.000046,-0.000066,-0.000043


In [156]:
female_pathway_df.to_csv(f"[{date}_{num}]Female_Pathway_Importance_Dataframe.csv", header = True, index = False)

---

In [157]:
gene_nodes = x[:, :-1].cpu().detach().numpy()
gene_nodes, gene_nodes.shape

(array([[ 1.77771616e+00,  3.89231533e-01, -1.17751443e+00, ...,
         -1.16429639e+00, -1.44852364e+00, -8.09574306e-01],
        [ 2.33933672e-01,  7.07874417e-01, -7.94603288e-01, ...,
          3.23295236e-01, -5.05196929e-01,  8.18143368e-01],
        [ 7.11468458e-01,  8.24952304e-01, -2.71857411e-01, ...,
         -4.58007276e-01,  1.28614321e-01, -2.13299081e-01],
        ...,
        [-2.18842149e-01,  7.15807080e-01, -8.77743006e-01, ...,
          7.69277275e-01,  8.89907420e-01, -8.37303758e-01],
        [-5.06558299e-01, -1.06509574e-01, -5.08298218e-01, ...,
         -8.36791754e-01, -9.23037827e-01, -4.67300683e-01],
        [-8.48914862e-01, -6.13847747e-04,  2.38007635e-01, ...,
         -1.99058795e+00, -4.88763362e-01, -6.90799892e-01]], dtype=float32),
 (246, 4789))

In [158]:
gene_nodes[female_indices], gene_nodes[female_indices].shape

(array([[ 1.7777162 ,  0.38923153, -1.1775144 , ..., -1.1642964 ,
         -1.4485236 , -0.8095743 ],
        [ 0.71146846,  0.8249523 , -0.2718574 , ..., -0.45800728,
          0.12861432, -0.21329908],
        [ 0.28942788, -0.08987974, -0.10909093, ...,  0.36037305,
          1.1215334 , -0.44160473],
        ...,
        [-1.8018974 ,  1.841666  , -0.67077196, ..., -1.5048593 ,
         -0.9997603 ,  0.6268103 ],
        [ 0.7001799 ,  2.0940764 , -0.80445904, ..., -0.07264604,
          0.27525246,  0.27501622],
        [-0.39945918, -1.4817204 ,  2.8892083 , ...,  0.28612214,
          0.13211496, -0.4700736 ]], dtype=float32),
 (105, 4789))

In [159]:
female_gene_into_w1 = np.dot(gene_nodes[female_indices], avg_female_w1.T)
female_gene_into_w1, female_gene_into_w1.shape

(array([[-1.73148623e-03, -2.03627220e-03, -4.37557443e-02, ...,
          2.30775944e-02,  5.47389964e-03, -4.63585861e-03],
        [-1.03709083e-02,  3.16494768e-03,  1.68262739e-02, ...,
         -1.46942380e-05,  1.41933067e-03,  3.77446621e-04],
        [-2.09274446e-02,  4.02340652e-02,  8.38203840e-03, ...,
         -4.33938197e-03,  3.69665478e-02,  8.49253883e-03],
        ...,
        [ 6.49894168e-02, -1.26891262e-03,  1.62537903e-01, ...,
          2.41172143e-02, -3.53637605e-02,  3.51196453e-02],
        [-7.23899936e-02, -4.35383698e-02, -9.69000408e-03, ...,
          1.63727694e-02, -4.76082848e-02,  2.25112325e-02],
        [ 1.59675736e-02, -7.79254152e-03,  2.58646199e-02, ...,
          4.58415511e-02, -1.12440501e-02,  1.99787975e-02]]),
 (105, 395))

In [160]:
df_female_gene_into_w1 = dSigmoid(female_gene_into_w1)
df_female_gene_into_w1, df_female_gene_into_w1.shape

(array([[0.24999981, 0.24999974, 0.24988038, ..., 0.24996672, 0.24999813,
         0.24999866],
        [0.24999328, 0.24999937, 0.24998231, ..., 0.25      , 0.24999987,
         0.24999999],
        [0.24997263, 0.24989885, 0.24999561, ..., 0.24999882, 0.24991461,
         0.24999549],
        ...,
        [0.24973621, 0.2499999 , 0.24835608, ..., 0.24996365, 0.24992185,
         0.24992293],
        [0.24967277, 0.24988156, 0.24999413, ..., 0.24998325, 0.24985839,
         0.24996833],
        [0.24998407, 0.2499962 , 0.24995819, ..., 0.24986871, 0.2499921 ,
         0.24997505]]),
 (105, 395))

In [161]:
p_updated_into_df_female_gene_into_w1 = np.multiply(female_pathway_updated, df_female_gene_into_w1)
p_updated_into_df_female_gene_into_w1, p_updated_into_df_female_gene_into_w1.shape

(array([[-1.11262384e-05,  3.36222545e-05, -8.47272953e-06, ...,
          1.08976864e-05, -1.55148362e-05, -9.66681595e-06],
        [-1.03901811e-05,  3.35365374e-05, -9.54427823e-06, ...,
          1.21004746e-05, -1.59110809e-05, -1.11527082e-05],
        [-9.36059662e-06,  3.54329548e-05, -1.07173945e-05, ...,
          1.20651768e-05, -1.67111843e-05, -1.10047863e-05],
        ...,
        [-1.03325972e-05,  3.28613351e-05, -1.15791783e-05, ...,
          1.24357381e-05, -1.73936381e-05, -1.20480050e-05],
        [-9.60046380e-06,  3.32897109e-05, -1.22916284e-05, ...,
          1.15456401e-05, -1.64922429e-05, -1.07048776e-05],
        [-1.25123467e-05,  3.42865891e-05, -9.90288416e-06, ...,
          9.56462715e-06, -1.69318662e-05, -1.02233871e-05]]),
 (105, 395))

In [162]:
female_gene_updated = np.dot(p_updated_into_df_female_gene_into_w1, avg_female_w1)
female_gene_updated, female_gene_updated.shape

(array([[-3.50916635e-08,  1.06446758e-07,  1.96445455e-07, ...,
         -5.18982068e-08,  2.64550471e-07,  3.66129667e-10],
        [-3.56299438e-08,  1.05504136e-07,  2.06898638e-07, ...,
         -4.91391289e-08,  2.68065772e-07, -8.97874346e-09],
        [-3.72557483e-08,  1.03100761e-07,  2.16311522e-07, ...,
         -5.88274746e-08,  2.70724249e-07, -6.99895596e-11],
        ...,
        [-3.75421086e-08,  1.12529654e-07,  2.13965776e-07, ...,
         -6.50039547e-08,  2.68134626e-07,  7.56227893e-09],
        [-3.67271452e-08,  1.00964191e-07,  2.13391403e-07, ...,
         -5.99975724e-08,  2.67041496e-07,  1.13741221e-08],
        [-3.58363039e-08,  9.41178064e-08,  1.88039373e-07, ...,
         -6.60420638e-08,  2.65070778e-07,  6.01662503e-09]]),
 (105, 4789))

In [163]:
female_gene_df = pd.DataFrame(female_gene_updated, columns=data.columns[:-1])
female_gene_df

,HMGB1P1,A2M,AACS,AADAT,AANAT,AARS2,AASDHPPT,AASDH,AASS,ABAT,...,ZFYVE9,ZIC2,ZMAT2,ZMAT3,ZNF274,ZNRF3,ZW10,ZWILCH,ZWINT,ZYX
0,-3.509166e-08,1.064468e-07,1.964455e-07,-1.125372e-07,-2.012071e-08,-6.542545e-08,-2.181658e-07,-1.940028e-08,8.232706e-08,-2.775337e-07,...,7.567415e-08,-5.969537e-08,3.109365e-07,-6.451551e-08,2.384874e-08,6.060892e-08,1.521491e-08,-5.189821e-08,2.645505e-07,3.661297e-10
1,-3.562994e-08,1.055041e-07,2.068986e-07,-1.149415e-07,-1.979900e-08,-6.665374e-08,-2.421047e-07,-2.152903e-08,9.136062e-08,-2.921553e-07,...,8.559855e-08,-6.248825e-08,3.153046e-07,-6.542104e-08,2.105156e-08,5.555528e-08,1.440604e-08,-4.913913e-08,2.680658e-07,-8.978743e-09
2,-3.725575e-08,1.031008e-07,2.163115e-07,-1.216612e-07,-2.123176e-08,-7.141405e-08,-2.491985e-07,-2.215984e-08,9.403754e-08,-2.930373e-07,...,1.025722e-07,-6.493520e-08,2.997172e-07,-6.301588e-08,2.422480e-08,3.424576e-08,1.724635e-08,-5.882747e-08,2.707242e-07,-6.998956e-11
3,-3.716229e-08,9.817565e-08,2.107803e-07,-1.204801e-07,-2.115914e-08,-7.269088e-08,-2.433544e-07,-2.164016e-08,9.183222e-08,-3.122628e-07,...,1.031409e-07,-6.337080e-08,2.962539e-07,-6.548383e-08,2.107956e-08,4.123762e-08,1.435394e-08,-4.896144e-08,2.726052e-07,6.996574e-09
4,-3.519677e-08,1.044708e-07,1.693845e-07,-1.058960e-07,-1.903983e-08,-6.610149e-08,-2.025593e-07,-1.801248e-08,7.643779e-08,-2.229288e-07,...,7.640399e-08,-5.583559e-08,2.677405e-07,-6.626993e-08,2.289301e-08,4.623579e-08,1.314010e-08,-4.482102e-08,2.620070e-07,2.339728e-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,-3.717382e-08,1.012337e-07,2.135229e-07,-1.246008e-07,-2.123587e-08,-7.127884e-08,-2.682721e-07,-2.385595e-08,1.012352e-07,-3.107080e-07,...,9.439689e-08,-6.145435e-08,3.063640e-07,-6.961815e-08,1.672856e-08,5.291102e-08,1.709853e-08,-5.832324e-08,2.744246e-07,-4.677315e-09
101,-3.553148e-08,1.092954e-07,1.966588e-07,-1.107317e-07,-2.012115e-08,-6.737243e-08,-2.063744e-07,-1.835173e-08,7.787745e-08,-2.656391e-07,...,7.513447e-08,-6.123214e-08,3.033144e-07,-6.611383e-08,1.246684e-08,4.591387e-08,1.430203e-08,-4.878435e-08,2.607617e-07,9.013461e-09
102,-3.754211e-08,1.125297e-07,2.139658e-07,-1.270578e-07,-2.221468e-08,-7.057638e-08,-2.591971e-07,-2.304896e-08,9.781060e-08,-3.052055e-07,...,9.912386e-08,-6.638804e-08,2.893353e-07,-6.699161e-08,1.847921e-08,3.965668e-08,1.905710e-08,-6.500395e-08,2.681346e-07,7.562279e-09
103,-3.672715e-08,1.009642e-07,2.133914e-07,-1.250444e-07,-2.240626e-08,-7.178974e-08,-2.411461e-07,-2.144379e-08,9.099890e-08,-3.127323e-07,...,1.006808e-07,-6.336588e-08,2.936256e-07,-6.433959e-08,2.360967e-08,4.401555e-08,1.758939e-08,-5.999757e-08,2.670415e-07,1.137412e-08


In [164]:
female_gene_df.to_csv(f"[{date}_{num}]Female_Gene_Importance_Dataframe.csv", header = True, index = False)

---

---

In [165]:
cmn_pathway_importance_df = cmn_pathway_df.abs()
cmn_pathway_importance_df

,KEGG_ABC_TRANSPORTERS,KEGG_ACUTE_MYELOID_LEUKEMIA,KEGG_ADHERENS_JUNCTION,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,KEGG_ALDOSTERONE_REGULATED_SODIUM_REABSORPTION,KEGG_ALLOGRAFT_REJECTION,KEGG_ALPHA_LINOLENIC_ACID_METABOLISM,KEGG_ALZHEIMERS_DISEASE,KEGG_AMINOACYL_TRNA_BIOSYNTHESIS,...,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PRKAR1A_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PTCH1_TO_HEDGEHOG_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_RASD1_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_UBQLN2_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_VCP_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_NOTCH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_MGLUR5_CA2_APOPTOTIC_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_PRNP_PI3K_NOX2_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_TRANSPORT_OF_CALCIUM
0,0.000112,0.000071,0.000187,0.000103,0.000196,0.000312,0.000106,0.000160,0.000131,0.000028,...,0.000081,1.079780e-07,0.000059,0.000109,0.000026,0.000158,0.000054,0.000026,0.000003,0.000011
1,0.000116,0.000053,0.000187,0.000091,0.000187,0.000311,0.000111,0.000155,0.000124,0.000027,...,0.000084,1.768239e-06,0.000060,0.000107,0.000011,0.000147,0.000061,0.000023,0.000006,0.000011
2,0.000111,0.000070,0.000194,0.000103,0.000208,0.000312,0.000100,0.000160,0.000136,0.000028,...,0.000082,2.863122e-06,0.000061,0.000111,0.000027,0.000155,0.000054,0.000031,0.000009,0.000014
3,0.000117,0.000068,0.000198,0.000098,0.000203,0.000316,0.000105,0.000164,0.000134,0.000026,...,0.000086,9.628364e-08,0.000064,0.000113,0.000029,0.000163,0.000058,0.000022,0.000004,0.000009
4,0.000115,0.000071,0.000195,0.000103,0.000206,0.000318,0.000105,0.000165,0.000135,0.000032,...,0.000079,1.716539e-06,0.000060,0.000113,0.000027,0.000160,0.000052,0.000033,0.000011,0.000012
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
241,0.000122,0.000069,0.000190,0.000095,0.000199,0.000321,0.000103,0.000164,0.000135,0.000028,...,0.000079,1.683155e-06,0.000063,0.000112,0.000023,0.000155,0.000054,0.000039,0.000010,0.000005
242,0.000110,0.000065,0.000195,0.000089,0.000200,0.000308,0.000110,0.000162,0.000127,0.000024,...,0.000079,1.144773e-06,0.000061,0.000108,0.000024,0.000145,0.000057,0.000031,0.000008,0.000007
243,0.000114,0.000070,0.000201,0.000099,0.000208,0.000315,0.000108,0.000164,0.000137,0.000028,...,0.000082,4.426949e-06,0.000064,0.000110,0.000034,0.000154,0.000057,0.000036,0.000006,0.000007
244,0.000116,0.000070,0.000196,0.000102,0.000203,0.000317,0.000105,0.000165,0.000134,0.000027,...,0.000084,1.105756e-06,0.000061,0.000114,0.000030,0.000158,0.000055,0.000029,0.000006,0.000011


In [166]:
cmn_feature_importance_df = cmn_gene_df.abs()
cmn_feature_importance_df

,HMGB1P1,A2M,AACS,AADAT,AANAT,AARS2,AASDHPPT,AASDH,AASS,ABAT,...,ZFYVE9,ZIC2,ZMAT2,ZMAT3,ZNF274,ZNRF3,ZW10,ZWILCH,ZWINT,ZYX
0,1.572409e-07,1.308864e-10,1.428225e-07,4.024398e-07,9.694059e-09,1.592726e-09,3.446902e-08,7.421311e-07,6.649394e-07,8.168771e-07,...,3.053214e-08,4.145366e-08,2.932781e-08,1.901506e-08,1.461214e-07,6.285771e-09,1.754992e-07,7.776835e-08,2.493975e-07,3.546062e-08
1,1.631927e-07,1.357067e-10,1.405346e-07,3.861722e-07,1.000760e-08,1.529972e-09,3.389818e-08,7.298408e-07,6.539274e-07,8.149336e-07,...,3.126189e-08,6.509326e-08,3.232577e-08,1.732916e-08,1.378858e-07,8.452137e-09,1.671042e-07,7.404830e-08,2.425640e-07,2.481430e-08
2,1.609560e-07,1.328584e-10,1.443677e-07,4.119727e-07,9.221083e-09,1.544799e-09,3.446627e-08,7.420720e-07,6.648865e-07,8.255333e-07,...,3.055070e-08,4.744472e-08,2.995696e-08,1.856484e-08,1.430642e-07,6.131696e-09,1.716823e-07,7.607699e-08,2.500991e-07,3.225108e-08
3,1.697293e-07,1.370235e-10,1.466823e-07,4.160705e-07,9.425442e-09,1.439584e-09,3.494044e-08,7.522810e-07,6.740335e-07,8.422985e-07,...,3.195782e-08,5.289466e-08,3.460804e-08,1.650757e-08,1.450545e-07,7.270973e-09,1.852241e-07,8.207774e-08,2.546801e-07,2.182655e-08
4,1.619695e-07,1.385700e-10,1.362884e-07,3.933535e-07,9.307880e-09,1.781716e-09,3.349569e-08,7.211751e-07,6.461631e-07,8.117805e-07,...,3.093460e-08,3.251402e-08,3.220633e-08,1.702684e-08,1.468351e-07,6.253101e-09,1.789269e-07,7.928729e-08,2.391218e-07,3.651810e-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
241,1.725463e-07,9.325295e-11,1.258283e-07,4.181908e-07,8.340723e-09,1.587638e-09,3.379771e-08,7.276776e-07,6.519893e-07,7.678357e-07,...,3.140589e-08,7.208366e-08,3.225540e-08,1.805860e-08,1.501580e-07,6.468860e-09,1.690839e-07,7.492556e-08,2.457643e-07,3.321745e-08
242,1.631375e-07,1.469547e-10,1.396224e-07,3.875112e-07,9.501811e-09,1.323327e-09,3.338551e-08,7.188029e-07,6.440376e-07,8.159623e-07,...,3.091795e-08,5.263204e-08,3.060873e-08,1.697772e-08,1.435197e-07,6.944685e-09,1.715935e-07,7.603767e-08,2.453299e-07,2.770432e-08
243,1.748440e-07,1.399656e-10,1.542950e-07,4.208745e-07,8.931765e-09,1.550533e-09,3.464135e-08,7.458415e-07,6.682638e-07,8.687987e-07,...,3.090543e-08,5.697609e-08,3.204529e-08,1.707305e-08,1.445101e-07,6.396599e-09,1.835787e-07,8.134863e-08,2.577729e-07,2.639469e-08
244,1.657765e-07,1.315774e-10,1.453125e-07,4.119418e-07,9.138313e-09,1.508004e-09,3.436799e-08,7.399559e-07,6.629904e-07,8.372546e-07,...,3.107222e-08,4.562949e-08,3.211573e-08,1.807357e-08,1.455206e-07,6.564548e-09,1.842743e-07,8.165683e-08,2.580680e-07,3.185456e-08


In [167]:
male_pathway_importance_df = male_pathway_df.abs()
male_pathway_importance_df

,KEGG_ABC_TRANSPORTERS,KEGG_ACUTE_MYELOID_LEUKEMIA,KEGG_ADHERENS_JUNCTION,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,KEGG_ALDOSTERONE_REGULATED_SODIUM_REABSORPTION,KEGG_ALLOGRAFT_REJECTION,KEGG_ALPHA_LINOLENIC_ACID_METABOLISM,KEGG_ALZHEIMERS_DISEASE,KEGG_AMINOACYL_TRNA_BIOSYNTHESIS,...,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PRKAR1A_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PTCH1_TO_HEDGEHOG_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_RASD1_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_UBQLN2_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_VCP_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_NOTCH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_MGLUR5_CA2_APOPTOTIC_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_PRNP_PI3K_NOX2_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_TRANSPORT_OF_CALCIUM
0,0.000132,0.000033,0.000023,0.000082,0.000085,0.000189,0.000035,0.000070,0.000140,0.000086,...,3.836328e-07,0.000247,0.000027,0.000060,0.000087,0.000048,0.000038,0.000055,0.000164,0.000088
1,0.000141,0.000029,0.000025,0.000078,0.000081,0.000197,0.000039,0.000069,0.000141,0.000091,...,5.729461e-07,0.000244,0.000026,0.000071,0.000091,0.000053,0.000034,0.000057,0.000167,0.000096
2,0.000145,0.000026,0.000025,0.000080,0.000084,0.000199,0.000043,0.000066,0.000149,0.000092,...,1.943093e-06,0.000254,0.000025,0.000067,0.000089,0.000048,0.000031,0.000068,0.000168,0.000099
3,0.000140,0.000035,0.000023,0.000088,0.000078,0.000193,0.000036,0.000068,0.000140,0.000092,...,1.140293e-05,0.000249,0.000026,0.000064,0.000088,0.000049,0.000039,0.000064,0.000166,0.000090
4,0.000132,0.000029,0.000027,0.000080,0.000083,0.000194,0.000037,0.000074,0.000134,0.000091,...,7.041079e-06,0.000246,0.000027,0.000070,0.000087,0.000046,0.000034,0.000060,0.000157,0.000097
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136,0.000138,0.000029,0.000023,0.000082,0.000084,0.000197,0.000042,0.000074,0.000143,0.000091,...,8.639964e-06,0.000245,0.000029,0.000064,0.000090,0.000045,0.000035,0.000061,0.000166,0.000090
137,0.000138,0.000030,0.000028,0.000081,0.000081,0.000196,0.000043,0.000063,0.000149,0.000084,...,3.756694e-06,0.000244,0.000028,0.000061,0.000088,0.000045,0.000041,0.000063,0.000156,0.000095
138,0.000140,0.000030,0.000025,0.000083,0.000080,0.000201,0.000047,0.000070,0.000145,0.000092,...,8.417630e-06,0.000249,0.000023,0.000072,0.000091,0.000048,0.000040,0.000062,0.000159,0.000094
139,0.000142,0.000025,0.000023,0.000082,0.000080,0.000199,0.000043,0.000071,0.000147,0.000095,...,5.150428e-06,0.000250,0.000027,0.000069,0.000088,0.000049,0.000038,0.000064,0.000168,0.000097


In [168]:
male_feature_importance_df = male_gene_df.abs()
male_feature_importance_df

,HMGB1P1,A2M,AACS,AADAT,AANAT,AARS2,AASDHPPT,AASDH,AASS,ABAT,...,ZFYVE9,ZIC2,ZMAT2,ZMAT3,ZNF274,ZNRF3,ZW10,ZWILCH,ZWINT,ZYX
0,6.250791e-07,5.672813e-09,6.923179e-09,6.211402e-08,2.396544e-07,2.276046e-07,5.478589e-07,9.701522e-08,4.033995e-07,1.782961e-07,...,8.728684e-08,1.478508e-07,6.002200e-08,2.058850e-07,1.173557e-07,2.766050e-07,1.333146e-07,1.634060e-07,3.215451e-08,7.382663e-08
1,6.247149e-07,3.106215e-09,6.620719e-09,6.759893e-08,2.357103e-07,2.384544e-07,5.914177e-07,1.047287e-07,4.354728e-07,1.900802e-07,...,9.047989e-08,1.298457e-07,5.167089e-08,2.111320e-07,1.122433e-07,2.843229e-07,1.393202e-07,1.707671e-07,3.228731e-08,7.656132e-08
2,6.285345e-07,5.083062e-09,7.009029e-09,6.751605e-08,2.338888e-07,2.418922e-07,5.903985e-07,1.045482e-07,4.347223e-07,1.847194e-07,...,9.195165e-08,1.404341e-07,5.851587e-08,2.078595e-07,1.100720e-07,2.670476e-07,1.337352e-07,1.639216e-07,3.291782e-08,7.813693e-08
3,6.251969e-07,3.630910e-09,6.830019e-09,6.369154e-08,2.440345e-07,2.415967e-07,5.614454e-07,9.942113e-08,4.134035e-07,2.010746e-07,...,9.157572e-08,1.376055e-07,5.637221e-08,2.097452e-07,1.121494e-07,2.827464e-07,1.398664e-07,1.714366e-07,3.233278e-08,7.733524e-08
4,6.222525e-07,5.741904e-09,5.662533e-09,6.322551e-08,2.350166e-07,2.408771e-07,5.559490e-07,9.844783e-08,4.093565e-07,1.700842e-07,...,8.973872e-08,1.387313e-07,5.522175e-08,2.122231e-07,1.098607e-07,2.701180e-07,1.383347e-07,1.695592e-07,3.149339e-08,7.895552e-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136,6.205690e-07,4.743541e-09,6.528122e-09,6.418947e-08,2.396465e-07,2.390791e-07,5.646262e-07,9.998439e-08,4.157456e-07,1.908154e-07,...,8.935086e-08,1.357988e-07,5.735156e-08,2.103476e-07,1.168029e-07,2.833804e-07,1.463733e-07,1.794122e-07,3.192181e-08,7.718810e-08
137,6.127714e-07,3.484433e-09,6.343067e-09,6.419562e-08,2.326163e-07,2.209736e-07,5.633264e-07,9.975422e-08,4.147886e-07,1.793506e-07,...,8.787157e-08,1.373754e-07,6.507361e-08,2.019867e-07,1.177443e-07,2.672790e-07,1.390942e-07,1.704901e-07,3.234010e-08,7.610453e-08
138,6.292921e-07,8.276666e-09,6.261482e-09,6.677714e-08,2.208956e-07,2.415386e-07,5.819343e-07,1.030493e-07,4.284900e-07,1.890310e-07,...,9.083551e-08,1.190301e-07,5.799908e-08,2.114922e-07,1.152296e-07,2.830260e-07,1.572746e-07,1.927742e-07,3.239557e-08,8.025252e-08
139,6.244205e-07,5.985030e-09,6.913902e-09,6.643939e-08,2.410528e-07,2.503159e-07,5.830746e-07,1.032512e-07,4.293296e-07,1.852491e-07,...,9.109623e-08,1.257767e-07,6.190167e-08,2.074964e-07,1.124434e-07,2.814423e-07,1.464534e-07,1.795104e-07,3.254876e-08,7.844224e-08


In [169]:
female_pathway_importance_df = female_pathway_df.abs()
female_pathway_importance_df

,KEGG_ABC_TRANSPORTERS,KEGG_ACUTE_MYELOID_LEUKEMIA,KEGG_ADHERENS_JUNCTION,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,KEGG_ALDOSTERONE_REGULATED_SODIUM_REABSORPTION,KEGG_ALLOGRAFT_REJECTION,KEGG_ALPHA_LINOLENIC_ACID_METABOLISM,KEGG_ALZHEIMERS_DISEASE,KEGG_AMINOACYL_TRNA_BIOSYNTHESIS,...,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PRKAR1A_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PTCH1_TO_HEDGEHOG_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_RASD1_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_UBQLN2_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_VCP_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_NOTCH_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_MGLUR5_CA2_APOPTOTIC_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_PRNP_PI3K_NOX2_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_TRANSPORT_OF_CALCIUM
0,0.000045,0.000134,0.000034,0.000372,0.000134,0.000017,0.000020,6.479428e-06,0.000006,0.000122,...,0.000083,0.000065,0.000134,0.000010,0.000051,0.000031,0.000144,0.000044,0.000062,0.000039
1,0.000042,0.000134,0.000038,0.000371,0.000140,0.000018,0.000020,5.266082e-07,0.000005,0.000125,...,0.000088,0.000063,0.000134,0.000012,0.000053,0.000030,0.000149,0.000048,0.000064,0.000045
2,0.000037,0.000142,0.000043,0.000384,0.000144,0.000022,0.000022,5.507924e-06,0.000008,0.000134,...,0.000093,0.000071,0.000137,0.000009,0.000051,0.000032,0.000164,0.000048,0.000067,0.000044
3,0.000039,0.000136,0.000052,0.000380,0.000140,0.000026,0.000021,4.372620e-06,0.000006,0.000136,...,0.000093,0.000073,0.000137,0.000011,0.000049,0.000034,0.000159,0.000049,0.000064,0.000045
4,0.000046,0.000135,0.000029,0.000363,0.000124,0.000005,0.000013,1.846329e-05,0.000012,0.000124,...,0.000067,0.000058,0.000136,0.000007,0.000042,0.000035,0.000139,0.000046,0.000054,0.000032
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,0.000046,0.000132,0.000043,0.000382,0.000136,0.000027,0.000020,3.352919e-06,0.000010,0.000133,...,0.000090,0.000066,0.000143,0.000010,0.000053,0.000031,0.000155,0.000049,0.000063,0.000048
101,0.000048,0.000134,0.000033,0.000368,0.000132,0.000011,0.000017,1.157617e-05,0.000009,0.000126,...,0.000083,0.000069,0.000135,0.000008,0.000051,0.000029,0.000143,0.000048,0.000064,0.000037
102,0.000041,0.000131,0.000047,0.000378,0.000142,0.000019,0.000017,4.639647e-06,0.000009,0.000132,...,0.000095,0.000070,0.000139,0.000003,0.000050,0.000030,0.000157,0.000050,0.000070,0.000048
103,0.000038,0.000133,0.000049,0.000380,0.000143,0.000022,0.000025,7.279073e-06,0.000011,0.000134,...,0.000091,0.000070,0.000137,0.000008,0.000050,0.000033,0.000159,0.000046,0.000066,0.000043


In [170]:
female_feature_importance_df = female_gene_df.abs()
female_feature_importance_df

,HMGB1P1,A2M,AACS,AADAT,AANAT,AARS2,AASDHPPT,AASDH,AASS,ABAT,...,ZFYVE9,ZIC2,ZMAT2,ZMAT3,ZNF274,ZNRF3,ZW10,ZWILCH,ZWINT,ZYX
0,3.509166e-08,1.064468e-07,1.964455e-07,1.125372e-07,2.012071e-08,6.542545e-08,2.181658e-07,1.940028e-08,8.232706e-08,2.775337e-07,...,7.567415e-08,5.969537e-08,3.109365e-07,6.451551e-08,2.384874e-08,6.060892e-08,1.521491e-08,5.189821e-08,2.645505e-07,3.661297e-10
1,3.562994e-08,1.055041e-07,2.068986e-07,1.149415e-07,1.979900e-08,6.665374e-08,2.421047e-07,2.152903e-08,9.136062e-08,2.921553e-07,...,8.559855e-08,6.248825e-08,3.153046e-07,6.542104e-08,2.105156e-08,5.555528e-08,1.440604e-08,4.913913e-08,2.680658e-07,8.978743e-09
2,3.725575e-08,1.031008e-07,2.163115e-07,1.216612e-07,2.123176e-08,7.141405e-08,2.491985e-07,2.215984e-08,9.403754e-08,2.930373e-07,...,1.025722e-07,6.493520e-08,2.997172e-07,6.301588e-08,2.422480e-08,3.424576e-08,1.724635e-08,5.882747e-08,2.707242e-07,6.998956e-11
3,3.716229e-08,9.817565e-08,2.107803e-07,1.204801e-07,2.115914e-08,7.269088e-08,2.433544e-07,2.164016e-08,9.183222e-08,3.122628e-07,...,1.031409e-07,6.337080e-08,2.962539e-07,6.548383e-08,2.107956e-08,4.123762e-08,1.435394e-08,4.896144e-08,2.726052e-07,6.996574e-09
4,3.519677e-08,1.044708e-07,1.693845e-07,1.058960e-07,1.903983e-08,6.610149e-08,2.025593e-07,1.801248e-08,7.643779e-08,2.229288e-07,...,7.640399e-08,5.583559e-08,2.677405e-07,6.626993e-08,2.289301e-08,4.623579e-08,1.314010e-08,4.482102e-08,2.620070e-07,2.339728e-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,3.717382e-08,1.012337e-07,2.135229e-07,1.246008e-07,2.123587e-08,7.127884e-08,2.682721e-07,2.385595e-08,1.012352e-07,3.107080e-07,...,9.439689e-08,6.145435e-08,3.063640e-07,6.961815e-08,1.672856e-08,5.291102e-08,1.709853e-08,5.832324e-08,2.744246e-07,4.677315e-09
101,3.553148e-08,1.092954e-07,1.966588e-07,1.107317e-07,2.012115e-08,6.737243e-08,2.063744e-07,1.835173e-08,7.787745e-08,2.656391e-07,...,7.513447e-08,6.123214e-08,3.033144e-07,6.611383e-08,1.246684e-08,4.591387e-08,1.430203e-08,4.878435e-08,2.607617e-07,9.013461e-09
102,3.754211e-08,1.125297e-07,2.139658e-07,1.270578e-07,2.221468e-08,7.057638e-08,2.591971e-07,2.304896e-08,9.781060e-08,3.052055e-07,...,9.912386e-08,6.638804e-08,2.893353e-07,6.699161e-08,1.847921e-08,3.965668e-08,1.905710e-08,6.500395e-08,2.681346e-07,7.562279e-09
103,3.672715e-08,1.009642e-07,2.133914e-07,1.250444e-07,2.240626e-08,7.178974e-08,2.411461e-07,2.144379e-08,9.099890e-08,3.127323e-07,...,1.006808e-07,6.336588e-08,2.936256e-07,6.433959e-08,2.360967e-08,4.401555e-08,1.758939e-08,5.999757e-08,2.670415e-07,1.137412e-08


---

---

In [171]:
cmn_feature_importance_df.mean().sort_values(ascending=False)

MAPK3     3.960071e-06
GNB1      2.510114e-06
KRAS      2.504046e-06
GLUD1     2.448520e-06
IFNA17    2.343861e-06
              ...     
NOTUM     1.264745e-10
CRB3      1.134861e-10
EFNB2     5.479895e-11
TMED10    4.181061e-11
CNTN1     1.766487e-11
Length: 4789, dtype: float64

In [172]:
cmn_feature_importance_df.mean().sort_values(ascending=False).index[:30]

Index(['MAPK3', 'GNB1', 'KRAS', 'GLUD1', 'IFNA17', 'SOS2', 'NTF3', 'ITGB5',
       'NDUFS7', 'GNG5', 'GNG12', 'RHOA', 'NDUFB7', 'TNF', 'IFNA4', 'ADCY6',
       'PIK3R2', 'COX4I2', 'TCF7', 'IFNA7', 'ALDH2', 'GNB5', 'NDUFV1',
       'PPP3R1', 'ATF2', 'GSK3B', 'SRC', 'CAMK4', 'TUBA4A', 'GNG2'],
      dtype='object')

In [173]:
male_feature_importance_df.mean().sort_values(ascending=False)

PRKACG      3.335432e-06
ACTB        2.644358e-06
CREB5       2.366191e-06
FGFR2       2.333007e-06
ITGA7       2.295670e-06
                ...     
HTRA2       1.211216e-10
ARHGEF11    1.204718e-10
GMPR2       1.166520e-10
PEX26       7.073120e-11
LPCAT3      6.635819e-11
Length: 4789, dtype: float64

In [174]:
male_feature_importance_df.mean().sort_values(ascending=False).index[:30]

Index(['PRKACG', 'ACTB', 'CREB5', 'FGFR2', 'ITGA7', 'CALM3', 'PRKACB',
       'CACNA1C', 'ITGB7', 'PIK3R1', 'IFNA1', 'HSP90AB1', 'HLA-DRB1', 'CACNG6',
       'ITGA6', 'TAB2', 'SMPD3', 'CALM1', 'TGFB3', 'EFNA3', 'TUBA3E', 'DLST',
       'HADHA', 'EMD', 'IL6', 'IGSF5', 'PPP2CA', 'TNNC1', 'ANGPT4', 'HLA-DOA'],
      dtype='object')

In [175]:
female_feature_importance_df.mean().sort_values(ascending=False)

PLCB4     3.910497e-06
MAP2K2    2.837611e-06
ADCY7     2.326484e-06
ADCY6     2.316249e-06
PRKACG    2.226943e-06
              ...     
PYDC1     2.153505e-10
TLR6      1.977197e-10
PHYH      1.583155e-10
LCORL     1.256247e-10
HAGHL     1.236292e-10
Length: 4789, dtype: float64

In [176]:
female_feature_importance_df.mean().sort_values(ascending=False).index[:30]

Index(['PLCB4', 'MAP2K2', 'ADCY7', 'ADCY6', 'PRKACG', 'GSK3B', 'PPP3CA',
       'CREB3L4', 'STK11', 'WNT10B', 'MAPK3', 'MYC', 'IFNA1', 'JAK2', 'ARPC1B',
       'PLCG2', 'PRKAG3', 'AKT2', 'ATF2', 'BMP2', 'GNAI1', 'ATP6V1E2', 'JUN',
       'NDUFV3', 'VDAC2', 'KRAS', 'PTPN11', 'UGT2B15', 'FGF10', 'UGT2B28'],
      dtype='object')

In [177]:
np.intersect1d(cmn_feature_importance_df.mean().sort_values(ascending=False).index[:30], male_feature_importance_df.mean().sort_values(ascending=False).index[:30])

array([], dtype=object)

In [178]:
np.intersect1d(cmn_feature_importance_df.mean().sort_values(ascending=False).index[:30], female_feature_importance_df.mean().sort_values(ascending=False).index[:30])

array(['ADCY6', 'ATF2', 'GSK3B', 'KRAS', 'MAPK3'], dtype=object)

In [179]:
np.intersect1d(male_feature_importance_df.mean().sort_values(ascending=False).index[:30], female_feature_importance_df.mean().sort_values(ascending=False).index[:30])

array(['IFNA1', 'PRKACG'], dtype=object)

---

In [180]:
np.intersect1d(cmn_feature_importance_df.mean().sort_values(ascending=False).index[:20], male_feature_importance_df.mean().sort_values(ascending=False).index[:20])

array([], dtype=object)

In [181]:
np.intersect1d(cmn_feature_importance_df.mean().sort_values(ascending=False).index[:20], female_feature_importance_df.mean().sort_values(ascending=False).index[:20])

array(['ADCY6', 'MAPK3'], dtype=object)

In [182]:
np.intersect1d(male_feature_importance_df.mean().sort_values(ascending=False).index[:20], female_feature_importance_df.mean().sort_values(ascending=False).index[:20])

array(['IFNA1', 'PRKACG'], dtype=object)

---

# 1-sample sex-specific t-test compared with zero

---

In [183]:
from scipy.stats import ttest_1samp
from scipy.stats import ttest_ind
from datetime import datetime
from statsmodels.stats.multitest import multipletests

In [184]:
cmn_feature_importance_df

,HMGB1P1,A2M,AACS,AADAT,AANAT,AARS2,AASDHPPT,AASDH,AASS,ABAT,...,ZFYVE9,ZIC2,ZMAT2,ZMAT3,ZNF274,ZNRF3,ZW10,ZWILCH,ZWINT,ZYX
0,1.572409e-07,1.308864e-10,1.428225e-07,4.024398e-07,9.694059e-09,1.592726e-09,3.446902e-08,7.421311e-07,6.649394e-07,8.168771e-07,...,3.053214e-08,4.145366e-08,2.932781e-08,1.901506e-08,1.461214e-07,6.285771e-09,1.754992e-07,7.776835e-08,2.493975e-07,3.546062e-08
1,1.631927e-07,1.357067e-10,1.405346e-07,3.861722e-07,1.000760e-08,1.529972e-09,3.389818e-08,7.298408e-07,6.539274e-07,8.149336e-07,...,3.126189e-08,6.509326e-08,3.232577e-08,1.732916e-08,1.378858e-07,8.452137e-09,1.671042e-07,7.404830e-08,2.425640e-07,2.481430e-08
2,1.609560e-07,1.328584e-10,1.443677e-07,4.119727e-07,9.221083e-09,1.544799e-09,3.446627e-08,7.420720e-07,6.648865e-07,8.255333e-07,...,3.055070e-08,4.744472e-08,2.995696e-08,1.856484e-08,1.430642e-07,6.131696e-09,1.716823e-07,7.607699e-08,2.500991e-07,3.225108e-08
3,1.697293e-07,1.370235e-10,1.466823e-07,4.160705e-07,9.425442e-09,1.439584e-09,3.494044e-08,7.522810e-07,6.740335e-07,8.422985e-07,...,3.195782e-08,5.289466e-08,3.460804e-08,1.650757e-08,1.450545e-07,7.270973e-09,1.852241e-07,8.207774e-08,2.546801e-07,2.182655e-08
4,1.619695e-07,1.385700e-10,1.362884e-07,3.933535e-07,9.307880e-09,1.781716e-09,3.349569e-08,7.211751e-07,6.461631e-07,8.117805e-07,...,3.093460e-08,3.251402e-08,3.220633e-08,1.702684e-08,1.468351e-07,6.253101e-09,1.789269e-07,7.928729e-08,2.391218e-07,3.651810e-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
241,1.725463e-07,9.325295e-11,1.258283e-07,4.181908e-07,8.340723e-09,1.587638e-09,3.379771e-08,7.276776e-07,6.519893e-07,7.678357e-07,...,3.140589e-08,7.208366e-08,3.225540e-08,1.805860e-08,1.501580e-07,6.468860e-09,1.690839e-07,7.492556e-08,2.457643e-07,3.321745e-08
242,1.631375e-07,1.469547e-10,1.396224e-07,3.875112e-07,9.501811e-09,1.323327e-09,3.338551e-08,7.188029e-07,6.440376e-07,8.159623e-07,...,3.091795e-08,5.263204e-08,3.060873e-08,1.697772e-08,1.435197e-07,6.944685e-09,1.715935e-07,7.603767e-08,2.453299e-07,2.770432e-08
243,1.748440e-07,1.399656e-10,1.542950e-07,4.208745e-07,8.931765e-09,1.550533e-09,3.464135e-08,7.458415e-07,6.682638e-07,8.687987e-07,...,3.090543e-08,5.697609e-08,3.204529e-08,1.707305e-08,1.445101e-07,6.396599e-09,1.835787e-07,8.134863e-08,2.577729e-07,2.639469e-08
244,1.657765e-07,1.315774e-10,1.453125e-07,4.119418e-07,9.138313e-09,1.508004e-09,3.436799e-08,7.399559e-07,6.629904e-07,8.372546e-07,...,3.107222e-08,4.562949e-08,3.211573e-08,1.807357e-08,1.455206e-07,6.564548e-09,1.842743e-07,8.165683e-08,2.580680e-07,3.185456e-08


In [185]:
common_gene_pvalues = []
male_gene_pvalues = []
female_gene_pvalues = []
for idx in range(len(cmn_feature_importance_df.columns)):
    common_gene_pvalues.append(ttest_1samp(cmn_feature_importance_df.iloc[:, idx], popmean=0))
    male_gene_pvalues.append(ttest_1samp(male_feature_importance_df.iloc[:, idx], popmean=0))
    female_gene_pvalues.append(ttest_1samp(female_feature_importance_df.iloc[:, idx], popmean=0))

In [186]:
common_pathway_pvalues = []
male_pathway_pvalues = []
female_pathway_pvalues = []
for idx in range(len(cmn_pathway_importance_df.columns)):
    common_pathway_pvalues.append(ttest_1samp(cmn_pathway_importance_df.iloc[:, idx], popmean=0))
    male_pathway_pvalues.append(ttest_1samp(male_pathway_importance_df.iloc[:, idx], popmean=0))
    female_pathway_pvalues.append(ttest_1samp(female_pathway_importance_df.iloc[:, idx], popmean=0))

---

### Common

In [187]:
common_gene_pval_df = pd.DataFrame(common_gene_pvalues).rename(columns = {"pvalue" : "common_pval"})
common_gene_pval_df

,statistic,common_pval
0,247.178236,7.589358e-296
1,138.707042,7.855208e-235
2,324.092444,0.000000e+00
3,287.857320,5.307535e-312
4,218.923206,5.471425e-283
...,...,...
4784,101.614705,2.709341e-202
4785,234.101537,4.358821e-290
4786,234.101537,4.358821e-290
4787,476.938915,0.000000e+00


In [188]:
common_gene_pval_df["-log10(common_pval)"] = -np.log10(common_gene_pval_df["common_pval"])

/home/koe3/conda/envs/cl/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [189]:
common_gene_pval_df["Gene"] = cmn_feature_importance_df.columns.values

In [190]:
cmn_feature_importance_df.abs().mean().values

array([1.67721336e-07, 1.32445562e-10, 1.42175176e-07, ...,
       7.89419789e-08, 2.52550259e-07, 2.92643069e-08])

In [191]:
common_gene_pval_df["ABS_Importance"] = cmn_feature_importance_df.abs().mean().values
common_gene_pval_df

,statistic,common_pval,-log10(common_pval),Gene,ABS_Importance
0,247.178236,7.589358e-296,295.119795,HMGB1P1,1.677213e-07
1,138.707042,7.855208e-235,234.104842,A2M,1.324456e-10
2,324.092444,0.000000e+00,inf,AACS,1.421752e-07
3,287.857320,5.307535e-312,311.275107,AADAT,4.094231e-07
4,218.923206,5.471425e-283,282.261900,AANAT,9.353754e-09
...,...,...,...,...,...
4784,101.614705,2.709341e-202,201.567136,ZNRF3,6.821541e-09
4785,234.101537,4.358821e-290,289.360631,ZW10,1.781477e-07
4786,234.101537,4.358821e-290,289.360631,ZWILCH,7.894198e-08
4787,476.938915,0.000000e+00,inf,ZWINT,2.525503e-07


In [192]:
common_gene_pval_df["Importance"] = cmn_feature_importance_df.mean().values
common_gene_pval_df

,statistic,common_pval,-log10(common_pval),Gene,ABS_Importance,Importance
0,247.178236,7.589358e-296,295.119795,HMGB1P1,1.677213e-07,1.677213e-07
1,138.707042,7.855208e-235,234.104842,A2M,1.324456e-10,1.324456e-10
2,324.092444,0.000000e+00,inf,AACS,1.421752e-07,1.421752e-07
3,287.857320,5.307535e-312,311.275107,AADAT,4.094231e-07,4.094231e-07
4,218.923206,5.471425e-283,282.261900,AANAT,9.353754e-09,9.353754e-09
...,...,...,...,...,...,...
4784,101.614705,2.709341e-202,201.567136,ZNRF3,6.821541e-09,6.821541e-09
4785,234.101537,4.358821e-290,289.360631,ZW10,1.781477e-07,1.781477e-07
4786,234.101537,4.358821e-290,289.360631,ZWILCH,7.894198e-08,7.894198e-08
4787,476.938915,0.000000e+00,inf,ZWINT,2.525503e-07,2.525503e-07


---

In [193]:
common_pathway_pval_df = pd.DataFrame(common_pathway_pvalues).rename(columns = {"pvalue" : "common_pval"})
common_pathway_pval_df

,statistic,common_pval
0,408.963138,0.000000e+00
1,119.139799,6.903559e-219
2,506.814709,0.000000e+00
3,286.493945,1.692587e-311
4,251.800306,8.253290e-298
...,...,...
390,381.061377,0.000000e+00
391,318.132559,1.284571e-322
392,49.597084,9.167552e-130
393,38.159532,4.438809e-105


In [194]:
common_pathway_pval_df["-log10(common_pval)"] = -np.log10(common_pathway_pval_df["common_pval"])

/home/koe3/conda/envs/cl/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [195]:
common_pathway_pval_df["pathway"] = cmn_pathway_importance_df.columns.values

In [196]:
cmn_pathway_importance_df.abs().mean().values

array([1.16425509e-04, 6.72989327e-05, 1.94258724e-04, 9.80917183e-05,
       1.99489371e-04, 3.14647495e-04, 1.05858523e-04, 1.62783927e-04,
       1.31305048e-04, 2.57611920e-05, 9.76693593e-05, 9.25125085e-05,
       1.68020620e-04, 6.09829445e-05, 6.85870060e-05, 7.06921482e-05,
       1.06237557e-04, 9.21909831e-05, 1.71039248e-04, 1.26198490e-04,
       5.57872624e-05, 4.47570883e-05, 1.37864172e-04, 1.02540299e-04,
       2.16864922e-04, 3.26469137e-05, 4.63546700e-05, 9.20302912e-05,
       1.92018087e-04, 6.53815619e-05, 1.71193204e-04, 1.15719573e-04,
       2.78274871e-04, 1.03440841e-04, 4.70462485e-05, 1.64116770e-05,
       1.16922368e-04, 4.26461996e-05, 6.98008875e-05, 8.29130870e-05,
       2.55517553e-04, 5.29069524e-05, 5.00460231e-05, 1.16904992e-05,
       9.34765041e-05, 5.21837807e-06, 8.89503454e-05, 1.66715661e-04,
       3.65764482e-05, 1.36302820e-04, 2.38378489e-05, 4.02515966e-05,
       7.44298469e-05, 9.31273153e-05, 5.65464330e-05, 2.32149326e-05,
      

In [197]:
common_pathway_pval_df["ABS_Importance"] = cmn_pathway_importance_df.abs().mean().values
common_pathway_pval_df

,statistic,common_pval,-log10(common_pval),pathway,ABS_Importance
0,408.963138,0.000000e+00,inf,KEGG_ABC_TRANSPORTERS,0.000116
1,119.139799,6.903559e-219,218.160927,KEGG_ACUTE_MYELOID_LEUKEMIA,0.000067
2,506.814709,0.000000e+00,inf,KEGG_ADHERENS_JUNCTION,0.000194
3,286.493945,1.692587e-311,310.771449,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,0.000098
4,251.800306,8.253290e-298,297.083373,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,0.000199
...,...,...,...,...,...
390,381.061377,0.000000e+00,inf,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,0.000155
391,318.132559,1.284571e-322,321.891242,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000056
392,49.597084,9.167552e-130,129.037747,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000027
393,38.159532,4.438809e-105,104.352734,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000007


In [198]:
common_pathway_pval_df["Importance"] = cmn_pathway_importance_df.mean().values
common_pathway_pval_df

,statistic,common_pval,-log10(common_pval),pathway,ABS_Importance,Importance
0,408.963138,0.000000e+00,inf,KEGG_ABC_TRANSPORTERS,0.000116,0.000116
1,119.139799,6.903559e-219,218.160927,KEGG_ACUTE_MYELOID_LEUKEMIA,0.000067,0.000067
2,506.814709,0.000000e+00,inf,KEGG_ADHERENS_JUNCTION,0.000194,0.000194
3,286.493945,1.692587e-311,310.771449,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,0.000098,0.000098
4,251.800306,8.253290e-298,297.083373,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,0.000199,0.000199
...,...,...,...,...,...,...
390,381.061377,0.000000e+00,inf,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,0.000155,0.000155
391,318.132559,1.284571e-322,321.891242,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000056,0.000056
392,49.597084,9.167552e-130,129.037747,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000027,0.000027
393,38.159532,4.438809e-105,104.352734,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000007,0.000007


---

---

### Male

In [199]:
male_gene_pval_df = pd.DataFrame(male_gene_pvalues).rename(columns = {"pvalue" : "male_pval"})
male_gene_pval_df

,statistic,male_pval
0,858.038948,2.292678e-262
1,23.771198,5.347159e-51
2,159.096034,4.552499e-160
3,280.382399,2.080887e-194
4,238.529940,1.339279e-184
...,...,...
4784,273.727413,5.972434e-193
4785,197.715584,3.187289e-173
4786,197.715584,3.187289e-173
4787,712.499566,4.559265e-251


In [200]:
male_gene_pval_df["-log10(male_pval)"] = -np.log10(male_gene_pval_df["male_pval"])

In [201]:
male_gene_pval_df["Gene"] = male_feature_importance_df.columns.values

In [202]:
male_gene_pval_df["ABS_Importance"] = male_feature_importance_df.abs().mean().values
male_gene_pval_df

,statistic,male_pval,-log10(male_pval),Gene,ABS_Importance
0,858.038948,2.292678e-262,261.639657,HMGB1P1,6.238669e-07
1,23.771198,5.347159e-51,50.271877,A2M,5.936987e-09
2,159.096034,4.552499e-160,159.341750,AACS,6.656735e-09
3,280.382399,2.080887e-194,193.681751,AADAT,6.509067e-08
4,238.529940,1.339279e-184,183.873129,AANAT,2.368513e-07
...,...,...,...,...,...
4784,273.727413,5.972434e-193,192.223849,ZNRF3,2.785301e-07
4785,197.715584,3.187289e-173,172.496579,ZW10,1.442847e-07
4786,197.715584,3.187289e-173,172.496579,ZWILCH,1.768522e-07
4787,712.499566,4.559265e-251,250.341105,ZWINT,3.228468e-08


In [203]:
male_gene_pval_df["Importance"] = male_feature_importance_df.mean().values
male_gene_pval_df

,statistic,male_pval,-log10(male_pval),Gene,ABS_Importance,Importance
0,858.038948,2.292678e-262,261.639657,HMGB1P1,6.238669e-07,6.238669e-07
1,23.771198,5.347159e-51,50.271877,A2M,5.936987e-09,5.936987e-09
2,159.096034,4.552499e-160,159.341750,AACS,6.656735e-09,6.656735e-09
3,280.382399,2.080887e-194,193.681751,AADAT,6.509067e-08,6.509067e-08
4,238.529940,1.339279e-184,183.873129,AANAT,2.368513e-07,2.368513e-07
...,...,...,...,...,...,...
4784,273.727413,5.972434e-193,192.223849,ZNRF3,2.785301e-07,2.785301e-07
4785,197.715584,3.187289e-173,172.496579,ZW10,1.442847e-07,1.442847e-07
4786,197.715584,3.187289e-173,172.496579,ZWILCH,1.768522e-07,1.768522e-07
4787,712.499566,4.559265e-251,250.341105,ZWINT,3.228468e-08,3.228468e-08


---

In [204]:
male_pathway_pval_df = pd.DataFrame(male_pathway_pvalues).rename(columns = {"pvalue" : "male_pval"})
male_pathway_pval_df

,statistic,male_pval
0,260.598681,5.738358e-190
1,91.774108,6.006174e-127
2,62.403780,4.585260e-104
3,276.384251,1.548562e-193
4,310.078077,1.611358e-200
...,...,...
390,165.576908,1.751210e-162
391,113.083925,1.787562e-139
392,122.474941,2.818496e-144
393,354.925839,1.009874e-208


In [205]:
male_pathway_pval_df["-log10(male_pval)"] = -np.log10(male_pathway_pval_df["male_pval"])

In [206]:
male_pathway_pval_df["pathway"] = male_pathway_importance_df.columns.values

In [207]:
male_pathway_importance_df.abs().mean().values

array([1.38293871e-04, 3.01986504e-05, 2.45965763e-05, 8.13798094e-05,
       8.39535272e-05, 1.95263715e-04, 4.01305304e-05, 7.00622109e-05,
       1.44325056e-04, 9.00226505e-05, 2.95946923e-05, 4.45678181e-05,
       2.34750062e-04, 1.59907260e-05, 1.21851612e-05, 9.71140964e-06,
       1.77031204e-04, 1.25513859e-04, 1.30521964e-05, 6.20475652e-05,
       1.92597105e-04, 1.98646493e-05, 2.70380944e-05, 2.60387259e-04,
       1.25674494e-04, 2.23340930e-04, 6.37129700e-06, 4.33947781e-05,
       8.64887367e-05, 8.35351192e-05, 1.83926300e-05, 2.29797821e-05,
       4.21664072e-05, 9.49787754e-05, 6.03520461e-05, 2.69154891e-04,
       3.24636999e-05, 7.46491143e-06, 1.23478365e-04, 1.44636675e-04,
       2.89457326e-04, 2.09603529e-05, 2.18765280e-05, 1.11533044e-04,
       5.17971745e-05, 4.88898507e-05, 8.17590291e-05, 8.08800567e-05,
       8.44574675e-05, 3.46173087e-04, 4.59036595e-05, 6.03284570e-05,
       7.37825992e-05, 5.07534111e-06, 1.26133129e-04, 1.90452840e-04,
      

In [208]:
male_pathway_pval_df["ABS_Importance"] = male_pathway_importance_df.abs().mean().values
male_pathway_pval_df

,statistic,male_pval,-log10(male_pval),pathway,ABS_Importance
0,260.598681,5.738358e-190,189.241212,KEGG_ABC_TRANSPORTERS,0.000138
1,91.774108,6.006174e-127,126.221402,KEGG_ACUTE_MYELOID_LEUKEMIA,0.000030
2,62.403780,4.585260e-104,103.338636,KEGG_ADHERENS_JUNCTION,0.000025
3,276.384251,1.548562e-193,192.810071,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,0.000081
4,310.078077,1.611358e-200,199.792808,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,0.000084
...,...,...,...,...,...
390,165.576908,1.751210e-162,161.756662,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,0.000047
391,113.083925,1.787562e-139,138.747739,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000035
392,122.474941,2.818496e-144,143.549983,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000061
393,354.925839,1.009874e-208,207.995733,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000165


In [209]:
male_pathway_pval_df["Importance"] = male_pathway_importance_df.mean().values
male_pathway_pval_df

,statistic,male_pval,-log10(male_pval),pathway,ABS_Importance,Importance
0,260.598681,5.738358e-190,189.241212,KEGG_ABC_TRANSPORTERS,0.000138,0.000138
1,91.774108,6.006174e-127,126.221402,KEGG_ACUTE_MYELOID_LEUKEMIA,0.000030,0.000030
2,62.403780,4.585260e-104,103.338636,KEGG_ADHERENS_JUNCTION,0.000025,0.000025
3,276.384251,1.548562e-193,192.810071,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,0.000081,0.000081
4,310.078077,1.611358e-200,199.792808,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,0.000084,0.000084
...,...,...,...,...,...,...
390,165.576908,1.751210e-162,161.756662,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,0.000047,0.000047
391,113.083925,1.787562e-139,138.747739,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000035,0.000035
392,122.474941,2.818496e-144,143.549983,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000061,0.000061
393,354.925839,1.009874e-208,207.995733,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000165,0.000165


---

---

### Female

In [210]:
female_gene_pval_df = pd.DataFrame(female_gene_pvalues).rename(columns = {"pvalue" : "female_pval"})
female_gene_pval_df

,statistic,female_pval
0,374.586731,1.294464e-164
1,105.684858,1.184278e-107
2,121.137989,9.098566e-114
3,166.169126,5.711892e-128
4,165.131685,1.092934e-127
...,...,...
4784,40.328821,2.555714e-65
4785,59.080969,7.776466e-82
4786,59.080969,7.776466e-82
4787,506.676595,2.999472e-178


In [211]:
female_gene_pval_df["-log10(female_pval)"] = -np.log10(female_gene_pval_df["female_pval"])

In [212]:
female_gene_pval_df["Gene"] = female_feature_importance_df.columns.values

In [213]:
female_gene_pval_df["ABS_Importance"] = female_feature_importance_df.abs().mean().values
female_gene_pval_df

,statistic,female_pval,-log10(female_pval),Gene,ABS_Importance
0,374.586731,1.294464e-164,163.887910,HMGB1P1,3.613371e-08
1,105.684858,1.184278e-107,106.926546,A2M,1.005518e-07
2,121.137989,9.098566e-114,113.041027,AACS,2.017065e-07
3,166.169126,5.711892e-128,127.243220,AADAT,1.159655e-07
4,165.131685,1.092934e-127,126.961406,AANAT,2.035128e-08
...,...,...,...,...,...
4784,40.328821,2.555714e-65,64.592488,ZNRF3,4.477053e-08
4785,59.080969,7.776466e-82,81.109218,ZW10,1.550857e-08
4786,59.080969,7.776466e-82,81.109218,ZWILCH,5.289988e-08
4787,506.676595,2.999472e-178,177.522955,ZWINT,2.665970e-07


In [214]:
female_gene_pval_df["Importance"] = female_feature_importance_df.mean().values
female_gene_pval_df

,statistic,female_pval,-log10(female_pval),Gene,ABS_Importance,Importance
0,374.586731,1.294464e-164,163.887910,HMGB1P1,3.613371e-08,3.613371e-08
1,105.684858,1.184278e-107,106.926546,A2M,1.005518e-07,1.005518e-07
2,121.137989,9.098566e-114,113.041027,AACS,2.017065e-07,2.017065e-07
3,166.169126,5.711892e-128,127.243220,AADAT,1.159655e-07,1.159655e-07
4,165.131685,1.092934e-127,126.961406,AANAT,2.035128e-08,2.035128e-08
...,...,...,...,...,...,...
4784,40.328821,2.555714e-65,64.592488,ZNRF3,4.477053e-08,4.477053e-08
4785,59.080969,7.776466e-82,81.109218,ZW10,1.550857e-08,1.550857e-08
4786,59.080969,7.776466e-82,81.109218,ZWILCH,5.289988e-08,5.289988e-08
4787,506.676595,2.999472e-178,177.522955,ZWINT,2.665970e-07,2.665970e-07


---

In [215]:
female_pathway_pval_df = pd.DataFrame(female_pathway_pvalues).rename(columns = {"pvalue" : "female_pval"})
female_pathway_pval_df

,statistic,female_pval
0,90.736759,7.724063e-101
1,379.978140,2.931634e-165
2,40.688473,1.071961e-65
3,400.431575,1.260851e-167
4,167.603096,2.344840e-128
...,...,...
390,143.751629,1.879673e-121
391,159.522234,3.921640e-126
392,159.428421,4.168026e-126
393,139.195480,5.264350e-120


In [216]:
female_pathway_pval_df["-log10(female_pval)"] = -np.log10(female_pathway_pval_df["female_pval"])

In [217]:
female_pathway_pval_df["pathway"] = female_pathway_importance_df.columns.values

In [218]:
female_pathway_importance_df.abs().mean().values

array([4.46845170e-05, 1.35071416e-04, 3.73081367e-05, 3.74000825e-04,
       1.36290111e-04, 1.64779677e-05, 1.83141824e-05, 6.76705383e-06,
       9.28878446e-06, 1.27240313e-04, 1.60398288e-04, 1.47198100e-04,
       1.43373696e-04, 3.79962013e-05, 1.15839902e-04, 6.04993716e-05,
       1.23706888e-04, 3.40657713e-05, 2.65166536e-04, 8.54205895e-05,
       5.47690144e-05, 3.23622259e-04, 1.94879833e-04, 2.05276296e-04,
       1.30210934e-04, 6.73648026e-05, 5.77945766e-06, 8.96126686e-05,
       2.33155464e-04, 2.33284418e-04, 1.48049513e-05, 1.24035251e-04,
       9.29111023e-05, 1.97204426e-04, 3.31028669e-05, 8.96573366e-05,
       5.84296033e-05, 3.33547279e-05, 9.30430755e-06, 1.45945960e-05,
       1.51624861e-04, 5.98995812e-06, 1.41768638e-04, 1.60854004e-04,
       1.91482870e-04, 6.44395437e-05, 1.42432611e-04, 1.31952099e-04,
       1.02802165e-04, 6.20738637e-05, 1.01384599e-04, 7.27409840e-05,
       7.34878451e-05, 2.72672285e-04, 3.69096059e-05, 3.26735173e-06,
      

In [219]:
female_pathway_pval_df["ABS_Importance"] = female_pathway_importance_df.abs().mean().values
female_pathway_pval_df

,statistic,female_pval,-log10(female_pval),pathway,ABS_Importance
0,90.736759,7.724063e-101,100.112154,KEGG_ABC_TRANSPORTERS,0.000045
1,379.978140,2.931634e-165,164.532890,KEGG_ACUTE_MYELOID_LEUKEMIA,0.000135
2,40.688473,1.071961e-65,64.969821,KEGG_ADHERENS_JUNCTION,0.000037
3,400.431575,1.260851e-167,166.899336,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,0.000374
4,167.603096,2.344840e-128,127.629887,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,0.000136
...,...,...,...,...,...
390,143.751629,1.879673e-121,120.725918,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,0.000032
391,159.522234,3.921640e-126,125.406532,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000151
392,159.428421,4.168026e-126,125.380070,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000046
393,139.195480,5.264350e-120,119.278655,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000062


In [220]:
female_pathway_pval_df["Importance"] = female_pathway_importance_df.mean().values
female_pathway_pval_df

,statistic,female_pval,-log10(female_pval),pathway,ABS_Importance,Importance
0,90.736759,7.724063e-101,100.112154,KEGG_ABC_TRANSPORTERS,0.000045,0.000045
1,379.978140,2.931634e-165,164.532890,KEGG_ACUTE_MYELOID_LEUKEMIA,0.000135,0.000135
2,40.688473,1.071961e-65,64.969821,KEGG_ADHERENS_JUNCTION,0.000037,0.000037
3,400.431575,1.260851e-167,166.899336,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,0.000374,0.000374
4,167.603096,2.344840e-128,127.629887,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,0.000136,0.000136
...,...,...,...,...,...,...
390,143.751629,1.879673e-121,120.725918,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,0.000032,0.000032
391,159.522234,3.921640e-126,125.406532,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000151,0.000151
392,159.428421,4.168026e-126,125.380070,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000046,0.000046
393,139.195480,5.264350e-120,119.278655,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000062,0.000062


---

---

# FDR-correction

---

---

# BH approach - controlling FDR

---

## Significant Genes

---

In [221]:
male_multi_test = multipletests(male_gene_pval_df["male_pval"], alpha=0.01, method='fdr_bh')

In [222]:
male_multi_test

(array([ True,  True,  True, ...,  True,  True,  True]),
 array([4.39185465e-260, 5.60708204e-051, 7.17876775e-160, ...,
        6.04751452e-173, 3.89898542e-249, 2.51351673e-200]),
 2.0986271261902445e-06,
 2.0881186051367717e-06)

In [223]:
male_gene_pval_df["FDR_BH"] = male_multi_test[1]
male_gene_pval_df["-log10(FDR_BH)"] = -np.log10(male_gene_pval_df["FDR_BH"])

In [224]:
male_gene_pval_df

,statistic,male_pval,-log10(male_pval),Gene,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
0,858.038948,2.292678e-262,261.639657,HMGB1P1,6.238669e-07,6.238669e-07,4.391855e-260,259.357352
1,23.771198,5.347159e-51,50.271877,A2M,5.936987e-09,5.936987e-09,5.607082e-51,50.251263
2,159.096034,4.552499e-160,159.341750,AACS,6.656735e-09,6.656735e-09,7.178768e-160,159.143950
3,280.382399,2.080887e-194,193.681751,AADAT,6.509067e-08,6.509067e-08,6.243966e-194,193.204539
4,238.529940,1.339279e-184,183.873129,AANAT,2.368513e-07,2.368513e-07,3.125637e-184,183.505061
...,...,...,...,...,...,...,...,...
4784,273.727413,5.972434e-193,192.223849,ZNRF3,2.785301e-07,2.785301e-07,1.674589e-192,191.776092
4785,197.715584,3.187289e-173,172.496579,ZW10,1.442847e-07,1.442847e-07,6.047515e-173,172.218423
4786,197.715584,3.187289e-173,172.496579,ZWILCH,1.768522e-07,1.768522e-07,6.047515e-173,172.218423
4787,712.499566,4.559265e-251,250.341105,ZWINT,3.228468e-08,3.228468e-08,3.898985e-249,248.409048


In [225]:
male_gene_pval_df.to_csv(f"[{date}_{num}]Male_Genes_p-values_FDR_Correction.csv", header = True, index = False)

---

In [226]:
female_multi_test = multipletests(female_gene_pval_df["female_pval"], alpha=0.01, method='fdr_bh')

In [227]:
female_multi_test

(array([ True,  True,  True, ...,  True,  True,  True]),
 array([1.67094074e-163, 1.70983063e-107, 1.51085411e-113, ...,
        9.25254499e-082, 1.63232642e-176, 2.28009868e-024]),
 2.0986271261902445e-06,
 2.0881186051367717e-06)

In [228]:
female_gene_pval_df["FDR_BH"] = female_multi_test[1]
female_gene_pval_df["-log10(FDR_BH)"] = -np.log10(female_gene_pval_df["FDR_BH"])

In [229]:
female_gene_pval_df

,statistic,female_pval,-log10(female_pval),Gene,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
0,374.586731,1.294464e-164,163.887910,HMGB1P1,3.613371e-08,3.613371e-08,1.670941e-163,162.777039
1,105.684858,1.184278e-107,106.926546,A2M,1.005518e-07,1.005518e-07,1.709831e-107,106.767047
2,121.137989,9.098566e-114,113.041027,AACS,2.017065e-07,2.017065e-07,1.510854e-113,112.820777
3,166.169126,5.711892e-128,127.243220,AADAT,1.159655e-07,1.159655e-07,1.249623e-127,126.903221
4,165.131685,1.092934e-127,126.961406,AANAT,2.035128e-08,2.035128e-08,2.301698e-127,126.637952
...,...,...,...,...,...,...,...,...
4784,40.328821,2.555714e-65,64.592488,ZNRF3,4.477053e-08,4.477053e-08,2.850330e-65,64.545105
4785,59.080969,7.776466e-82,81.109218,ZW10,1.550857e-08,1.550857e-08,9.252545e-82,81.033739
4786,59.080969,7.776466e-82,81.109218,ZWILCH,5.289988e-08,5.289988e-08,9.252545e-82,81.033739
4787,506.676595,2.999472e-178,177.522955,ZWINT,2.665970e-07,2.665970e-07,1.632326e-176,175.787193


In [230]:
female_gene_pval_df.to_csv(f"[{date}_{num}]Female_Genes_p-values_FDR_Correction.csv", header = True, index = False)

---

In [231]:
common_multi_test = multipletests(common_gene_pval_df["common_pval"], alpha=0.01, method='fdr_bh')

In [232]:
common_multi_test

(array([ True,  True,  True, ...,  True,  True,  True]),
 array([1.38617224e-295, 1.01343183e-234, 0.00000000e+000, ...,
        7.73698725e-290, 0.00000000e+000, 6.98733349e-176]),
 2.0986271261902445e-06,
 2.0881186051367717e-06)

In [233]:
common_gene_pval_df["FDR_BH"] = common_multi_test[1]
common_gene_pval_df["-log10(FDR_BH)"] = -np.log10(common_gene_pval_df["FDR_BH"])

/home/koe3/conda/envs/cl/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [234]:
common_gene_pval_df

,statistic,common_pval,-log10(common_pval),Gene,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
0,247.178236,7.589358e-296,295.119795,HMGB1P1,1.677213e-07,1.677213e-07,1.386172e-295,294.858183
1,138.707042,7.855208e-235,234.104842,A2M,1.324456e-10,1.324456e-10,1.013432e-234,233.994205
2,324.092444,0.000000e+00,inf,AACS,1.421752e-07,1.421752e-07,0.000000e+00,inf
3,287.857320,5.307535e-312,311.275107,AADAT,4.094231e-07,4.094231e-07,1.142372e-311,310.942192
4,218.923206,5.471425e-283,282.261900,AANAT,9.353754e-09,9.353754e-09,9.354749e-283,282.028968
...,...,...,...,...,...,...,...,...
4784,101.614705,2.709341e-202,201.567136,ZNRF3,6.821541e-09,6.821541e-09,3.250259e-202,201.488082
4785,234.101537,4.358821e-290,289.360631,ZW10,1.781477e-07,1.781477e-07,7.736987e-290,289.111428
4786,234.101537,4.358821e-290,289.360631,ZWILCH,7.894198e-08,7.894198e-08,7.736987e-290,289.111428
4787,476.938915,0.000000e+00,inf,ZWINT,2.525503e-07,2.525503e-07,0.000000e+00,inf


In [235]:
common_gene_pval_df.to_csv(f"[{date}_{num}]Common_Genes_p-values_FDR_Correction.csv", header = True, index = False)

---

In [236]:
common_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)

,statistic,common_pval,-log10(common_pval),Gene,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
4787,476.938915,0.000000e+00,inf,ZWINT,2.525503e-07,2.525503e-07,0.000000e+00,inf
4783,412.652957,0.000000e+00,inf,ZNF274,1.454005e-07,1.454005e-07,0.000000e+00,inf
4779,1029.690702,0.000000e+00,inf,ZFYVE9,3.114283e-08,3.114283e-08,0.000000e+00,inf
2,324.092444,0.000000e+00,inf,AACS,1.421752e-07,1.421752e-07,0.000000e+00,inf
4778,676.139580,0.000000e+00,inf,RBSN,7.027009e-08,7.027009e-08,0.000000e+00,inf
...,...,...,...,...,...,...,...,...
3234,16.161889,1.744360e-40,39.758364,PIF1,7.550197e-09,7.550197e-09,1.745818e-40,39.758001
2096,15.770953,3.760273e-39,38.424781,HRAS,5.228157e-08,5.228157e-08,3.762630e-39,38.424509
3204,15.580817,1.675163e-38,37.775943,PGF,8.840744e-09,8.840744e-09,1.675863e-38,37.775762
1140,14.747382,1.167185e-35,34.932860,CXCR5,3.392877e-08,3.392877e-08,1.167429e-35,34.932769


In [237]:
male_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)

,statistic,male_pval,-log10(male_pval),Gene,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
1368,951.141201,1.253409e-268,267.901907,DVL1,1.136109e-06,1.136109e-06,6.002574e-265,264.221662
1073,918.275930,1.721090e-266,265.764196,CSK,5.965139e-07,5.965139e-07,4.121149e-263,262.384982
273,896.507836,4.945757e-265,264.305767,ANGPT1,5.362229e-07,5.362229e-07,7.895077e-262,261.102644
4495,892.050695,9.936139e-265,264.002782,TRAIP,2.085064e-07,2.085064e-07,1.189604e-261,260.924598
1813,890.330086,1.301923e-264,263.885415,GIT1,6.616741e-07,6.616741e-07,1.246982e-261,260.904140
...,...,...,...,...,...,...,...,...
610,11.949085,3.990498e-23,22.398973,CACNG1,9.811970e-09,9.811970e-09,3.993834e-23,22.398610
597,11.889396,5.694209e-23,22.244567,CACNA1F,7.989132e-09,7.989132e-09,5.697778e-23,22.244294
3298,11.850910,7.161429e-23,22.145000,PLA2G2A,1.579191e-08,1.579191e-08,7.164421e-23,22.144819
4733,11.343837,1.468534e-21,20.833116,WNT5A,4.073160e-08,4.073160e-08,1.468840e-21,20.833025


In [238]:
female_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)

,statistic,female_pval,-log10(female_pval),Gene,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
1438,854.693898,7.355729e-202,201.133374,ELMO1,2.606597e-07,2.606597e-07,3.522659e-198,197.453129
2523,729.916750,9.847374e-195,194.006680,LRP5,1.047326e-06,1.047326e-06,2.357954e-191,190.627465
3476,719.798449,4.204253e-194,193.376311,PRKAG1,2.815580e-07,2.815580e-07,5.033542e-191,190.298126
2859,721.170919,3.448791e-194,193.462333,NDUFS1,9.379335e-07,9.379335e-07,5.033542e-191,190.298126
3389,704.880909,3.710166e-193,192.430607,POLR2K,3.702926e-07,3.702926e-07,2.961331e-190,189.528513
...,...,...,...,...,...,...,...,...
1205,10.546079,4.053562e-18,17.392163,DAPK2,1.752516e-08,1.752516e-08,4.055256e-18,17.391982
1206,10.546079,4.053562e-18,17.392163,DAPK3,3.465984e-10,3.465984e-10,4.055256e-18,17.391982
1204,10.546079,4.053562e-18,17.392163,DAPK1,3.521928e-09,3.521928e-09,4.055256e-18,17.391982
633,10.428160,7.432812e-18,17.128847,CAMK1,9.898419e-09,9.898419e-09,7.434364e-18,17.128756


In [242]:
common_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["Gene"].iloc[:15].values

array(['ZWINT', 'ZNF274', 'ZFYVE9', 'AACS', 'RBSN', 'ZFYVE16', 'ZFP36',
       'AASDHPPT', 'AASDH', 'WNT8B', 'WNT9A', 'WRAP53', 'WNT3A', 'WNT3',
       'WNT7A'], dtype=object)

In [243]:
male_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["Gene"].iloc[:15].values

array(['DVL1', 'CSK', 'ANGPT1', 'TRAIP', 'GIT1', 'ATP6V0C', 'PARP1',
       'HMGB1', 'NEIL3', 'NEIL2', 'NTHL1', 'PARP3', 'PARP2', 'OGG1',
       'APEX2'], dtype=object)

In [244]:
female_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["Gene"].iloc[:15].values

array(['ELMO1', 'LRP5', 'PRKAG1', 'NDUFS1', 'POLR2K', 'CSF2', 'GK2',
       'WIPF1', 'CREB3L4', 'CX3CL1', 'EED', 'PRKAG3', 'CCR10', 'KDM2B',
       'TRIM27'], dtype=object)

In [248]:
np.intersect1d(common_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["Gene"].iloc[:15].values, male_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["Gene"].iloc[:15].values)

array([], dtype=object)

In [249]:
np.intersect1d(common_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["Gene"].iloc[:15].values, female_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["Gene"].iloc[:15].values)

array([], dtype=object)

In [250]:
np.intersect1d(male_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["Gene"].iloc[:15].values, female_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["Gene"].iloc[:15].values)

array([], dtype=object)

---

---

---

In [283]:
significant_cmn_gene_df = cmn_gene_df[['ZWINT', 'ZNF274', 'ZFYVE9', 'AACS', 'RBSN', 'ZFYVE16', 'ZFP36', 'AASDHPPT', 'AASDH', 'WNT8B',
                                       'WNT9A', 'WRAP53', 'WNT3A', 'WNT3', 'WNT7A']]
significant_cmn_gene_df

,ZWINT,ZNF274,ZFYVE9,AACS,RBSN,ZFYVE16,ZFP36,AASDHPPT,AASDH,WNT8B,WNT9A,WRAP53,WNT3A,WNT3,WNT7A
0,2.493975e-07,-1.461214e-07,-3.053214e-08,-1.428225e-07,-7.055137e-08,-3.367502e-08,-3.791014e-07,-3.446902e-08,-7.421311e-07,3.941508e-07,0.000001,-3.421457e-07,3.670877e-07,-0.000002,-6.806924e-07
1,2.425640e-07,-1.378858e-07,-3.126189e-08,-1.405346e-07,-7.043025e-08,-3.447989e-08,-3.856710e-07,-3.389818e-08,-7.298408e-07,3.873113e-07,0.000001,-3.343586e-07,3.951176e-07,-0.000001,-6.783924e-07
2,2.500991e-07,-1.430642e-07,-3.055070e-08,-1.443677e-07,-6.830725e-08,-3.369549e-08,-3.897938e-07,-3.446627e-08,-7.420720e-07,3.720943e-07,0.000001,-3.425756e-07,3.620675e-07,-0.000002,-6.774139e-07
3,2.546801e-07,-1.450545e-07,-3.195782e-08,-1.466823e-07,-7.315651e-08,-3.524745e-08,-3.935047e-07,-3.494044e-08,-7.522810e-07,4.053553e-07,0.000001,-3.500726e-07,3.956652e-07,-0.000002,-6.995838e-07
4,2.391218e-07,-1.468351e-07,-3.093460e-08,-1.362884e-07,-7.023620e-08,-3.411890e-08,-3.976207e-07,-3.349569e-08,-7.211751e-07,4.018457e-07,0.000001,-3.508373e-07,4.042040e-07,-0.000002,-6.903204e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
241,2.457643e-07,-1.501580e-07,-3.140589e-08,-1.258283e-07,-6.769570e-08,-3.463871e-08,-4.006528e-07,-3.379771e-08,-7.276776e-07,3.984675e-07,0.000001,-3.475952e-07,4.018220e-07,-0.000002,-6.998058e-07
242,2.453299e-07,-1.435197e-07,-3.091795e-08,-1.396224e-07,-6.858413e-08,-3.410054e-08,-3.960260e-07,-3.338551e-08,-7.188029e-07,3.943670e-07,0.000001,-3.318441e-07,3.702230e-07,-0.000001,-6.618951e-07
243,2.577729e-07,-1.445101e-07,-3.090543e-08,-1.542950e-07,-6.771907e-08,-3.408673e-08,-3.854960e-07,-3.464135e-08,-7.458415e-07,3.786765e-07,0.000001,-3.361338e-07,3.872565e-07,-0.000002,-6.937704e-07
244,2.580680e-07,-1.455206e-07,-3.107222e-08,-1.453125e-07,-7.032818e-08,-3.427069e-08,-4.010696e-07,-3.436799e-08,-7.399559e-07,3.959582e-07,0.000001,-3.467941e-07,3.833336e-07,-0.000002,-6.888815e-07


In [284]:
np.sign(significant_cmn_gene_df).apply(pd.Series.value_counts)

,ZWINT,ZNF274,ZFYVE9,AACS,RBSN,ZFYVE16,ZFP36,AASDHPPT,AASDH,WNT8B,WNT9A,WRAP53,WNT3A,WNT3,WNT7A
-1.0,NaN,246.0,246.0,246.0,246.0,246.0,246.0,246.0,246.0,NaN,NaN,246.0,NaN,246.0,246.0
1.0,246.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,246.0,246.0,NaN,246.0,NaN,NaN


In [286]:
significant_male_gene_df = male_gene_df[['DVL1', 'CSK', 'ANGPT1', 'TRAIP', 'GIT1', 'ATP6V0C', 'PARP1', 'HMGB1', 'NEIL3',
                                         'NEIL2', 'NTHL1', 'PARP3', 'PARP2', 'OGG1', 'APEX2']]
significant_male_gene_df

,DVL1,CSK,ANGPT1,TRAIP,GIT1,ATP6V0C,PARP1,HMGB1,NEIL3,NEIL2,NTHL1,PARP3,PARP2,OGG1,APEX2
0,0.000001,-6.064187e-07,-5.386347e-07,2.048135e-07,6.676596e-07,5.294752e-07,-2.668785e-07,-6.213873e-07,0.000001,1.530064e-07,2.279382e-07,5.615480e-08,-4.104348e-09,-3.872411e-07,-4.816145e-07
1,0.000001,-5.945883e-07,-5.399125e-07,2.092921e-07,6.532391e-07,5.223014e-07,-2.667230e-07,-6.210252e-07,0.000001,1.529172e-07,2.278054e-07,5.612208e-08,-4.101957e-09,-3.870154e-07,-4.813339e-07
2,0.000001,-5.946712e-07,-5.357300e-07,2.128589e-07,6.583463e-07,5.304459e-07,-2.683538e-07,-6.248223e-07,0.000001,1.538522e-07,2.291982e-07,5.646522e-08,-4.127037e-09,-3.893817e-07,-4.842768e-07
3,0.000001,-6.032310e-07,-5.368711e-07,2.077156e-07,6.649270e-07,5.378778e-07,-2.669288e-07,-6.215044e-07,0.000001,1.530352e-07,2.279812e-07,5.616539e-08,-4.105122e-09,-3.873141e-07,-4.817053e-07
4,0.000001,-5.898412e-07,-5.466644e-07,2.072480e-07,6.388928e-07,5.163123e-07,-2.656717e-07,-6.185774e-07,0.000001,1.523145e-07,2.269075e-07,5.590087e-08,-4.085788e-09,-3.854900e-07,-4.794366e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136,0.000001,-6.060562e-07,-5.389728e-07,2.081436e-07,6.693064e-07,5.266752e-07,-2.649529e-07,-6.169038e-07,0.000001,1.519024e-07,2.262936e-07,5.574963e-08,-4.074734e-09,-3.844470e-07,-4.781396e-07
137,0.000001,-5.970144e-07,-5.295132e-07,2.099285e-07,6.622418e-07,5.267429e-07,-2.616237e-07,-6.091523e-07,0.000001,1.499937e-07,2.234501e-07,5.504912e-08,-4.023534e-09,-3.796163e-07,-4.721316e-07
138,0.000001,-5.993895e-07,-5.260042e-07,2.123800e-07,6.633550e-07,5.262655e-07,-2.686772e-07,-6.255754e-07,0.000001,1.540376e-07,2.294745e-07,5.653328e-08,-4.132011e-09,-3.898510e-07,-4.848605e-07
139,0.000001,-5.986045e-07,-5.324042e-07,2.092093e-07,6.594698e-07,5.311702e-07,-2.665973e-07,-6.207326e-07,0.000001,1.528452e-07,2.276981e-07,5.609564e-08,-4.100024e-09,-3.868331e-07,-4.811071e-07


In [287]:
np.sign(significant_male_gene_df).apply(pd.Series.value_counts)

,DVL1,CSK,ANGPT1,TRAIP,GIT1,ATP6V0C,PARP1,HMGB1,NEIL3,NEIL2,NTHL1,PARP3,PARP2,OGG1,APEX2
-1.0,NaN,141.0,141.0,NaN,NaN,NaN,141.0,141.0,NaN,NaN,NaN,NaN,141.0,141.0,141.0
1.0,141.0,NaN,NaN,141.0,141.0,141.0,NaN,NaN,141.0,141.0,141.0,141.0,NaN,NaN,NaN


In [289]:
significant_female_gene_df = female_gene_df[['ELMO1', 'LRP5', 'PRKAG1', 'NDUFS1', 'POLR2K', 'CSF2', 'GK2', 'WIPF1',
                                             'CREB3L4', 'CX3CL1', 'EED', 'PRKAG3', 'CCR10', 'KDM2B', 'TRIM27']]
significant_female_gene_df

,ELMO1,LRP5,PRKAG1,NDUFS1,POLR2K,CSF2,GK2,WIPF1,CREB3L4,CX3CL1,EED,PRKAG3,CCR10,KDM2B,TRIM27
0,-2.572785e-07,0.000001,-2.765470e-07,9.377643e-07,3.698254e-07,-4.340899e-07,-3.551341e-07,0.000001,0.000002,-4.092133e-07,8.808606e-07,0.000002,-2.328745e-07,-1.050919e-08,3.911507e-07
1,-2.584919e-07,0.000001,-2.796771e-07,9.439295e-07,3.719824e-07,-4.405300e-07,-3.633535e-07,0.000001,0.000002,-4.047355e-07,8.877166e-07,0.000002,-2.307075e-07,-1.077663e-08,4.011046e-07
2,-2.623930e-07,0.000001,-2.809026e-07,9.502319e-07,3.711908e-07,-4.335740e-07,-3.660691e-07,0.000001,0.000002,-4.179472e-07,9.001676e-07,0.000002,-2.291549e-07,-1.078990e-08,4.015986e-07
3,-2.608665e-07,0.000001,-2.772699e-07,9.301222e-07,3.659003e-07,-4.275170e-07,-3.633623e-07,0.000001,0.000002,-4.082010e-07,9.048727e-07,0.000002,-2.307011e-07,-1.073668e-08,3.996176e-07
4,-2.598684e-07,0.000001,-2.838273e-07,9.124216e-07,3.685060e-07,-4.310008e-07,-3.577534e-07,0.000001,0.000002,-4.186290e-07,8.923904e-07,0.000002,-2.293096e-07,-1.067084e-08,3.971672e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,-2.650767e-07,0.000001,-2.828307e-07,9.332153e-07,3.714521e-07,-4.377367e-07,-3.678903e-07,0.000001,0.000002,-4.107907e-07,9.009492e-07,0.000002,-2.370892e-07,-1.079296e-08,4.017123e-07
101,-2.612193e-07,0.000001,-2.804211e-07,9.505358e-07,3.713116e-07,-4.279141e-07,-3.567083e-07,0.000001,0.000002,-4.101879e-07,9.143002e-07,0.000002,-2.383405e-07,-1.073901e-08,3.997044e-07
102,-2.617530e-07,0.000001,-2.783451e-07,9.479030e-07,3.801485e-07,-4.196752e-07,-3.606450e-07,0.000001,0.000002,-4.013046e-07,8.968273e-07,0.000002,-2.284429e-07,-1.092265e-08,4.065396e-07
103,-2.623090e-07,0.000001,-2.830625e-07,9.257656e-07,3.775380e-07,-4.237423e-07,-3.573082e-07,0.000001,0.000002,-4.113608e-07,9.008251e-07,0.000002,-2.290843e-07,-1.079035e-08,4.016152e-07


In [290]:
np.sign(significant_female_gene_df).apply(pd.Series.value_counts)

,ELMO1,LRP5,PRKAG1,NDUFS1,POLR2K,CSF2,GK2,WIPF1,CREB3L4,CX3CL1,EED,PRKAG3,CCR10,KDM2B,TRIM27
-1.0,105.0,NaN,105.0,NaN,NaN,105.0,105.0,NaN,NaN,105.0,NaN,NaN,105.0,105.0,NaN
1.0,NaN,105.0,NaN,105.0,105.0,NaN,NaN,105.0,105.0,NaN,105.0,105.0,NaN,NaN,105.0


---

In [401]:
pd.set_option("display.max_rows", 11)

In [402]:
data

,HMGB1P1,A2M,AACS,AADAT,AANAT,AARS2,AASDHPPT,AASDH,AASS,ABAT,...,ZIC2,ZMAT2,ZMAT3,ZNF274,ZNRF3,ZW10,ZWILCH,ZWINT,ZYX,Sex
0,1.777716,0.389232,-1.177514,0.517646,-0.527263,0.475746,-1.120785,0.469608,-0.263035,0.845147,...,0.837401,0.514185,-0.578702,1.071558,0.765242,-0.439994,-1.164296,-1.448524,-0.809574,0
1,0.233934,0.707874,-0.794603,-0.908896,0.586772,-0.776142,-0.485973,-0.160776,0.959209,0.646833,...,1.963239,-0.155367,0.486913,-0.527983,0.277609,-0.035972,0.323295,-0.505197,0.818143,1
2,0.711468,0.824952,-0.271857,1.699544,-0.146400,-1.522602,0.322414,-0.092264,-1.469562,-0.260285,...,-1.301798,0.803925,0.508958,-0.842355,-0.304808,0.249729,-0.458007,0.128614,-0.213299,0
3,1.791092,0.007414,-0.303279,-0.655909,-1.619284,0.736664,0.497619,-1.197012,-1.604390,-0.459375,...,-1.647006,0.889620,-0.325959,1.012508,-1.208278,1.686594,2.562832,2.373111,-0.283270,1
4,1.157604,-0.112626,-0.342409,-1.595685,-0.101821,-0.458447,-0.456942,-0.001335,-2.456418,0.024421,...,0.952211,1.766846,-0.962883,-1.490118,0.266920,-0.494128,-1.748391,-1.310929,-0.705312,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
241,-0.399459,-1.481720,2.889208,-0.322603,-0.221191,1.158866,0.301227,-0.936558,0.256070,-0.789024,...,-0.383512,-0.226762,-1.715259,-1.081246,0.072150,-0.608469,0.286122,0.132115,-0.470074,0
242,-0.413783,-0.370962,-0.024195,1.461690,0.159579,-0.219504,-0.095633,0.518314,0.278427,0.283403,...,0.264129,0.068849,-0.081157,1.076537,0.171947,-1.680221,-0.745575,-0.659906,-0.929458,1
243,-0.218842,0.715807,-0.877743,1.792280,-0.769873,0.728809,0.759816,0.728172,0.676274,-0.173310,...,-0.776744,0.608524,-1.197759,1.242038,-1.331913,1.028516,0.769277,0.889907,-0.837304,1
244,-0.506558,-0.106510,-0.508298,1.856691,0.339280,0.724135,1.219016,0.623828,0.863336,0.192061,...,1.101532,0.308432,-1.414768,2.049526,0.537743,2.332232,-0.836792,-0.923038,-0.467301,1


In [409]:
label

,Label
0,1
1,0
2,1
3,1
4,1
...,...
241,1
242,1
243,1
244,1


In [410]:
df = pd.concat([data, label], axis=1)
df

,HMGB1P1,A2M,AACS,AADAT,AANAT,AARS2,AASDHPPT,AASDH,AASS,ABAT,...,ZMAT2,ZMAT3,ZNF274,ZNRF3,ZW10,ZWILCH,ZWINT,ZYX,Sex,Label
0,1.777716,0.389232,-1.177514,0.517646,-0.527263,0.475746,-1.120785,0.469608,-0.263035,0.845147,...,0.514185,-0.578702,1.071558,0.765242,-0.439994,-1.164296,-1.448524,-0.809574,0,1
1,0.233934,0.707874,-0.794603,-0.908896,0.586772,-0.776142,-0.485973,-0.160776,0.959209,0.646833,...,-0.155367,0.486913,-0.527983,0.277609,-0.035972,0.323295,-0.505197,0.818143,1,0
2,0.711468,0.824952,-0.271857,1.699544,-0.146400,-1.522602,0.322414,-0.092264,-1.469562,-0.260285,...,0.803925,0.508958,-0.842355,-0.304808,0.249729,-0.458007,0.128614,-0.213299,0,1
3,1.791092,0.007414,-0.303279,-0.655909,-1.619284,0.736664,0.497619,-1.197012,-1.604390,-0.459375,...,0.889620,-0.325959,1.012508,-1.208278,1.686594,2.562832,2.373111,-0.283270,1,1
4,1.157604,-0.112626,-0.342409,-1.595685,-0.101821,-0.458447,-0.456942,-0.001335,-2.456418,0.024421,...,1.766846,-0.962883,-1.490118,0.266920,-0.494128,-1.748391,-1.310929,-0.705312,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
241,-0.399459,-1.481720,2.889208,-0.322603,-0.221191,1.158866,0.301227,-0.936558,0.256070,-0.789024,...,-0.226762,-1.715259,-1.081246,0.072150,-0.608469,0.286122,0.132115,-0.470074,0,1
242,-0.413783,-0.370962,-0.024195,1.461690,0.159579,-0.219504,-0.095633,0.518314,0.278427,0.283403,...,0.068849,-0.081157,1.076537,0.171947,-1.680221,-0.745575,-0.659906,-0.929458,1,1
243,-0.218842,0.715807,-0.877743,1.792280,-0.769873,0.728809,0.759816,0.728172,0.676274,-0.173310,...,0.608524,-1.197759,1.242038,-1.331913,1.028516,0.769277,0.889907,-0.837304,1,1
244,-0.506558,-0.106510,-0.508298,1.856691,0.339280,0.724135,1.219016,0.623828,0.863336,0.192061,...,0.308432,-1.414768,2.049526,0.537743,2.332232,-0.836792,-0.923038,-0.467301,1,1


In [412]:
lts_df = df[df['Label'] == 1]
nlts_df = df[df['Label'] == 0]

In [417]:
significant_cmn_gene_df = lts_df[['ARG2', 'P4HA2', 'OAT', 'LAP3', 'PYCR2']]
significant_cmn_gene_df

,ARG2,P4HA2,OAT,LAP3,PYCR2
0,-0.757234,-0.493408,0.567935,0.773066,1.671560
2,-1.141306,-1.694909,0.184990,1.011812,-0.484031
3,-0.363951,-1.282228,-0.051062,0.360129,-0.844247
4,-1.126196,0.413104,0.395017,0.923061,2.335249
5,0.679285,-0.722623,-0.052122,-0.041162,-0.658168
...,...,...,...,...,...
241,0.556539,0.394785,1.565734,-1.645623,-0.159741
242,-0.623621,-1.322756,1.624819,0.104214,-0.209193
243,-0.422115,-1.858729,-0.368946,0.095580,-0.719983
244,-0.206741,0.355397,-1.009824,-0.656903,-0.607766


In [419]:
significant_cmn_gene_df = nlts_df[['ARG2', 'P4HA2', 'OAT', 'LAP3', 'PYCR2']]
significant_cmn_gene_df

,ARG2,P4HA2,OAT,LAP3,PYCR2
1,-0.794492,-0.733443,-2.374664,1.609984,1.208108
7,-0.390963,-0.348856,0.899314,0.764532,-2.416982
9,0.642027,1.091445,0.003879,0.605000,-0.727591
10,-0.523437,-1.411120,-0.797001,1.159598,-0.290873
11,0.018362,1.721856,-1.226791,1.289614,-0.702231
...,...,...,...,...,...
230,0.274100,0.256118,0.285714,-1.015525,-0.170413
231,-1.124954,-1.755653,0.044650,-0.286234,0.336150
232,1.167681,1.234384,-0.048652,1.412401,0.892694
235,-0.373472,0.164717,-1.123368,4.057788,-0.247972


In [420]:
np.sign(significant_cmn_gene_df).apply(pd.Series.value_counts)

,ARG2,P4HA2,OAT,LAP3,PYCR2
-1.0,46,30,58,27,44
1.0,33,49,21,52,35


In [275]:
np.sign(significant_cmn_gene_df.iloc[:, :15]).apply(pd.Series.value_counts)

,ARG2,P4HA2,OAT,LAP3,PYCR2,CKM,PYCR3,PYCR1,SAT1,SAT2,CKB,AZIN2,NAGS,PRODH,AGMAT
-1.0,NaN,NaN,NaN,246.0,246.0,246.0,246.0,NaN,246.0,NaN,NaN,NaN,246.0,246.0,NaN
1.0,246.0,246.0,246.0,NaN,NaN,NaN,NaN,246.0,NaN,246.0,246.0,246.0,NaN,NaN,246.0


In [405]:
significant_male_gene_df = data[['ATIC', 'PRKAG1', 'PRPS1L1', 'ABCB6', 'ABCB7']]
significant_male_gene_df

,ATIC,PRKAG1,PRPS1L1,ABCB6,ABCB7
0,2.595742,1.236613,0.293187,-0.406353,0.734279
1,0.206469,-1.671234,2.832021,-0.181077,0.804089
2,0.990807,1.092926,3.690276,-0.498562,0.546298
3,1.614328,0.463702,0.789314,-0.674014,0.935344
4,1.765549,3.094616,2.509040,0.443705,-0.048960
...,...,...,...,...,...
241,0.052203,-2.508751,-1.047370,0.765572,-1.542733
242,0.710508,0.167542,-0.114840,0.663173,-0.476288
243,0.652474,-1.146002,-0.259933,0.220059,1.098983
244,-2.019313,-0.316961,-1.047370,0.459396,1.680193


In [406]:
np.sign(significant_male_gene_df).apply(pd.Series.value_counts)

,ATIC,PRKAG1,PRPS1L1,ABCB6,ABCB7
-1.0,119,119,141,119,114
1.0,127,127,105,127,132


In [276]:
np.sign(significant_male_gene_df.iloc[:, :15]).apply(pd.Series.value_counts)

,ATIC,PRKAG1,PRPS1L1,ABCB6,ABCB7,ABCC10,ABCC2,ABCC3,ABCA8,ABCC12,ABCB5,ABCA3,ABCC1,ABCG4,ABCG5
-1.0,NaN,141.0,NaN,NaN,141.0,NaN,141.0,141.0,141.0,141.0,NaN,NaN,141.0,NaN,141.0
1.0,141.0,NaN,141.0,141.0,NaN,141.0,NaN,NaN,NaN,NaN,141.0,141.0,NaN,141.0,NaN


In [407]:
significant_female_gene_df = data[['HLA-DMA', 'RAD23A', 'CD40LG', 'CD80', 'IL4']]
significant_female_gene_df

,HLA-DMA,RAD23A,CD40LG,CD80,IL4
0,-0.318480,-0.359545,-0.436322,-0.798112,-0.412091
1,1.167376,0.382150,1.883167,1.913271,-0.412091
2,1.183984,-0.723027,0.303551,1.210233,-0.412091
3,0.059778,1.202476,-0.533534,0.181138,-0.412091
4,1.060280,-0.075730,-0.936367,-1.207483,-0.412091
...,...,...,...,...,...
241,-1.365384,0.960050,-0.936367,-0.625187,-0.412091
242,0.001298,-0.627385,-0.936367,-0.712369,-0.412091
243,0.858670,-0.262554,0.492360,-0.789385,-0.412091
244,-0.178974,0.112547,-0.447353,-0.509573,1.867703


In [408]:
np.sign(significant_female_gene_df).apply(pd.Series.value_counts)

,HLA-DMA,RAD23A,CD40LG,CD80,IL4
-1.0,127,138,156,147,206
1.0,119,108,90,99,40


In [277]:
np.sign(significant_female_gene_df.iloc[:,:15]).apply(pd.Series.value_counts)

,HLA-DMA,RAD23A,CD40LG,CD80,IL4,KIR3DL2,GNB4,HROB,BRIP1,DPF1,DPF3,SMARCC2,ARID1A,KDM3B,EED
-1.0,105.0,105.0,NaN,105.0,105.0,NaN,105.0,NaN,NaN,105.0,105.0,105.0,105.0,NaN,105.0
1.0,NaN,NaN,105.0,NaN,NaN,105.0,NaN,105.0,105.0,NaN,NaN,NaN,NaN,105.0,NaN


---

---

---

## Significant Pathways

---

In [251]:
male_multi_test = multipletests(male_pathway_pval_df["male_pval"], alpha=0.01, method='fdr_bh')

In [252]:
male_multi_test

(array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
      

In [253]:
male_pathway_pval_df["FDR_BH"] = male_multi_test[1]
male_pathway_pval_df["-log10(FDR_BH)"] = -np.log10(male_pathway_pval_df["FDR_BH"])

In [254]:
male_pathway_pval_df

,statistic,male_pval,-log10(male_pval),pathway,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
0,260.598681,5.738358e-190,189.241212,KEGG_ABC_TRANSPORTERS,0.000138,0.000138,1.425567e-189,188.846012
1,91.774108,6.006174e-127,126.221402,KEGG_ACUTE_MYELOID_LEUKEMIA,0.000030,0.000030,7.603970e-127,126.118960
2,62.403780,4.585260e-104,103.338636,KEGG_ADHERENS_JUNCTION,0.000025,0.000025,5.422688e-104,103.265785
3,276.384251,1.548562e-193,192.810071,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,0.000081,0.000081,4.161101e-193,192.380792
4,310.078077,1.611358e-200,199.792808,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,0.000084,0.000084,5.486953e-200,199.260669
...,...,...,...,...,...,...,...,...
390,165.576908,1.751210e-162,161.756662,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,0.000047,0.000047,2.894260e-162,161.538462
391,113.083925,1.787562e-139,138.747739,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000035,0.000035,2.401656e-139,138.619489
392,122.474941,2.818496e-144,143.549983,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000061,0.000061,3.892678e-144,143.409752
393,354.925839,1.009874e-208,207.995733,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000165,0.000165,4.532957e-208,207.343618


In [255]:
male_pathway_pval_df.to_csv(f"[{date}_{num}]Male_Pathways_p-values_FDR_Correction.csv", header = True, index = False)

---

In [256]:
female_multi_test = multipletests(female_pathway_pval_df["female_pval"], alpha=0.01, method='fdr_bh')

In [257]:
female_multi_test

(array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
      

In [258]:
female_pathway_pval_df["FDR_BH"] = female_multi_test[1]
female_pathway_pval_df["-log10(FDR_BH)"] = -np.log10(female_pathway_pval_df["FDR_BH"])

In [259]:
female_pathway_pval_df

,statistic,female_pval,-log10(female_pval),pathway,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
0,90.736759,7.724063e-101,100.112154,KEGG_ABC_TRANSPORTERS,0.000045,0.000045,1.101446e-100,99.958037
1,379.978140,2.931634e-165,164.532890,KEGG_ACUTE_MYELOID_LEUKEMIA,0.000135,0.000135,2.894988e-164,163.538353
2,40.688473,1.071961e-65,64.969821,KEGG_ADHERENS_JUNCTION,0.000037,0.000037,1.238083e-65,64.907250
3,400.431575,1.260851e-167,166.899336,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,0.000374,0.000374,1.556363e-166,165.807889
4,167.603096,2.344840e-128,127.629887,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,0.000136,0.000136,5.061266e-128,127.295741
...,...,...,...,...,...,...,...,...
390,143.751629,1.879673e-121,120.725918,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,0.000032,0.000032,3.535576e-121,120.451540
391,159.522234,3.921640e-126,125.406532,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000151,0.000151,7.863186e-126,125.104401
392,159.428421,4.168026e-126,125.380070,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000046,0.000046,8.315002e-126,125.080138
393,139.195480,5.264350e-120,119.278655,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000062,0.000062,9.671712e-120,119.014497


In [260]:
female_pathway_pval_df.to_csv(f"[{date}_{num}]Female_Pathways_p-values_FDR_Correction.csv", header = True, index = False)

---

In [261]:
common_multi_test = multipletests(common_pathway_pval_df["common_pval"], alpha=0.01, method='fdr_bh')

In [262]:
common_multi_test

(array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
      

In [263]:
common_pathway_pval_df["FDR_BH"] = common_multi_test[1]
common_pathway_pval_df["-log10(FDR_BH)"] = -np.log10(common_pathway_pval_df["FDR_BH"])

/home/koe3/conda/envs/cl/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [264]:
common_pathway_pval_df

,statistic,common_pval,-log10(common_pval),pathway,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
0,408.963138,0.000000e+00,inf,KEGG_ABC_TRANSPORTERS,0.000116,0.000116,0.000000e+00,inf
1,119.139799,6.903559e-219,218.160927,KEGG_ACUTE_MYELOID_LEUKEMIA,0.000067,0.000067,8.629449e-219,218.064017
2,506.814709,0.000000e+00,inf,KEGG_ADHERENS_JUNCTION,0.000194,0.000194,0.000000e+00,inf
3,286.493945,1.692587e-311,310.771449,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,0.000098,0.000098,3.633543e-311,310.439670
4,251.800306,8.253290e-298,297.083373,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,0.000199,0.000199,1.559832e-297,296.806922
...,...,...,...,...,...,...,...,...
390,381.061377,0.000000e+00,inf,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,0.000155,0.000155,0.000000e+00,inf
391,318.132559,1.284571e-322,321.891242,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000056,0.000056,3.162020e-322,321.500035
392,49.597084,9.167552e-130,129.037747,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000027,0.000027,9.921050e-130,129.003442
393,38.159532,4.438809e-105,104.352734,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000007,0.000007,4.663110e-105,104.331324


In [265]:
common_pathway_pval_df.to_csv(f"[{date}_{num}]Common_Pathways_p-values_FDR_Correction.csv", header = True, index = False)

---

In [266]:
common_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)

,statistic,common_pval,-log10(common_pval),pathway,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
390,381.061377,0.000000e+00,inf,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,0.000155,0.000155,0.000000e+00,inf
0,408.963138,0.000000e+00,inf,KEGG_ABC_TRANSPORTERS,0.000116,0.000116,0.000000e+00,inf
388,509.448312,0.000000e+00,inf,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_UBQL...,0.000110,0.000110,0.000000e+00,inf
2,506.814709,0.000000e+00,inf,KEGG_ADHERENS_JUNCTION,0.000194,0.000194,0.000000e+00,inf
354,460.935401,0.000000e+00,inf,KEGG_MEDICUS_VARIANT_MUTATION_ACTIVATED_GNAS_T...,0.000148,0.000148,0.000000e+00,inf
...,...,...,...,...,...,...,...,...
378,19.312260,3.938427e-51,50.404677,KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_...,0.000005,0.000005,3.978718e-51,50.400257
180,19.221960,7.880142e-51,50.103466,KEGG_MEDICUS_PATHOGEN_ESCHERICHIA_EAE_TIR_TO_A...,0.000003,0.000003,7.940449e-51,50.100155
339,19.150033,1.369844e-50,49.863329,KEGG_MEDICUS_REFERENCE_TYPE_I_INTERFERON_TO_JA...,0.000003,0.000003,1.376815e-50,49.861124
316,16.548039,8.425919e-42,41.074383,KEGG_MEDICUS_REFERENCE_RIG_I_IRF7_3_SIGNALING_...,0.000007,0.000007,8.447305e-42,41.073282


In [267]:
male_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)

,statistic,male_pval,-log10(male_pval),pathway,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
363,920.649283,1.199187e-266,265.921113,KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_...,0.000169,0.000169,4.736787e-264,263.324516
333,894.888792,6.369666e-265,264.195883,KEGG_MEDICUS_REFERENCE_TRAIP_DEPENDENT_REPLISO...,0.000228,0.000228,1.258009e-262,261.900316
23,879.523636,7.193633e-264,263.143052,KEGG_BASE_EXCISION_REPAIR,0.000260,0.000260,9.471616e-262,261.023576
259,851.416277,6.782050e-262,261.168639,KEGG_MEDICUS_REFERENCE_GF_RTK_PI3K_SIGNALING_P...,0.000238,0.000238,6.697274e-260,259.174102
323,826.256292,4.516523e-260,259.345196,KEGG_MEDICUS_REFERENCE_TIGHT_JUNCTION_ACTIN_SI...,0.000182,0.000182,3.568053e-258,257.447569
...,...,...,...,...,...,...,...,...
216,16.145756,8.964602e-34,33.047469,KEGG_MEDICUS_REFERENCE_BMP_SIGNALING_PATHWAY,0.000004,0.000004,9.056311e-34,33.043049
217,14.569040,7.671844e-30,29.115100,KEGG_MEDICUS_REFERENCE_BRANCHING_MICROTUBULE_N...,0.000003,0.000003,7.730557e-30,29.111789
105,13.349767,9.772655e-27,26.009987,KEGG_NUCLEOTIDE_EXCISION_REPAIR,0.000004,0.000004,9.822388e-27,26.007783
147,13.343057,1.016766e-26,25.992779,KEGG_SMALL_CELL_LUNG_CANCER,0.000003,0.000003,1.019347e-26,25.991678


In [268]:
female_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)

,statistic,female_pval,-log10(female_pval),pathway,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
180,760.484127,1.382658e-196,195.859285,KEGG_MEDICUS_PATHOGEN_ESCHERICHIA_EAE_TIR_TO_A...,0.000277,0.000277,5.461498e-194,193.262688
232,632.605946,2.847346e-188,187.545560,KEGG_MEDICUS_REFERENCE_CX3CR1_GNAI_AC_PKA_SIGN...,0.000246,0.000246,4.335232e-186,185.362988
204,631.722607,3.292581e-188,187.482463,KEGG_MEDICUS_REFERENCE_ACTIVATION_OF_PRC2.2_BY...,0.000267,0.000267,4.335232e-186,185.362988
190,620.960375,1.965332e-187,186.706564,KEGG_MEDICUS_PATHOGEN_HIV_TAT_TO_TLR2_4_NFKB_S...,0.000312,0.000312,1.940766e-185,184.712027
347,547.195344,1.008174e-181,180.996464,KEGG_MEDICUS_VARIANT_FZD7_OVEREXPRESSION_TO_WN...,0.000393,0.000393,7.964575e-180,179.098837
...,...,...,...,...,...,...,...,...
341,12.097802,1.468021e-21,20.833268,KEGG_MEDICUS_REFERENCE_WNT5A_ROR_SIGNALING_PAT...,0.000005,0.000005,1.483039e-21,20.828847
247,11.290271,8.925388e-20,19.049373,KEGG_MEDICUS_REFERENCE_E2_ER_RAS_ERK_SIGNALING...,0.000009,0.000009,8.993694e-20,19.046062
286,11.193165,1.466519e-19,18.833712,KEGG_MEDICUS_REFERENCE_KINETOCHORE_MICROTUBULE...,0.000007,0.000007,1.473982e-19,18.831508
206,11.038100,3.243975e-19,18.488923,KEGG_MEDICUS_REFERENCE_ADRB3_UCP1_SIGNALING_PA...,0.000003,0.000003,3.252208e-19,18.487822


In [271]:
common_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:15].values

array(['KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_NOTCH_SIGNALING_PATHWAY',
       'KEGG_ABC_TRANSPORTERS',
       'KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_UBQLN2_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION',
       'KEGG_ADHERENS_JUNCTION',
       'KEGG_MEDICUS_VARIANT_MUTATION_ACTIVATED_GNAS_TO_CRHR_PKA_ACTH_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_WNT5A_ROR_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_WNT_SIGNALING_MODULATION_WNT_ACYLATION',
       'KEGG_MEDICUS_REFERENCE_WNT_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_VARIANT_CYP11B1_CYP11B2_FUSION_TO_ACTH_CORTISOL_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_VARIANT_FZD7_OVEREXPRESSION_TO_WNT_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_TNF_P38_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_TRANSCRIPTION_COUPLED_NER',
       'KEGG_MEDICUS_REFERENCE_TRANSLATION_INITIATION',
       'KEGG_MEDICUS_REFERENCE_TSH_TG_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_TYPE_I_IFN_SIGNALING_PATHWAY'],
      dtype=

In [272]:
male_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:15].values

array(['KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_ABETA_TO_ELECTRON_TRANSFER_IN_COMPLEX_IV',
       'KEGG_MEDICUS_REFERENCE_TRAIP_DEPENDENT_REPLISOME_DISASSEMBLY',
       'KEGG_BASE_EXCISION_REPAIR',
       'KEGG_MEDICUS_REFERENCE_GF_RTK_PI3K_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_TIGHT_JUNCTION_ACTIN_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_GLOBAL_GENOME_NER',
       'KEGG_MEDICUS_PATHOGEN_HTLV_1_TAX_TO_SPINDLE_ASSEMBLY_CHECKPOINT_SIGNALING',
       'KEGG_MEDICUS_REFERENCE_IFN_RIPK1_3_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_TUBA4A_TO_RETROGRADE_AXONAL_TRANSPORT',
       'KEGG_MEDICUS_REFERENCE_ORGANIZATION_OF_THE_OUTER_KINETOCHORE',
       'KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PTCH1_TO_HEDGEHOG_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_TNF_P38_SIGNALING_PATHWAY',
       'KEGG_VIBRIO_CHOLERAE_INFECTION', 'KEGG_MELANOGENESIS',
       'KEGG_MEDICUS_REFERENCE_DISASSEMBLY_OF_MCC'], dtype=object)

In [273]:
female_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:15].values

array(['KEGG_MEDICUS_PATHOGEN_ESCHERICHIA_EAE_TIR_TO_ACTIN_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_CX3CR1_GNAI_AC_PKA_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_ACTIVATION_OF_PRC2.2_BY_UBIQUITINATION_OF_H2AK119',
       'KEGG_MEDICUS_PATHOGEN_HIV_TAT_TO_TLR2_4_NFKB_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_VARIANT_FZD7_OVEREXPRESSION_TO_WNT_SIGNALING_PATHWAY',
       'KEGG_ASTHMA',
       'KEGG_MEDICUS_REFERENCE_HOMOLOGOUS_RECOMBINATION_IN_ICLR',
       'KEGG_MEDICUS_REFERENCE_KEAP1_NRF2_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_ORGANIZATION_OF_THE_OUTER_KINETOCHORE',
       'KEGG_MEDICUS_REFERENCE_TIGHT_JUNCTION_ACTIN_SIGNALING_PATHWAY',
       'KEGG_B_CELL_RECEPTOR_SIGNALING_PATHWAY',
       'KEGG_NICOTINATE_AND_NICOTINAMIDE_METABOLISM',
       'KEGG_MEDICUS_REFERENCE_ITGA_B_FAK_RAC_SIGNALING_PATHWAY',
       'KEGG_AMYOTROPHIC_LATERAL_SCLEROSIS_ALS',
       'KEGG_MEDICUS_REFERENCE_PRNP_PI3K_NOX2_SIGNALING_PATHWAY'],
      dtype=object)

In [280]:
np.intersect1d(common_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:10].values, male_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:10].values)

array([], dtype=object)

In [281]:
np.intersect1d(common_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:10].values, female_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:10].values)

array(['KEGG_MEDICUS_VARIANT_FZD7_OVEREXPRESSION_TO_WNT_SIGNALING_PATHWAY'],
      dtype=object)

In [282]:
np.intersect1d(male_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:10].values, female_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:10].values)

array(['KEGG_MEDICUS_REFERENCE_ORGANIZATION_OF_THE_OUTER_KINETOCHORE',
       'KEGG_MEDICUS_REFERENCE_TIGHT_JUNCTION_ACTIN_SIGNALING_PATHWAY'],
      dtype=object)

In [277]:
np.intersect1d(common_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:15].values, male_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:15].values)

array(['KEGG_MEDICUS_REFERENCE_TNF_P38_SIGNALING_PATHWAY'], dtype=object)

In [278]:
np.intersect1d(common_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:15].values, female_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:15].values)

array(['KEGG_MEDICUS_VARIANT_FZD7_OVEREXPRESSION_TO_WNT_SIGNALING_PATHWAY'],
      dtype=object)

In [279]:
np.intersect1d(male_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:15].values, female_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)["pathway"].iloc[:15].values)

array(['KEGG_MEDICUS_REFERENCE_ORGANIZATION_OF_THE_OUTER_KINETOCHORE',
       'KEGG_MEDICUS_REFERENCE_TIGHT_JUNCTION_ACTIN_SIGNALING_PATHWAY'],
      dtype=object)

---

In [6]:
male_pathway_pval_df = pd.read_csv(f"[0825_1]Male_Pathways_p-values_FDR_Correction.csv")
male_pathway_pval_df

,statistic,male_pval,-log10(male_pval),pathway,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
0,260.598681,5.738358e-190,189.241212,KEGG_ABC_TRANSPORTERS,0.000138,0.000138,1.425567e-189,188.846012
1,91.774108,6.006174e-127,126.221402,KEGG_ACUTE_MYELOID_LEUKEMIA,0.000030,0.000030,7.603970e-127,126.118960
2,62.403780,4.585260e-104,103.338636,KEGG_ADHERENS_JUNCTION,0.000025,0.000025,5.422688e-104,103.265785
3,276.384251,1.548562e-193,192.810071,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,0.000081,0.000081,4.161101e-193,192.380792
4,310.078077,1.611358e-200,199.792808,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,0.000084,0.000084,5.486953e-200,199.260669
...,...,...,...,...,...,...,...,...
390,165.576908,1.751210e-162,161.756662,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,0.000047,0.000047,2.894260e-162,161.538462
391,113.083925,1.787562e-139,138.747739,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000035,0.000035,2.401656e-139,138.619489
392,122.474941,2.818496e-144,143.549983,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000061,0.000061,3.892678e-144,143.409752
393,354.925839,1.009874e-208,207.995733,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000165,0.000165,4.532957e-208,207.343618


In [267]:
male_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)

,statistic,male_pval,-log10(male_pval),pathway,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
363,920.649283,1.199187e-266,265.921113,KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_...,0.000169,0.000169,4.736787e-264,263.324516
333,894.888792,6.369666e-265,264.195883,KEGG_MEDICUS_REFERENCE_TRAIP_DEPENDENT_REPLISO...,0.000228,0.000228,1.258009e-262,261.900316
23,879.523636,7.193633e-264,263.143052,KEGG_BASE_EXCISION_REPAIR,0.000260,0.000260,9.471616e-262,261.023576
259,851.416277,6.782050e-262,261.168639,KEGG_MEDICUS_REFERENCE_GF_RTK_PI3K_SIGNALING_P...,0.000238,0.000238,6.697274e-260,259.174102
323,826.256292,4.516523e-260,259.345196,KEGG_MEDICUS_REFERENCE_TIGHT_JUNCTION_ACTIN_SI...,0.000182,0.000182,3.568053e-258,257.447569
...,...,...,...,...,...,...,...,...
216,16.145756,8.964602e-34,33.047469,KEGG_MEDICUS_REFERENCE_BMP_SIGNALING_PATHWAY,0.000004,0.000004,9.056311e-34,33.043049
217,14.569040,7.671844e-30,29.115100,KEGG_MEDICUS_REFERENCE_BRANCHING_MICROTUBULE_N...,0.000003,0.000003,7.730557e-30,29.111789
105,13.349767,9.772655e-27,26.009987,KEGG_NUCLEOTIDE_EXCISION_REPAIR,0.000004,0.000004,9.822388e-27,26.007783
147,13.343057,1.016766e-26,25.992779,KEGG_SMALL_CELL_LUNG_CANCER,0.000003,0.000003,1.019347e-26,25.991678


In [8]:
male_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)[:30]

,statistic,male_pval,-log10(male_pval),pathway,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
363,920.649283,1.199187e-266,265.921113,KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_...,0.000169,0.000169,4.736787e-264,263.324516
333,894.888792,6.369666e-265,264.195883,KEGG_MEDICUS_REFERENCE_TRAIP_DEPENDENT_REPLISO...,0.000228,0.000228,1.258009e-262,261.900316
23,879.523636,7.193633e-264,263.143052,KEGG_BASE_EXCISION_REPAIR,0.000260,0.000260,9.471616e-262,261.023576
259,851.416277,6.782050e-262,261.168639,KEGG_MEDICUS_REFERENCE_GF_RTK_PI3K_SIGNALING_P...,0.000238,0.000238,6.697274e-260,259.174102
323,826.256292,4.516523e-260,259.345196,KEGG_MEDICUS_REFERENCE_TIGHT_JUNCTION_ACTIN_SI...,0.000182,0.000182,3.568053e-258,257.447569
...,...,...,...,...,...,...,...,...
368,539.005782,4.163330e-234,233.380559,KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_...,0.000225,0.000225,6.325059e-233,232.198935
336,530.900477,3.469258e-233,232.459763,KEGG_MEDICUS_REFERENCE_TSH_TG_SIGNALING_PATHWAY,0.000134,0.000134,5.075397e-232,231.294530
16,530.317822,4.045468e-233,232.393031,KEGG_ARRHYTHMOGENIC_RIGHT_VENTRICULAR_CARDIOMY...,0.000177,0.000177,5.707000e-232,231.243592
377,522.844411,2.947553e-232,231.530538,KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_...,0.000137,0.000137,4.014771e-231,230.396339


In [9]:
male_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)[:30].sort_values(by='ABS_Importance', ascending=False)

,statistic,male_pval,-log10(male_pval),pathway,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
49,580.178317,1.399270e-238,237.854099,KEGG_EPITHELIAL_CELL_SIGNALING_IN_HELICOBACTER...,0.000346,0.000346,2.909008e-237,236.536255
241,603.412238,5.747547e-241,240.240517,KEGG_MEDICUS_REFERENCE_DISASSEMBLY_OF_MCC,0.000262,0.000262,1.513521e-239,238.820012
23,879.523636,7.193633e-264,263.143052,KEGG_BASE_EXCISION_REPAIR,0.000260,0.000260,9.471616e-262,261.023576
302,712.123102,4.909306e-251,250.308980,KEGG_MEDICUS_REFERENCE_ORGANIZATION_OF_THE_OUT...,0.000248,0.000248,1.939176e-249,248.712383
386,692.253142,2.577238e-249,248.588845,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PTCH...,0.000247,0.000247,9.254628e-248,247.033641
...,...,...,...,...,...,...,...,...
377,522.844411,2.947553e-232,231.530538,KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_...,0.000137,0.000137,4.014771e-231,230.396339
136,586.607324,2.993208e-239,238.523863,KEGG_REGULATION_OF_ACTIN_CYTOSKELETON,0.000135,0.000135,6.568428e-238,237.182539
336,530.900477,3.469258e-233,232.459763,KEGG_MEDICUS_REFERENCE_TSH_TG_SIGNALING_PATHWAY,0.000134,0.000134,5.075397e-232,231.294530
153,520.828874,5.060347e-232,231.295820,KEGG_STEROID_HORMONE_BIOSYNTHESIS,0.000129,0.000129,6.662790e-231,230.176344


In [7]:
male_pathway_pval_df.sort_values(by='ABS_Importance', ascending=False)

,statistic,male_pval,-log10(male_pval),pathway,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
78,349.003178,1.062364e-207,206.973727,KEGG_HYPERTROPHIC_CARDIOMYOPATHY_HCM,0.000452,0.000452,4.611360e-207,206.336171
149,364.110336,2.835188e-210,209.547418,KEGG_SPHINGOLIPID_METABOLISM,0.000352,0.000352,1.382592e-209,208.859306
49,580.178317,1.399270e-238,237.854099,KEGG_EPITHELIAL_CELL_SIGNALING_IN_HELICOBACTER...,0.000346,0.000346,2.909008e-237,236.536255
126,438.954926,1.248383e-221,220.903652,KEGG_PRION_DISEASES,0.000327,0.000327,9.303989e-221,220.031331
203,301.336921,8.774641e-199,198.056771,KEGG_MEDICUS_REFERENCE_ACTH_CORTISOL_SIGNALING...,0.000321,0.000321,2.795148e-198,197.553595
...,...,...,...,...,...,...,...,...
206,19.342659,2.235338e-41,40.650657,KEGG_MEDICUS_REFERENCE_ADRB3_UCP1_SIGNALING_PA...,0.000003,0.000003,2.311410e-41,40.636123
307,22.001517,2.787229e-47,46.554827,KEGG_MEDICUS_REFERENCE_PINK_PARKIN_MEDIATED_AU...,0.000003,0.000003,2.935881e-47,46.532262
147,13.343057,1.016766e-26,25.992779,KEGG_SMALL_CELL_LUNG_CANCER,0.000003,0.000003,1.019347e-26,25.991678
339,16.549080,9.216082e-35,34.035454,KEGG_MEDICUS_REFERENCE_TYPE_I_INTERFERON_TO_JA...,0.000003,0.000003,9.334237e-35,34.029921


---

---

In [ ]:
pd.set_option("display.max_rows", None)

In [383]:
significant_cmn_pathway_df = cmn_pathway_df[['KEGG_ARGININE_AND_PROLINE_METABOLISM',
       'KEGG_MEDICUS_REFERENCE_LPAR_GNB_G_RHO_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_BMP_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_TYPE_I_IFN_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_VARIANT_FZD7_OVEREXPRESSION_TO_WNT_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PINK1_TO_ELECTRON_TRANSFER_IN_COMPLEX_I',
       'KEGG_MEDICUS_REFERENCE_LONG_PATCH_BER',
       'KEGG_T_CELL_RECEPTOR_SIGNALING_PATHWAY',
       'KEGG_AUTOIMMUNE_THYROID_DISEASE',
       'KEGG_MEDICUS_REFERENCE_VGCC_CA2_APOPTOTIC_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_IFN_RIPK1_3_SIGNALING_PATHWAY',
       'KEGG_RNA_DEGRADATION',
       'KEGG_MEDICUS_REFERENCE_ELECTRON_TRANSFER_IN_COMPLEX_I',
       'KEGG_MEDICUS_REFERENCE_ITGA_B_RHOGEF_RHOA_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_MICROTUBULE_RHOA_SIGNALING_PATHWAY']]
significant_cmn_pathway_df

,KEGG_ARGININE_AND_PROLINE_METABOLISM,KEGG_MEDICUS_REFERENCE_LPAR_GNB_G_RHO_SIGNALING_PATHWAY,KEGG_MEDICUS_REFERENCE_BMP_SIGNALING_PATHWAY,KEGG_MEDICUS_REFERENCE_TYPE_I_IFN_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_FZD7_OVEREXPRESSION_TO_WNT_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PINK1_TO_ELECTRON_TRANSFER_IN_COMPLEX_I,KEGG_MEDICUS_REFERENCE_LONG_PATCH_BER,KEGG_T_CELL_RECEPTOR_SIGNALING_PATHWAY,KEGG_AUTOIMMUNE_THYROID_DISEASE,KEGG_MEDICUS_REFERENCE_VGCC_CA2_APOPTOTIC_PATHWAY,KEGG_MEDICUS_REFERENCE_IFN_RIPK1_3_SIGNALING_PATHWAY,KEGG_RNA_DEGRADATION,KEGG_MEDICUS_REFERENCE_ELECTRON_TRANSFER_IN_COMPLEX_I,KEGG_MEDICUS_REFERENCE_ITGA_B_RHOGEF_RHOA_SIGNALING_PATHWAY,KEGG_MEDICUS_REFERENCE_MICROTUBULE_RHOA_SIGNALING_PATHWAY
0,0.000658,0.000722,0.000463,-0.001250,0.000677,0.000584,0.001268,0.002092,-0.001217,0.001329,0.000425,0.000987,0.001072,-0.001767,-0.000747
1,0.000679,0.000556,0.000432,-0.001462,0.000779,0.000616,0.001427,0.002249,-0.001529,0.001635,0.000508,0.001061,0.001054,-0.001798,-0.000634
2,0.000731,0.000724,0.000419,-0.001433,0.000757,0.000601,0.001337,0.002081,-0.001332,0.001285,0.000469,0.001061,0.001164,-0.001867,-0.000721
3,0.000629,0.000652,0.000429,-0.001340,0.000625,0.000528,0.001408,0.002019,-0.001215,0.001331,0.000420,0.001012,0.001109,-0.001761,-0.000734
4,0.000761,0.000753,0.000496,-0.001469,0.000794,0.000688,0.001531,0.002395,-0.001453,0.001498,0.000421,0.001126,0.001292,-0.002006,-0.000835
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
241,0.000653,0.000660,0.000384,-0.001208,0.000746,0.000572,0.001223,0.001844,-0.001176,0.001233,0.000474,0.000895,0.001081,-0.001706,-0.000800
242,0.000510,0.000622,0.000361,-0.001064,0.000610,0.000449,0.000938,0.001522,-0.001019,0.001078,0.000412,0.000823,0.000896,-0.001298,-0.000697
243,0.000664,0.000678,0.000444,-0.001318,0.000716,0.000533,0.001253,0.001941,-0.001248,0.001221,0.000492,0.001000,0.001121,-0.001823,-0.000740
244,0.000753,0.000700,0.000520,-0.001493,0.000814,0.000601,0.001561,0.002345,-0.001454,0.001474,0.000504,0.001113,0.001324,-0.001998,-0.000799


In [391]:
np.sign(significant_cmn_pathway_df).apply(pd.Series.value_counts).T

,-1.0,1.0
KEGG_ARGININE_AND_PROLINE_METABOLISM,NaN,246.0
KEGG_MEDICUS_REFERENCE_LPAR_GNB_G_RHO_SIGNALING_PATHWAY,NaN,246.0
KEGG_MEDICUS_REFERENCE_BMP_SIGNALING_PATHWAY,NaN,246.0
KEGG_MEDICUS_REFERENCE_TYPE_I_IFN_SIGNALING_PATHWAY,246.0,NaN
KEGG_MEDICUS_VARIANT_FZD7_OVEREXPRESSION_TO_WNT_SIGNALING_PATHWAY,NaN,246.0
KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PINK1_TO_ELECTRON_TRANSFER_IN_COMPLEX_I,NaN,246.0
KEGG_MEDICUS_REFERENCE_LONG_PATCH_BER,NaN,246.0
KEGG_T_CELL_RECEPTOR_SIGNALING_PATHWAY,NaN,246.0
KEGG_AUTOIMMUNE_THYROID_DISEASE,246.0,NaN
KEGG_MEDICUS_REFERENCE_VGCC_CA2_APOPTOTIC_PATHWAY,NaN,246.0


In [386]:
significant_male_pathway_df = male_pathway_df[['KEGG_MEDICUS_REFERENCE_GNRH_GNRHR_PLCB_PKC_SIGNALING_PATHWAY',
       'KEGG_ABC_TRANSPORTERS',
       'KEGG_MEDICUS_ENV_FACTOR_IRON_TO_ANTEROGRADE_AXONAL_TRANSPORT',
       'KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_HTT_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION',
       'KEGG_PHOSPHATIDYLINOSITOL_SIGNALING_SYSTEM',
       'KEGG_MEDICUS_REFERENCE_CXCR4_GNB_G_RAC_SIGNALING_PATHWAY',
       'KEGG_ONE_CARBON_POOL_BY_FOLATE',
       'KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_ATP1A1_TO_ANGIOTENSIN_ALDOSTERONE_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_DRD1_GNAS_AC_PKA_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_ABETA_TO_AGE_RAGE_SIGNALING_PATHWAY',
       'KEGG_RETINOL_METABOLISM',
       'KEGG_MEDICUS_REFERENCE_ACH_CHRN_RAS_ERK_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_MGLUR5_CA2_APOPTOTIC_PATHWAY',
       'KEGG_TOLL_LIKE_RECEPTOR_SIGNALING_PATHWAY', 'KEGG_PROTEASOME']]
significant_male_pathway_df

,KEGG_MEDICUS_REFERENCE_GNRH_GNRHR_PLCB_PKC_SIGNALING_PATHWAY,KEGG_ABC_TRANSPORTERS,KEGG_MEDICUS_ENV_FACTOR_IRON_TO_ANTEROGRADE_AXONAL_TRANSPORT,KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_HTT_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,KEGG_PHOSPHATIDYLINOSITOL_SIGNALING_SYSTEM,KEGG_MEDICUS_REFERENCE_CXCR4_GNB_G_RAC_SIGNALING_PATHWAY,KEGG_ONE_CARBON_POOL_BY_FOLATE,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_ATP1A1_TO_ANGIOTENSIN_ALDOSTERONE_SIGNALING_PATHWAY,KEGG_MEDICUS_REFERENCE_DRD1_GNAS_AC_PKA_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_ABETA_TO_AGE_RAGE_SIGNALING_PATHWAY,KEGG_RETINOL_METABOLISM,KEGG_MEDICUS_REFERENCE_ACH_CHRN_RAS_ERK_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_MGLUR5_CA2_APOPTOTIC_PATHWAY,KEGG_TOLL_LIKE_RECEPTOR_SIGNALING_PATHWAY,KEGG_PROTEASOME
0,-0.000980,-0.001042,0.001565,0.001497,0.001043,0.000978,0.000990,0.000878,0.000321,-0.001587,0.000479,0.001503,0.001070,0.000504,-0.001198
1,-0.001075,-0.001155,0.001489,0.001815,0.001176,0.000948,0.001069,0.000632,0.000252,-0.001599,0.000490,0.001500,0.000916,0.000567,-0.001204
2,-0.001059,-0.001142,0.001425,0.001704,0.001215,0.000962,0.000879,0.000731,0.000330,-0.001422,0.000424,0.001375,0.000931,0.000621,-0.001037
3,-0.001003,-0.000978,0.001297,0.001651,0.001078,0.000887,0.000911,0.000656,0.000277,-0.001434,0.000457,0.001309,0.000873,0.000558,-0.001067
4,-0.001027,-0.001037,0.001666,0.001706,0.001014,0.001079,0.001039,0.000813,0.000331,-0.001692,0.000503,0.001569,0.001056,0.000577,-0.001235
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136,-0.001045,-0.001070,0.001336,0.001669,0.001115,0.000969,0.000944,0.000671,0.000281,-0.001437,0.000439,0.001394,0.000894,0.000521,-0.001105
137,-0.000929,-0.000932,0.001189,0.001503,0.000985,0.000783,0.000835,0.000568,0.000250,-0.001252,0.000396,0.001189,0.000796,0.000456,-0.000964
138,-0.001067,-0.001068,0.001446,0.001824,0.001183,0.000909,0.001007,0.000666,0.000282,-0.001652,0.000450,0.001502,0.000936,0.000561,-0.001181
139,-0.001043,-0.001087,0.001458,0.001745,0.001202,0.000956,0.001004,0.000738,0.000300,-0.001658,0.000443,0.001493,0.001043,0.000590,-0.001144


In [392]:
np.sign(significant_male_pathway_df).apply(pd.Series.value_counts).T

,-1.0,1.0
KEGG_MEDICUS_REFERENCE_GNRH_GNRHR_PLCB_PKC_SIGNALING_PATHWAY,141.0,NaN
KEGG_ABC_TRANSPORTERS,141.0,NaN
KEGG_MEDICUS_ENV_FACTOR_IRON_TO_ANTEROGRADE_AXONAL_TRANSPORT,NaN,141.0
KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_HTT_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,NaN,141.0
KEGG_PHOSPHATIDYLINOSITOL_SIGNALING_SYSTEM,NaN,141.0
KEGG_MEDICUS_REFERENCE_CXCR4_GNB_G_RAC_SIGNALING_PATHWAY,NaN,141.0
KEGG_ONE_CARBON_POOL_BY_FOLATE,NaN,141.0
KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_ATP1A1_TO_ANGIOTENSIN_ALDOSTERONE_SIGNALING_PATHWAY,NaN,141.0
KEGG_MEDICUS_REFERENCE_DRD1_GNAS_AC_PKA_SIGNALING_PATHWAY,NaN,141.0
KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_ABETA_TO_AGE_RAGE_SIGNALING_PATHWAY,141.0,NaN


In [388]:
significant_female_pathway_df = female_pathway_df[['KEGG_ALLOGRAFT_REJECTION',
       'KEGG_MEDICUS_REFERENCE_GLOBAL_GENOME_NER',
       'KEGG_MEDICUS_REFERENCE_ADRB3_UCP1_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_HOMOLOGOUS_RECOMBINATION_IN_ICLR',
       'KEGG_FC_EPSILON_RI_SIGNALING_PATHWAY', 'KEGG_ENDOMETRIAL_CANCER',
       'KEGG_HISTIDINE_METABOLISM',
       'KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABBERANT_ATXN2_3_TO_MGLUR5_CA2_APOPTOTIC_PATHWAY',
       'KEGG_CELL_CYCLE',
       'KEGG_MEDICUS_REFERENCE_CGAS_STING_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_ABETA_TO_AGE_RAGE_SIGNALING_PATHWAY',
       'KEGG_PYRIMIDINE_METABOLISM',
       'KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PDE11A_PDE8B_TO_ACTH_CORTISOL_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_ABETA_TO_ELECTRON_TRANSFER_IN_COMPLEX_IV',
       'KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PRKAR1A_TO_ACTH_CORTISOL_SIGNALING_PATHWAY']]
significant_female_pathway_df

,KEGG_ALLOGRAFT_REJECTION,KEGG_MEDICUS_REFERENCE_GLOBAL_GENOME_NER,KEGG_MEDICUS_REFERENCE_ADRB3_UCP1_SIGNALING_PATHWAY,KEGG_MEDICUS_REFERENCE_HOMOLOGOUS_RECOMBINATION_IN_ICLR,KEGG_FC_EPSILON_RI_SIGNALING_PATHWAY,KEGG_ENDOMETRIAL_CANCER,KEGG_HISTIDINE_METABOLISM,KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABBERANT_ATXN2_3_TO_MGLUR5_CA2_APOPTOTIC_PATHWAY,KEGG_CELL_CYCLE,KEGG_MEDICUS_REFERENCE_CGAS_STING_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_ABETA_TO_AGE_RAGE_SIGNALING_PATHWAY,KEGG_PYRIMIDINE_METABOLISM,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PDE11A_PDE8B_TO_ACTH_CORTISOL_SIGNALING_PATHWAY,KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_ABETA_TO_ELECTRON_TRANSFER_IN_COMPLEX_IV,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PRKAR1A_TO_ACTH_CORTISOL_SIGNALING_PATHWAY
0,0.001283,0.000979,-0.001426,0.001174,0.001879,-0.000658,0.001803,0.000989,0.001856,0.000955,0.001849,0.001060,-0.001098,-0.001627,0.001537
1,0.001198,0.000985,-0.001314,0.001169,0.001719,-0.000565,0.001507,0.000871,0.001628,0.000866,0.001700,0.000994,-0.001053,-0.001550,0.001394
2,0.001260,0.001024,-0.001467,0.001326,0.001883,-0.000707,0.001769,0.000971,0.001870,0.001103,0.001853,0.001131,-0.001209,-0.001802,0.001594
3,0.001246,0.000972,-0.001322,0.001313,0.001867,-0.000650,0.001637,0.000945,0.001772,0.001040,0.001783,0.000990,-0.001183,-0.001875,0.001547
4,0.001237,0.000938,-0.001370,0.001269,0.001829,-0.000642,0.001639,0.000991,0.001798,0.001042,0.001718,0.000972,-0.001132,-0.001816,0.001511
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,0.001200,0.000925,-0.001298,0.001268,0.001717,-0.000670,0.001596,0.000942,0.001706,0.001009,0.001639,0.000920,-0.001048,-0.001708,0.001378
101,0.001155,0.000950,-0.001356,0.001138,0.001697,-0.000668,0.001541,0.000846,0.001605,0.000957,0.001666,0.000998,-0.001125,-0.001487,0.001408
102,0.001338,0.001083,-0.001519,0.001330,0.002055,-0.000731,0.001838,0.001051,0.001970,0.001183,0.001948,0.001107,-0.001383,-0.001977,0.001623
103,0.001308,0.001016,-0.001500,0.001363,0.001952,-0.000798,0.001814,0.001064,0.001956,0.001166,0.001906,0.001130,-0.001268,-0.001837,0.001639


In [393]:
np.sign(significant_female_pathway_df).apply(pd.Series.value_counts).T

,-1.0,1.0
KEGG_ALLOGRAFT_REJECTION,NaN,105.0
KEGG_MEDICUS_REFERENCE_GLOBAL_GENOME_NER,NaN,105.0
KEGG_MEDICUS_REFERENCE_ADRB3_UCP1_SIGNALING_PATHWAY,105.0,NaN
KEGG_MEDICUS_REFERENCE_HOMOLOGOUS_RECOMBINATION_IN_ICLR,NaN,105.0
KEGG_FC_EPSILON_RI_SIGNALING_PATHWAY,NaN,105.0
KEGG_ENDOMETRIAL_CANCER,105.0,NaN
KEGG_HISTIDINE_METABOLISM,NaN,105.0
KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABBERANT_ATXN2_3_TO_MGLUR5_CA2_APOPTOTIC_PATHWAY,NaN,105.0
KEGG_CELL_CYCLE,NaN,105.0
KEGG_MEDICUS_REFERENCE_CGAS_STING_SIGNALING_PATHWAY,NaN,105.0


---

---

---

In [30]:
common_gene_pval_df = pd.read_csv(f"[0825_1]Common_Genes_p-values_FDR_Correction.csv")
male_gene_pval_df = pd.read_csv(f"[0825_1]Male_Genes_p-values_FDR_Correction.csv")
female_gene_pval_df = pd.read_csv(f"[0825_1]Female_Genes_p-values_FDR_Correction.csv")

In [31]:
common_gene_pval_df

,statistic,common_pval,-log10(common_pval),Gene,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
0,247.178236,7.589358e-296,295.119795,HMGB1P1,1.677213e-07,1.677213e-07,1.386172e-295,294.858183
1,138.707042,7.855208e-235,234.104842,A2M,1.324456e-10,1.324456e-10,1.013432e-234,233.994205
2,324.092444,0.000000e+00,inf,AACS,1.421752e-07,1.421752e-07,0.000000e+00,inf
3,287.857320,5.307535e-312,311.275107,AADAT,4.094231e-07,4.094231e-07,1.142372e-311,310.942192
4,218.923206,5.471425e-283,282.261900,AANAT,9.353754e-09,9.353754e-09,9.354749e-283,282.028968
...,...,...,...,...,...,...,...,...
4784,101.614705,2.709341e-202,201.567136,ZNRF3,6.821541e-09,6.821541e-09,3.250259e-202,201.488082
4785,234.101537,4.358821e-290,289.360631,ZW10,1.781477e-07,1.781477e-07,7.736987e-290,289.111428
4786,234.101537,4.358821e-290,289.360631,ZWILCH,7.894198e-08,7.894198e-08,7.736987e-290,289.111428
4787,476.938915,0.000000e+00,inf,ZWINT,2.525503e-07,2.525503e-07,0.000000e+00,inf


In [32]:
common_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)

,statistic,common_pval,-log10(common_pval),Gene,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
4787,476.938915,0.000000e+00,inf,ZWINT,2.525503e-07,2.525503e-07,0.000000e+00,inf
4783,412.652957,0.000000e+00,inf,ZNF274,1.454005e-07,1.454005e-07,0.000000e+00,inf
4779,1029.690702,0.000000e+00,inf,ZFYVE9,3.114283e-08,3.114283e-08,0.000000e+00,inf
2,324.092444,0.000000e+00,inf,AACS,1.421752e-07,1.421752e-07,0.000000e+00,inf
4778,676.139580,0.000000e+00,inf,RBSN,7.027009e-08,7.027009e-08,0.000000e+00,inf
...,...,...,...,...,...,...,...,...
3234,16.161889,1.744360e-40,39.758364,PIF1,7.550197e-09,7.550197e-09,1.745818e-40,39.758001
2096,15.770953,3.760273e-39,38.424781,HRAS,5.228157e-08,5.228157e-08,3.762630e-39,38.424509
3204,15.580817,1.675163e-38,37.775943,PGF,8.840744e-09,8.840744e-09,1.675863e-38,37.775762
1140,14.747382,1.167185e-35,34.932860,CXCR5,3.392877e-08,3.392877e-08,1.167429e-35,34.932769


In [54]:
common_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)[:30]

,statistic,common_pval,-log10(common_pval),Gene,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
4787,476.938915,0.0,inf,ZWINT,2.525503e-07,2.525503e-07,0.0,inf
4783,412.652957,0.0,inf,ZNF274,1.454005e-07,1.454005e-07,0.0,inf
4779,1029.690702,0.0,inf,ZFYVE9,3.114283e-08,3.114283e-08,0.0,inf
2,324.092444,0.0,inf,AACS,1.421752e-07,1.421752e-07,0.0,inf
4778,676.139580,0.0,inf,RBSN,7.027009e-08,7.027009e-08,0.0,inf
...,...,...,...,...,...,...,...,...
70,784.336777,0.0,inf,ACAT2,1.293829e-06,1.293829e-06,0.0,inf
71,343.748364,0.0,inf,ASIC2,1.976638e-08,1.976638e-08,0.0,inf
72,447.496987,0.0,inf,ACD,3.423261e-07,3.423261e-07,0.0,inf
61,657.889757,0.0,inf,ACADL,1.959919e-07,1.959919e-07,0.0,inf


In [37]:
common_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)[:30].sort_values(by='ABS_Importance', ascending=False)

,statistic,common_pval,-log10(common_pval),Gene,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
4731,565.084003,0.0,inf,WNT3,1.535475e-06,1.535475e-06,0.0,inf
4739,517.055657,0.0,inf,WNT9A,1.355551e-06,1.355551e-06,0.0,inf
70,784.336777,0.0,inf,ACAT2,1.293829e-06,1.293829e-06,0.0,inf
69,967.495422,0.0,inf,ACAT1,8.522471e-07,8.522471e-07,0.0,inf
4717,1009.897984,0.0,inf,WEE1,7.659942e-07,7.659942e-07,0.0,inf
...,...,...,...,...,...,...,...,...
4716,372.006597,0.0,inf,WDR5,3.796004e-08,3.796004e-08,0.0,inf
6,745.719401,0.0,inf,AASDHPPT,3.447420e-08,3.447420e-08,0.0,inf
4777,1029.690702,0.0,inf,ZFYVE16,3.434858e-08,3.434858e-08,0.0,inf
4779,1029.690702,0.0,inf,ZFYVE9,3.114283e-08,3.114283e-08,0.0,inf


In [38]:
common_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)[:30].sort_values(by='ABS_Importance', ascending=False)['Gene'].values

array(['WNT3', 'WNT9A', 'ACAT2', 'ACAT1', 'WEE1', 'AASDH', 'WNT10A',
       'WNT7A', 'WNT8B', 'ZFP36', 'ACAP1', 'WNT3A', 'ACD', 'WRAP53',
       'ACAA1', 'ZWINT', 'ACADL', 'ACADVL', 'WEE2', 'ZNF274', 'AACS',
       'WDHD1', 'RBSN', 'ACAP2', 'ACAP3', 'WDR5', 'AASDHPPT', 'ZFYVE16',
       'ZFYVE9', 'ASIC2'], dtype=object)

In [47]:
common_gene_pval_df.sort_values(
    by=['FDR_BH', 'statistic'], 
    ascending=[True, False]
)[:30].sort_values(
    by='ABS_Importance', 
    ascending=False
)['Gene'].values

array(['CDK7', 'YWHAQ', 'POLR2I', 'CDKN1B', 'POLR2K', 'CACNA1C', 'POLR1C',
       'MMP2', 'IL33', 'PTTG2', 'HBEGF', 'MMP1', 'POLR3D', 'CHRD',
       'KLRC3', 'E2F5', 'ZBP1', 'LTBP1', 'TREX1', 'SFN', 'RBL1', 'HSPA5',
       'PGAP1', 'AIM2', 'PIGX', 'ID1', 'NOG', 'SMC3', 'KLRC2', 'PIGA'],
      dtype=object)

In [43]:
male_gene_pval_df.sort_values(
    by=['FDR_BH', 'statistic'], 
    ascending=[True, False]
)[:30].sort_values(
    by='ABS_Importance', 
    ascending=False
)['Gene'].values

array(['NEIL3', 'DVL1', 'FGF23', 'DNAH11', 'NEIL1', 'GIT1', 'MBD4',
       'HMGB1P1', 'HMGB1', 'CSK', 'MPG', 'ANGPT1', 'ATP6V0C', 'APEX2',
       'OGG1', 'APEX1', 'ATP6V1E1', 'XRCC1', 'PARP1', 'CETN2', 'NTHL1',
       'TDG', 'TRAIP', 'POLL', 'NEIL2', 'SMUG1', 'CLDN25', 'PARP3',
       'PARP2', 'MUTYH'], dtype=object)

In [44]:
male_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)[:30].sort_values(by='ABS_Importance', ascending=False)['Gene'].values

array(['NEIL3', 'DVL1', 'FGF23', 'DNAH11', 'NEIL1', 'GIT1', 'MBD4',
       'HMGB1P1', 'HMGB1', 'CSK', 'MPG', 'ANGPT1', 'ATP6V0C', 'APEX2',
       'OGG1', 'APEX1', 'ATP6V1E1', 'XRCC1', 'PARP1', 'CETN2', 'NTHL1',
       'TDG', 'TRAIP', 'POLL', 'NEIL2', 'SMUG1', 'CLDN25', 'PARP3',
       'PARP2', 'MUTYH'], dtype=object)

In [67]:
male_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)[:30]['Gene'].values

array(['DVL1', 'CSK', 'ANGPT1', 'TRAIP', 'GIT1', 'ATP6V0C', 'PARP1',
       'HMGB1', 'NEIL3', 'NEIL2', 'NTHL1', 'PARP3', 'PARP2', 'OGG1',
       'APEX2', 'TDG', 'SMUG1', 'APEX1', 'POLL', 'MBD4', 'MUTYH', 'MPG',
       'HMGB1P1', 'NEIL1', 'XRCC1', 'CLDN25', 'CETN2', 'ATP6V1E1',
       'DNAH11', 'FGF23'], dtype=object)

In [45]:
female_gene_pval_df.sort_values(
    by=['FDR_BH', 'statistic'], 
    ascending=[True, False]
)[:30].sort_values(
    by='ABS_Importance', 
    ascending=False
)['Gene'].values

array(['CREB3L4', 'PRKAG3', 'WIPF1', 'ITGA10', 'LRP5', 'POLR1A', 'NDUFS1',
       'AKR1C4', 'CDK5', 'EED', 'IRS2', 'RAB5A', 'BCORL1', 'RFC5', 'CSF2',
       'CX3CL1', 'TRIM27', 'POLR2K', 'GK2', 'ANGPT1', 'AREG', 'PRKAG1',
       'ELMO1', 'CCL23', 'CCR10', 'BCOR', 'CXCL13', 'PCGF1', 'USP7',
       'KDM2B'], dtype=object)

In [46]:
female_gene_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)[:30].sort_values(by='ABS_Importance', ascending=False)['Gene'].values

array(['CREB3L4', 'PRKAG3', 'WIPF1', 'ITGA10', 'LRP5', 'POLR1A', 'NDUFS1',
       'AKR1C4', 'CDK5', 'EED', 'IRS2', 'RAB5A', 'BCORL1', 'RFC5', 'CSF2',
       'CX3CL1', 'TRIM27', 'POLR2K', 'GK2', 'ANGPT1', 'AREG', 'PRKAG1',
       'ELMO1', 'CCL23', 'CCR10', 'BCOR', 'CXCL13', 'PCGF1', 'USP7',
       'KDM2B'], dtype=object)

---

In [48]:
drop_log_common_gene_pval_df = common_gene_pval_df.drop(columns=['-log10(common_pval)', '-log10(FDR_BH)'])
drop_log_male_gene_pval_df = male_gene_pval_df.drop(columns=['-log10(male_pval)', '-log10(FDR_BH)'])
drop_log_female_gene_pval_df = female_gene_pval_df.drop(columns=['-log10(female_pval)', '-log10(FDR_BH)'])

In [49]:
drop_log_common_gene_pval_df.sort_values(by='FDR_BH', ascending=True)[:30].sort_values(by='ABS_Importance', ascending=False)

,statistic,common_pval,Gene,ABS_Importance,Importance,FDR_BH
3266,608.802246,0.0,PIK3R2,1.821205e-06,1.821205e-06,0.0
1622,529.749031,0.0,FGF1,1.019865e-06,1.019865e-06,0.0
3335,745.719401,0.0,PLOD3,9.807505e-07,9.807505e-07,0.0
3321,387.428893,0.0,PLCG1,9.356955e-07,9.356955e-07,0.0
1623,604.049350,0.0,FGF20,8.940329e-07,8.940329e-07,0.0
...,...,...,...,...,...,...
1605,329.553841,0.0,FDFT1,1.363854e-07,1.363854e-07,0.0
1592,384.924351,0.0,FBXO5,1.086755e-07,1.086755e-07,0.0
3284,1029.690702,0.0,PITX2,1.084361e-07,1.084361e-07,0.0
1613,644.396925,0.0,FGD3,7.230115e-08,7.230115e-08,0.0


In [50]:
drop_log_common_gene_pval_df.sort_values(by='FDR_BH', ascending=True)[:30].sort_values(by='ABS_Importance', ascending=False)['Gene'].values

array(['PIK3R2', 'FGF1', 'PLOD3', 'PLCG1', 'FGF20', 'PINK1', 'PIK3R6',
       'PLD2', 'PIGQ', 'FGF18', 'FER', 'FGF14', 'PIK3R4', 'PIK3AP1',
       'FGF17', 'PIP4K2B', 'FBXW11', 'PKLR', 'FGD1', 'PIP4K2C', 'PIN1',
       'PIKFYVE', 'PLOD2', 'FBXO43', 'PLIN1', 'FDFT1', 'FBXO5', 'PITX2',
       'FGD3', 'PLOD1'], dtype=object)

In [88]:
drop_log_common_gene_pval_df.sort_values(by='FDR_BH', ascending=True)[:15]['Gene'].values

array(['FGF18', 'PLD2', 'PLIN1', 'PLOD1', 'PIN1', 'FGD1', 'FGD3', 'FGF14',
       'FGF17', 'PLCG1', 'FGF1', 'FGF20', 'PIK3AP1', 'PIK3R2', 'PIK3R4'],
      dtype=object)

In [90]:
drop_log_common_gene_pval_df.sort_values(
    by=['FDR_BH', 'statistic'], 
    ascending=[True, False]
)[:15]['Gene'].values

array(['POLR3D', 'POLR1C', 'KLRC2', 'KLRC3', 'CDK7', 'E2F5', 'YWHAQ',
       'CDKN1B', 'RBL1', 'PTTG2', 'PGAP1', 'CACNA1C', 'SMC3', 'SFN',
       'POLR2I'], dtype=object)

In [61]:
top_common_gene_pval_df = drop_log_common_gene_pval_df.sort_values(
    by=['FDR_BH', 'statistic'], 
    ascending=[True, False]
)[:30].sort_values(
    by='ABS_Importance', 
    ascending=False
).reset_index(drop=True)

In [62]:
top_male_gene_pval_df = drop_log_male_gene_pval_df.sort_values(
    by=['FDR_BH', 'statistic'], 
    ascending=[True, False]
)[:30].sort_values(
    by='ABS_Importance', 
    ascending=False
).reset_index(drop=True)

In [63]:
top_female_gene_pval_df = drop_log_female_gene_pval_df.sort_values(
    by=['FDR_BH', 'statistic'], 
    ascending=[True, False]
)[:30].sort_values(
    by='ABS_Importance', 
    ascending=False
).reset_index(drop=True)

In [64]:
top_common_gene_pval_df

,statistic,common_pval,Gene,ABS_Importance,Importance,FDR_BH
0,1259.082617,0.0,CDK7,1.108202e-06,1.108202e-06,0.0
1,1235.184893,0.0,YWHAQ,9.910131e-07,9.910131e-07,0.0
2,1067.809635,0.0,POLR2I,9.454883e-07,9.454883e-07,0.0
3,1224.798753,0.0,CDKN1B,9.427747e-07,9.427747e-07,0.0
4,1060.921778,0.0,POLR2K,9.371005e-07,9.371005e-07,0.0
...,...,...,...,...,...,...
25,1029.690702,0.0,ID1,1.974507e-07,1.974507e-07,0.0
26,1029.690702,0.0,NOG,1.867881e-07,1.867881e-07,0.0
27,1106.466542,0.0,SMC3,1.740228e-07,1.740228e-07,0.0
28,1317.505001,0.0,KLRC2,1.618883e-07,1.618883e-07,0.0


In [65]:
top_male_gene_pval_df

,statistic,male_pval,Gene,ABS_Importance,Importance,FDR_BH
0,858.038948,2.292678e-262,NEIL3,1.197217e-06,1.197217e-06,4.391855e-260
1,951.141201,1.253409e-268,DVL1,1.136109e-06,1.136109e-06,6.002574e-265
2,795.800996,8.664902e-258,FGF23,9.372649e-07,9.372649e-07,1.383207e-255
3,797.976160,5.913418e-258,DNAH11,7.907574e-07,7.907574e-07,9.765296e-256
4,858.038948,2.292678e-262,NEIL1,7.390085e-07,7.390085e-07,4.391855e-260
...,...,...,...,...,...,...
25,858.038948,2.292678e-262,SMUG1,1.387143e-07,1.387143e-07,4.391855e-260
26,825.461437,5.167862e-260,CLDN25,8.614958e-08,8.614958e-08,9.518805e-258
27,858.038948,2.292678e-262,PARP3,5.604590e-08,5.604590e-08,4.391855e-260
28,858.038948,2.292678e-262,PARP2,4.096389e-09,4.096389e-09,4.391855e-260


In [66]:
top_female_gene_pval_df

,statistic,female_pval,Gene,ABS_Importance,Importance,FDR_BH
0,667.644701,1.047544e-190,CREB3L4,2.142052e-06,2.142052e-06,5.574096e-188
1,645.502356,3.492655e-189,PRKAG3,1.844238e-06,1.844238e-06,1.393861e-186
2,669.173701,8.258026e-191,WIPF1,1.208943e-06,1.208943e-06,4.943461e-188
3,592.504974,2.579401e-185,ITGA10,1.153180e-06,1.153180e-06,4.411697e-183
4,729.916750,9.847374e-195,LRP5,1.047326e-06,1.047326e-06,2.357954e-191
...,...,...,...,...,...,...
25,631.170499,3.605935e-188,BCOR,2.286271e-07,2.286271e-07,9.088855e-186
26,612.063434,8.811999e-187,CXCL13,2.111684e-07,2.111684e-07,1.834811e-184
27,631.170499,3.605935e-188,PCGF1,1.807807e-07,1.807807e-07,9.088855e-186
28,631.170499,3.605935e-188,USP7,1.110370e-07,1.110370e-07,9.088855e-186


In [68]:
top_common_gene_pval_df.to_csv(f"Significant_Common_Genes_p-values_FDR_Correction.csv", header = True, index = False)
top_male_gene_pval_df.to_csv(f"Significant_Male_Genes_p-values_FDR_Correction.csv", header = True, index = False)
top_female_gene_pval_df.to_csv(f"Significant_Female_Genes_p-values_FDR_Correction.csv", header = True, index = False)

---

---

In [69]:
common_pathway_pval_df = pd.read_csv(f"[0825_1]Common_Pathways_p-values_FDR_Correction.csv")
male_pathway_pval_df = pd.read_csv(f"[0825_1]Male_Pathways_p-values_FDR_Correction.csv")
female_pathway_pval_df = pd.read_csv(f"[0825_1]Female_Pathways_p-values_FDR_Correction.csv")

In [77]:
common_pathway_pval_df

,statistic,common_pval,-log10(common_pval),pathway,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
0,408.963138,0.000000e+00,inf,KEGG_ABC_TRANSPORTERS,0.000116,0.000116,0.000000e+00,inf
1,119.139799,6.903559e-219,218.160927,KEGG_ACUTE_MYELOID_LEUKEMIA,0.000067,0.000067,8.629449e-219,218.064017
2,506.814709,0.000000e+00,inf,KEGG_ADHERENS_JUNCTION,0.000194,0.000194,0.000000e+00,inf
3,286.493945,1.692587e-311,310.771449,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,0.000098,0.000098,3.633543e-311,310.439670
4,251.800306,8.253290e-298,297.083373,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,0.000199,0.000199,1.559832e-297,296.806922
...,...,...,...,...,...,...,...,...
390,381.061377,0.000000e+00,inf,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,0.000155,0.000155,0.000000e+00,inf
391,318.132559,1.284571e-322,321.891242,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000056,0.000056,3.162020e-322,321.500035
392,49.597084,9.167552e-130,129.037747,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000027,0.000027,9.921050e-130,129.003442
393,38.159532,4.438809e-105,104.352734,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000007,0.000007,4.663110e-105,104.331324


In [70]:
drop_log_common_pathway_pval_df = common_pathway_pval_df.drop(columns=['-log10(common_pval)', '-log10(FDR_BH)'])
drop_log_male_pathway_pval_df = male_pathway_pval_df.drop(columns=['-log10(male_pval)', '-log10(FDR_BH)'])
drop_log_female_pathway_pval_df = female_pathway_pval_df.drop(columns=['-log10(female_pval)', '-log10(FDR_BH)'])

In [72]:
top_common_pathway_pval_df = drop_log_common_pathway_pval_df.sort_values(
    by=['FDR_BH', 'statistic'], 
    ascending=[True, False]
)[:30].sort_values(
    by='ABS_Importance', 
    ascending=False
).reset_index(drop=True)

In [73]:
top_male_pathway_pval_df = drop_log_male_pathway_pval_df.sort_values(
    by=['FDR_BH', 'statistic'], 
    ascending=[True, False]
)[:30].sort_values(
    by='ABS_Importance', 
    ascending=False
).reset_index(drop=True)

In [74]:
top_female_pathway_pval_df = drop_log_female_pathway_pval_df.sort_values(
    by=['FDR_BH', 'statistic'], 
    ascending=[True, False]
)[:30].sort_values(
    by='ABS_Importance', 
    ascending=False
).reset_index(drop=True)

In [85]:
top_common_pathway_pval_df.to_csv(f"Significant_Common_Pathways_p-values_FDR_Correction.csv", header = True, index = False)
top_male_pathway_pval_df.to_csv(f"Significant_Male_Pathway_p-values_FDR_Correction.csv", header = True, index = False)
top_female_pathway_pval_df.to_csv(f"Significant_Female_Pathway_p-values_FDR_Correction.csv", header = True, index = False)

In [91]:
drop_log_common_pathway_pval_df.sort_values(
    by=['FDR_BH', 'statistic'], 
    ascending=[True, False]
)[:5]['pathway'].values

array(['KEGG_CYTOSOLIC_DNA_SENSING_PATHWAY',
       'KEGG_TGF_BETA_SIGNALING_PATHWAY', 'KEGG_CELL_CYCLE',
       'KEGG_GNRH_SIGNALING_PATHWAY', 'KEGG_PPAR_SIGNALING_PATHWAY'],
      dtype=object)

In [87]:
drop_log_male_pathway_pval_df.sort_values(
    by=['FDR_BH', 'statistic'], 
    ascending=[True, False]
)[:5]

,statistic,male_pval,pathway,ABS_Importance,Importance,FDR_BH
363,920.649283,1.199187e-266,KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_...,0.000169,0.000169,4.736787e-264
333,894.888792,6.369666e-265,KEGG_MEDICUS_REFERENCE_TRAIP_DEPENDENT_REPLISO...,0.000228,0.000228,1.258009e-262
23,879.523636,7.193633e-264,KEGG_BASE_EXCISION_REPAIR,0.000260,0.000260,9.471616e-262
259,851.416277,6.782050e-262,KEGG_MEDICUS_REFERENCE_GF_RTK_PI3K_SIGNALING_P...,0.000238,0.000238,6.697274e-260
323,826.256292,4.516523e-260,KEGG_MEDICUS_REFERENCE_TIGHT_JUNCTION_ACTIN_SI...,0.000182,0.000182,3.568053e-258


In [79]:
male_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)[:5].sort_values(by='ABS_Importance', ascending=False)['pathway'].values

array(['KEGG_BASE_EXCISION_REPAIR',
       'KEGG_MEDICUS_REFERENCE_GF_RTK_PI3K_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_TRAIP_DEPENDENT_REPLISOME_DISASSEMBLY',
       'KEGG_MEDICUS_REFERENCE_TIGHT_JUNCTION_ACTIN_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_ABETA_TO_ELECTRON_TRANSFER_IN_COMPLEX_IV'],
      dtype=object)

In [83]:
male_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)[:5]['pathway'].values

array(['KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_ABETA_TO_ELECTRON_TRANSFER_IN_COMPLEX_IV',
       'KEGG_MEDICUS_REFERENCE_TRAIP_DEPENDENT_REPLISOME_DISASSEMBLY',
       'KEGG_BASE_EXCISION_REPAIR',
       'KEGG_MEDICUS_REFERENCE_GF_RTK_PI3K_SIGNALING_PATHWAY',
       'KEGG_MEDICUS_REFERENCE_TIGHT_JUNCTION_ACTIN_SIGNALING_PATHWAY'],
      dtype=object)

In [84]:
top_male_pathway_pval_df[:5]['pathway'].values

array(['KEGG_EPITHELIAL_CELL_SIGNALING_IN_HELICOBACTER_PYLORI_INFECTION',
       'KEGG_MEDICUS_REFERENCE_DISASSEMBLY_OF_MCC',
       'KEGG_BASE_EXCISION_REPAIR',
       'KEGG_MEDICUS_REFERENCE_ORGANIZATION_OF_THE_OUTER_KINETOCHORE',
       'KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PTCH1_TO_HEDGEHOG_SIGNALING_PATHWAY'],
      dtype=object)

In [71]:
drop_log_common_pathway_pval_df

,statistic,common_pval,pathway,ABS_Importance,Importance,FDR_BH
0,408.963138,0.000000e+00,KEGG_ABC_TRANSPORTERS,0.000116,0.000116,0.000000e+00
1,119.139799,6.903559e-219,KEGG_ACUTE_MYELOID_LEUKEMIA,0.000067,0.000067,8.629449e-219
2,506.814709,0.000000e+00,KEGG_ADHERENS_JUNCTION,0.000194,0.000194,0.000000e+00
3,286.493945,1.692587e-311,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,0.000098,0.000098,3.633543e-311
4,251.800306,8.253290e-298,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,0.000199,0.000199,1.559832e-297
...,...,...,...,...,...,...
390,381.061377,0.000000e+00,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,0.000155,0.000155,0.000000e+00
391,318.132559,1.284571e-322,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000056,0.000056,3.162020e-322
392,49.597084,9.167552e-130,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000027,0.000027,9.921050e-130
393,38.159532,4.438809e-105,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000007,0.000007,4.663110e-105


In [17]:
common_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)

,statistic,common_pval,-log10(common_pval),pathway,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
390,381.061377,0.000000e+00,inf,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,0.000155,0.000155,0.000000e+00,inf
0,408.963138,0.000000e+00,inf,KEGG_ABC_TRANSPORTERS,0.000116,0.000116,0.000000e+00,inf
388,509.448312,0.000000e+00,inf,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_UBQL...,0.000110,0.000110,0.000000e+00,inf
2,506.814709,0.000000e+00,inf,KEGG_ADHERENS_JUNCTION,0.000194,0.000194,0.000000e+00,inf
354,460.935401,0.000000e+00,inf,KEGG_MEDICUS_VARIANT_MUTATION_ACTIVATED_GNAS_T...,0.000148,0.000148,0.000000e+00,inf
...,...,...,...,...,...,...,...,...
378,19.312260,3.938427e-51,50.404677,KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_...,0.000005,0.000005,3.978718e-51,50.400257
180,19.221960,7.880142e-51,50.103466,KEGG_MEDICUS_PATHOGEN_ESCHERICHIA_EAE_TIR_TO_A...,0.000003,0.000003,7.940449e-51,50.100155
339,19.150033,1.369844e-50,49.863329,KEGG_MEDICUS_REFERENCE_TYPE_I_INTERFERON_TO_JA...,0.000003,0.000003,1.376815e-50,49.861124
316,16.548039,8.425919e-42,41.074383,KEGG_MEDICUS_REFERENCE_RIG_I_IRF7_3_SIGNALING_...,0.000007,0.000007,8.447305e-42,41.073282


In [16]:
common_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)[:30].sort_values(by='ABS_Importance', ascending=False)

,statistic,common_pval,-log10(common_pval),pathway,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
345,633.965159,0.0,inf,KEGG_MEDICUS_REFERENCE_WNT_SIGNALING_PATHWAY,0.000264,0.000264,0.0,inf
343,539.168517,0.0,inf,KEGG_MEDICUS_REFERENCE_WNT_SIGNALING_MODULATIO...,0.000260,0.000260,0.0,inf
331,342.681027,0.0,inf,KEGG_MEDICUS_REFERENCE_TNF_NFKB_SIGNALING_PATHWAY,0.000258,0.000258,0.0,inf
40,1055.861677,0.0,inf,KEGG_CYTOSOLIC_DNA_SENSING_PATHWAY,0.000256,0.000256,0.0,inf
332,461.281143,0.0,inf,KEGG_MEDICUS_REFERENCE_TNF_P38_SIGNALING_PATHWAY,0.000225,0.000225,0.0,inf
...,...,...,...,...,...,...,...,...
66,326.939724,0.0,inf,KEGG_GLYCOSAMINOGLYCAN_BIOSYNTHESIS_HEPARAN_SU...,0.000099,0.000099,0.0,inf
320,400.106221,0.0,inf,KEGG_MEDICUS_REFERENCE_SPINDLE_ASSEMBLY_CHECKP...,0.000090,0.000090,0.0,inf
46,374.085484,0.0,inf,KEGG_ECM_RECEPTOR_INTERACTION,0.000089,0.000089,0.0,inf
52,356.242147,0.0,inf,KEGG_FATTY_ACID_METABOLISM,0.000074,0.000074,0.0,inf


In [14]:
male_pathway_pval_df

,statistic,male_pval,-log10(male_pval),pathway,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
0,260.598681,5.738358e-190,189.241212,KEGG_ABC_TRANSPORTERS,0.000138,0.000138,1.425567e-189,188.846012
1,91.774108,6.006174e-127,126.221402,KEGG_ACUTE_MYELOID_LEUKEMIA,0.000030,0.000030,7.603970e-127,126.118960
2,62.403780,4.585260e-104,103.338636,KEGG_ADHERENS_JUNCTION,0.000025,0.000025,5.422688e-104,103.265785
3,276.384251,1.548562e-193,192.810071,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,0.000081,0.000081,4.161101e-193,192.380792
4,310.078077,1.611358e-200,199.792808,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,0.000084,0.000084,5.486953e-200,199.260669
...,...,...,...,...,...,...,...,...
390,165.576908,1.751210e-162,161.756662,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,0.000047,0.000047,2.894260e-162,161.538462
391,113.083925,1.787562e-139,138.747739,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000035,0.000035,2.401656e-139,138.619489
392,122.474941,2.818496e-144,143.549983,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000061,0.000061,3.892678e-144,143.409752
393,354.925839,1.009874e-208,207.995733,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000165,0.000165,4.532957e-208,207.343618


In [9]:
male_pathway_pval_df.sort_values(by='-log10(FDR_BH)', ascending=False)[:30].sort_values(by='ABS_Importance', ascending=False)

,statistic,male_pval,-log10(male_pval),pathway,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
49,580.178317,1.399270e-238,237.854099,KEGG_EPITHELIAL_CELL_SIGNALING_IN_HELICOBACTER...,0.000346,0.000346,2.909008e-237,236.536255
241,603.412238,5.747547e-241,240.240517,KEGG_MEDICUS_REFERENCE_DISASSEMBLY_OF_MCC,0.000262,0.000262,1.513521e-239,238.820012
23,879.523636,7.193633e-264,263.143052,KEGG_BASE_EXCISION_REPAIR,0.000260,0.000260,9.471616e-262,261.023576
302,712.123102,4.909306e-251,250.308980,KEGG_MEDICUS_REFERENCE_ORGANIZATION_OF_THE_OUT...,0.000248,0.000248,1.939176e-249,248.712383
386,692.253142,2.577238e-249,248.588845,KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PTCH...,0.000247,0.000247,9.254628e-248,247.033641
...,...,...,...,...,...,...,...,...
377,522.844411,2.947553e-232,231.530538,KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_...,0.000137,0.000137,4.014771e-231,230.396339
136,586.607324,2.993208e-239,238.523863,KEGG_REGULATION_OF_ACTIN_CYTOSKELETON,0.000135,0.000135,6.568428e-238,237.182539
336,530.900477,3.469258e-233,232.459763,KEGG_MEDICUS_REFERENCE_TSH_TG_SIGNALING_PATHWAY,0.000134,0.000134,5.075397e-232,231.294530
153,520.828874,5.060347e-232,231.295820,KEGG_STEROID_HORMONE_BIOSYNTHESIS,0.000129,0.000129,6.662790e-231,230.176344


In [15]:
female_pathway_pval_df

,statistic,female_pval,-log10(female_pval),pathway,ABS_Importance,Importance,FDR_BH,-log10(FDR_BH)
0,90.736759,7.724063e-101,100.112154,KEGG_ABC_TRANSPORTERS,0.000045,0.000045,1.101446e-100,99.958037
1,379.978140,2.931634e-165,164.532890,KEGG_ACUTE_MYELOID_LEUKEMIA,0.000135,0.000135,2.894988e-164,163.538353
2,40.688473,1.071961e-65,64.969821,KEGG_ADHERENS_JUNCTION,0.000037,0.000037,1.238083e-65,64.907250
3,400.431575,1.260851e-167,166.899336,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,0.000374,0.000374,1.556363e-166,165.807889
4,167.603096,2.344840e-128,127.629887,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,0.000136,0.000136,5.061266e-128,127.295741
...,...,...,...,...,...,...,...,...
390,143.751629,1.879673e-121,120.725918,KEGG_MEDICUS_VARIANT_NOTCH_OVEREXPRESSION_TO_N...,0.000032,0.000032,3.535576e-121,120.451540
391,159.522234,3.921640e-126,125.406532,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000151,0.000151,7.863186e-126,125.104401
392,159.428421,4.168026e-126,125.380070,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000046,0.000046,8.315002e-126,125.080138
393,139.195480,5.264350e-120,119.278655,KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPS...,0.000062,0.000062,9.671712e-120,119.014497
